# KYC OCR pipeline v4 — latency redesign

Same model, same OCR requirements, redesigned execution. Edit **CELL 4**, then run top to bottom.

---

## Bottleneck diagnosis of the previous notebook

The previous notebook was functionally correct and took **> 1 hour per PDF**. Counting its calls
first, for a typical customer (identity 2p, domicile 1p, convention 1p, FATCA 3p, signature 1p):

| Document | Calls | Why |
|---|---|---|
| identity | 5–12 | lines + fields **per page**, title, up to 2 tile retries per page, up to 3 MRZ attempts |
| domicile / convention / signature | 2–4 each | title + name + up to 2 retries |
| FATCA | 4–6 | one title call **per page**, plus name + retries |
| **total** | **15–30** | |

15–30 calls cannot take an hour unless **each call costs 120–240 s**, which is absurd for a 27B
FP8 model on an H100 — a 300-token decode should be a few seconds. So the call count was a
secondary problem. The dominant cost was *inside* each call, and two of the three causes were
self-inflicted by the previous notebook's own generation plumbing:

### Cause 1 — a GPU→CPU synchronisation on every generated token (dominant)

`_LogprobRecorder` ran as a `LogitsProcessor` and did this at **every decoding step**:

```python
lp = torch.log_softmax(scores.float(), dim=-1)     # fp32 over a ~152k vocab
...
for i, (t, v) in enumerate(zip(top.tolist(), vals.tolist())):   # .tolist() = device sync
```

`.tolist()` forces `cudaDeviceSynchronize`. At `MAX_NEW_TOKENS_LINES = 768` that is **768 full
pipeline stalls per call**, each preceded by a fp32 softmax over the whole vocabulary. The H100
spent most of its time idle, waiting for Python. This is precisely the "GPU idle while the CPU
works" pattern, and it was inside the hottest possible loop.

### Cause 2 — an O(n²) stopping criterion

`_BraceStop` decoded the **entire generated tail** every 8 tokens and walked it character by
character in Python:

| Task | cap | tokens re-decoded per call |
|---|---|---|
| lines | 768 | ~36,500 |
| identity fields | 512 | ~16,100 |
| MRZ | 160 | ~1,500 |

All of it on the CPU, on the critical path between GPU steps.

### Cause 3 — redundant work around the calls

* every ROI re-opened the PDF (`pymupdf.open` per MRZ attempt, per title, per tile);
* the title was a separate Qwen call even when the PDF had a perfectly good text layer;
* `lines` and `fields` both ran on every page even when the first already produced everything;
* every call ran alone — batch size 1 on a GPU built for throughput.

### What this version changes

| Fix | Effect |
|---|---|
| Token log-probs off by default; when on, captured **after** generation, never per step | removes 128–768 device syncs per call |
| `_FastBraceStop` decodes **only the new tokens** since the last check, keeping brace depth as state | O(n) instead of O(n²) |
| `PageCache`: PDF opened once, pages rendered once, handle kept for ROI clips | no repeated `pymupdf.open` |
| PDF **text layer** used for titles and classification when present | removes ~1 call per document |
| One combined call per page: document type **and** all identity fields | 2 calls/page → 1 |
| `lines` pass demoted to a retry, only when mandatory fields are missing | ~1 call/page saved |
| Micro-batching with OOM fallback, tasks grouped by token budget | real H100 utilisation |
| Task planner: nothing runs unless the plan says it is needed | fewer calls, same coverage |

Expected call count per customer drops from **15–30 to roughly 6–10**, and per-call cost drops by
the sync and stop-criterion fixes. Both are **measured** by the benchmark cell, which runs the
same documents in `legacy` and `optimized` mode on your hardware rather than asserting a speedup.

**No OCR quality was traded away.** Resolution policy, the MRZ pipeline, handwriting crops,
anti-hallucination rules and every extracted field are unchanged.

In [ ]:
# =========================================================================
# CELL — DEPENDENCIES
# =========================================================================
# Already present in the Domino image (do not reinstall):
#   torch, transformers, accelerate, numpy, pandas, pillow
# Required by this notebook:
#   pymupdf                    PDF open/render/clip and text-layer anchors
#   opencv-python-headless     MRZ localisation, orientation, quality metrics
# Optional:
#   pytesseract + tesseract-ocr   page-orientation OSD only (never used for transcription)
#   openpyxl                      .xlsx customer database
#
# NOT installed and NOT imported anywhere: flash_attn. flash-linear-attention 0.5.2 is checked
# for compatibility below and is not used unless it genuinely applies to this architecture.
INSTALL_MISSING = False        # set True once, then back to False

if INSTALL_MISSING:
    import subprocess, sys
    for pkg in ["pymupdf", "opencv-python-headless", "openpyxl"]:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
    print("dependency install attempted; restart the kernel if pymupdf was just added")
else:
    print("INSTALL_MISSING=False (set True only if a dependency check below reports a gap)")

## Why the previous run took more than 5 hours

Runtime is **calls × output tokens per call**, and the previous architecture lost on both.

### The arithmetic

A 27B model under HF `transformers` decodes at roughly **10–25 tokens/s** on one H100 at batch 1
(FP8 reduces weight traffic but the decode loop is still memory-bound and Python-bound). At
`max_new_tokens = 384` a single call that does not stop early costs **15–40 s**. So:

| Customer profile | Calls (old plan) | At 25 s/call |
|---|---|---|
| identity 4p, domicile 2p, convention 7p, FATCA 4p, signature 1p | ~20 | ~8 min |
| × 40 customers | ~800 | **> 5 hours** |

Nothing was "hanging" — the pipeline was doing exactly what it was told, about 800 times.

### The three multipliers

1. **A call per page, on every document.** The planner created an identity task per page, a name
   task per secondary page, and a FATCA title task per page. Page counts vary per customer, so
   the call count scaled with total pages rather than with the information actually needed.
2. **Nested output format.** `{"surname_latin": {"value": "X", "confidence": "high"}}` costs
   ~20 output tokens per field; twelve fields plus scaffolding is ~250 tokens of pure JSON
   punctuation. Decode time is linear in that.
3. **No visibility before launch.** There was no way to see "this batch will issue 800 calls"
   until five hours in.

### What this version does about it

| Change | Effect on call count / tokens |
|---|---|
| **DRY_RUN plan preview** — page counts, tasks and predicted calls printed *before* any inference | makes an explosion visible in seconds |
| `MAX_QWEN_CALLS_PER_PDF` safety limit, checked at planning time | stops a runaway before it starts |
| **Flat JSON output** (`"surname_latin": "BEN ALI"` + one `unsure` list) | ~250 → ~130 output tokens per identity call |
| Identity extraction on **text-bearing pages only**, MRZ pages excluded from field extraction | fewer calls on multi-page passports |
| One name call per document, not per page | convention with 7 pages: 7 calls → 1 |
| FATCA/signature handwriting located by **PDF text-layer anchors** (`page.search_for`) | no per-page title calls when a text layer exists |
| Task-specific `max_new_tokens` (224 / 160 / 128 / 96) | shorter decode on every call |

Predicted result for the profile above: **~20 calls → ~7**, each with roughly half the output
tokens. The notebook prints the predicted count for *your* corpus in `DRY_RUN` mode, and measures
the achieved numbers afterwards. It does not assert a speedup it has not measured.

### Two corrections to earlier assumptions, now removed from the code

* **No fixed page counts.** `page_count` comes from the opened PDF. FATCA component 1 is no
  longer "two pages"; the page count is discovered and *reported*, and only flagged if a
  verification rule needs it.
* **`SPECIMEN DE SIGNATURE`**, not `SPICIMEN`. The earlier spelling has been removed as the
  expected title and is now only tolerated as an OCR variant when matching what was read.

In [ ]:
# =========================================================================
# CELL 2 — ENVIRONMENT DIAGNOSTICS
# =========================================================================
import os, sys, platform, subprocess, importlib, json

# Must be set before torch is imported anywhere in this process.
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

MODEL_PATH = "/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.8-27B-FP8/main"

print(f"{'python':<18}: {platform.python_version()}")
print(f"{'platform':<18}: {platform.platform()}")
for _m in ("torch", "transformers", "accelerate", "numpy", "pandas", "PIL", "cv2", "pymupdf"):
    try:
        _mod = importlib.import_module(_m)
        print(f"{_m:<18}: {getattr(_mod, '__version__', 'present')}")
    except Exception:
        print(f"{_m:<18}: NOT INSTALLED")

try:
    import torch
    print(f"{'cuda (torch)':<18}: {torch.version.cuda} | available={torch.cuda.is_available()}")
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            p = torch.cuda.get_device_properties(i)
            print(f"{'gpu[' + str(i) + ']':<18}: {p.name} | {p.total_memory/2**30:.1f} GiB | "
                  f"sm_{p.major}{p.minor}")
except Exception as exc:
    print("torch unavailable:", exc)

print(f"{'model path':<18}: {MODEL_PATH}")
print(f"{'model exists':<18}: {os.path.isdir(MODEL_PATH)}")
_cfg = os.path.join(MODEL_PATH, "config.json")
if os.path.isfile(_cfg):
    with open(_cfg) as fh:
        _raw = json.load(fh)
    print(f"{'model_type':<18}: {_raw.get('model_type')}")
    print(f"{'architectures':<18}: {_raw.get('architectures')}")
    print(f"{'vision tower':<18}: {'yes' if 'vision_config' in _raw else 'NO -- see CELL 7'}")
else:
    print("!! config.json not found at MODEL_PATH")

In [ ]:
# =========================================================================
# CELL 3 — IMPORTS
# =========================================================================
from __future__ import annotations

import io, re, gc, math, time, zipfile, hashlib, difflib, logging, unicodedata, traceback
from collections import OrderedDict, defaultdict
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone, date
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from PIL import Image

Image.MAX_IMAGE_PIXELS = None          # 300+ dpi A4 scans exceed Pillow's default guard

# PyMuPDF via its current import name only. The deprecated `fitz` alias is not used anywhere in
# this notebook, and no code path falls back to it.
try:
    import pymupdf
except Exception:
    pymupdf = None

try:
    import cv2
except Exception:
    cv2 = None

try:
    import pytesseract
    from pytesseract import Output as TessOutput
    pytesseract.get_tesseract_version()
except Exception:
    pytesseract, TessOutput = None, None

try:
    import torch
except Exception:
    torch = None

# NOTE: flash_attn is deliberately NOT imported, probed or installed anywhere in this notebook.
# The Domino environment cannot build it. CELL 6 selects a native PyTorch backend instead.

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)-7s %(message)s",
                    datefmt="%H:%M:%S")
log = logging.getLogger("kyc")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
utcnow = lambda: datetime.now(timezone.utc).isoformat()
print("run id:", RUN_ID)

In [ ]:
# =========================================================================
# CELL 4 — CONFIGURATION  (the only cell you normally edit)
# =========================================================================
@dataclass
class Config:
    # ---------------- paths -----------------------------------------------
    MODEL_PATH: str = MODEL_PATH
    ZIP_PATH: Path = Path("/domino/datasets/local/kyc/kyc_documents.zip")
    CUSTOMER_DATABASE_PATH: Optional[Path] = Path("/domino/datasets/local/kyc/customers.csv")
    OUTPUT_DIR: Path = Path("/domino/datasets/local/kyc/output_v3")

    # ---------------- rendering and resolution ----------------------------
    # Pages are rendered at this DPI and KEPT at full resolution. Downscaling happens per view,
    # never once destructively for the whole pipeline.
    PDF_RENDER_DPI: int = 300
    # Longest side of the FULL-PAGE view sent to the model. Qwen-VL spends roughly one visual
    # token per 28x28 px block, so 1600 px ~= 2300 tokens for a portrait page.
    MAX_IMAGE_DIMENSION: int = 1600
    MAX_VISUAL_TOKENS: int = 2400          # hard cap handed to the processor as max_pixels
    # Minimum glyph height for reliable OCR. Below this a region is escalated to a high-DPI
    # re-render rather than being sent as-is.
    MIN_TEXT_HEIGHT: float = 16.0
    MRZ_UPSCALE_FACTOR: float = 3.0
    HANDWRITING_UPSCALE_FACTOR: float = 2.5
    REGION_RENDER_DPI: int = 600           # ROI re-render: true resolution from the PDF
    MRZ_MAX_WIDTH: int = 1800
    MRZ_MIN_CHAR_HEIGHT: float = 22.0        # below this the MRZ ladder escalates
    MRZ_CANDIDATES_PER_PAGE: int = 3         # examined on EVERY page, then ranked globally
    MRZ_MIN_SCORE: float = 0.42              # below this a candidate is not an MRZ at all
    REGION_MAX_WIDTH: int = 1600
    PAGE_TILES: int = 2                    # tiles used when a page needs more effective resolution
    MAX_PAGES_PER_PDF: int = 12

    # ---------------- preprocessing (applied only when measured as needed) ----
    ENABLE_ORIENTATION_CORRECTION: bool = True
    ENABLE_DESKEW: bool = True
    ENABLE_CONTRAST_ENHANCEMENT: bool = True
    ENABLE_SHARPENING: bool = True         # mild unsharp, ROI crops only
    ENABLE_DENOISE: bool = True
    ENABLE_ADAPTIVE_THRESHOLD: bool = False  # last-resort MRZ variant only
    DESKEW_MIN_DEG: float = 0.4
    DESKEW_MAX_DEG: float = 15.0
    BLUR_VAR_POOR: float = 80.0            # variance of Laplacian at a fixed 1000 px work size
    BLUR_VAR_GOOD: float = 300.0
    CONTRAST_LOW: float = 0.30
    NOISE_SIGMA_HIGH: float = 6.0
    ILLUMINATION_POOR: float = 0.14
    INK_RATIO_BLANK: float = 0.002

    # ---------------- inference -------------------------------------------
    ATTENTION_IMPLEMENTATION: str = "sdpa"   # "sdpa" | "eager". FlashAttention is NEVER used.
    TORCH_DTYPE: str = "bfloat16"            # compute dtype around the FP8 weights
    # ---- FP8 (FineGrainedFP8Config) ----
    USE_FP8: bool = True
    FP8_ACTIVATION_SCHEME: str = "dynamic"   # the only scheme the class currently supports
    FP8_WEIGHT_BLOCK_SIZE: Tuple[int, int] = (128, 128)
    # Keep the VISION tower and the LM head out of FP8. Quantising the vision encoder costs
    # exactly the fine-detail fidelity that MRZ and handwriting OCR depend on, and the encoder is
    # a small fraction of the weights, so the memory saved is not worth the accuracy.
    FP8_SKIP_MODULES: Tuple[str, ...] = ("lm_head", "visual", "vision_tower", "merger",
                                         "vision_model", "multi_modal_projector")
    DEVICE_MAP: str = "auto"
    TEMPERATURE: float = 0.0                 # expressed as do_sample=False (see CELL 19)
    # Task-specific generation budgets. One global value wastes decode time on short answers
    # and truncates long ones; these are sized to each schema.
    MAX_NEW_TOKENS_IDENTITY: int = 224       # flat format: ~130 tokens of content + headroom
    MAX_NEW_TOKENS_LINES: int = 512          # line transcription, used only as a retry
    MAX_NEW_TOKENS_NAME: int = 128           # two names + title
    MAX_NEW_TOKENS_MRZ: int = 160            # 2-3 MRZ lines
    MAX_NEW_TOKENS_TITLE: int = 96           # a heading
    MAX_NEW_TOKENS: int = 384                # default for anything unlisted
    REPETITION_PENALTY: float = 1.0          # MUST be 1.0 -- see CELL 19
    NO_REPEAT_NGRAM_SIZE: int = 0            # MUST be 0 -- would forbid "<<" in an MRZ
    DISABLE_THINKING: bool = True
    RESERVE_VRAM_GIB: float = 8.0
    CPU_OFFLOAD_GIB: float = 64.0
    OOM_DOWNSCALE_FACTOR: float = 0.65
    OOM_MAX_DOWNSCALES: int = 2

    # ---------------- execution / latency ---------------------------------
    INFERENCE_BATCH_SIZE: int = 1            # 1 until tune_batch_size() proves 2+ is faster
    BRACE_CHECK_EVERY: int = 16              # stop-criterion check interval, O(n) total
    USE_PDF_TEXT_LAYER: bool = True          # free headings/classification on digital PDFs
    SECONDARY_FIRST_PAGE_ONLY: bool = True   # client name sits on page 1 of these forms
    PROFILE_CUDA_SYNC: bool = False          # sync around timings ONLY while benchmarking
    CPU_WORKERS: int = 4                     # bounded; avoids oversubscribing with cv2/torch
    # ---------------- run control: look before you leap --------------------
    DRY_RUN: bool = True                     # plan and print; NO inference. Set False to run.
    BENCHMARK_MODE: bool = True              # process a small sample first
    BENCHMARK_CUSTOMERS: int = 1             # ONE knob (two conflicting ones caused a 2x run)
    BENCHMARK_LEGACY: bool = False           # legacy comparison is OPT-IN: it deliberately
                                             # reproduces the slow architecture and takes hours
    BENCHMARK_MAX_SECONDS: float = 900.0     # watchdog: stop cleanly and report the live state
    QWEN_CALL_WARN_SECONDS: float = 90.0     # heartbeat warning for a single slow call
    # ---------------- hard ceilings, validated BEFORE any inference --------
    MAX_QWEN_CALLS_PER_PAGE: int = 3
    MAX_QWEN_CALLS_PER_PDF: int = 12
    MAX_QWEN_CALLS_PER_CUSTOMER: int = 25
    MAX_TOTAL_QWEN_CALLS: int = 5000
    DEBUG_SAMPLE_CUSTOMERS: Tuple[str, ...] = ()
    DEBUG_SAMPLE_DOCUMENTS: Tuple[str, ...] = ()
    MOCK_MODEL: bool = False                 # True = rehearse the whole pipeline with no GPU

    # ---------------- retries ---------------------------------------------
    MAX_TARGETED_RETRIES: int = 2            # per field group, hard limit
    MAX_MRZ_ATTEMPTS: int = 3

    # ---------------- verification ----------------------------------------
    # Only these four database columns are used. Nothing else is read.
    DB_COL_ID: str = "Id tiers"
    DB_COL_NAME: str = "Nom abrégé tiers"
    DB_COL_DOB: str = "Date de naissance"
    DB_COL_EXPIRY: str = "Date d'expiration du Document"
    NAME_FUZZY_THRESHOLD: float = 0.90       # exposed on every comparison, never auto-promoted
    TITLE_MATCH_THRESHOLD: float = 0.72

    # ---------------- run control -----------------------------------------
    SAVE_DEBUG_IMAGES: bool = False          # True = keep images for successful pages too
    SAVE_DEBUG_ON_FAILURE: bool = True       # always keep the crop behind a failed field
    # RESUME silently skipped every customer whose result file already existed, which made a
    # real run finish in seconds with "no Qwen data" and stale results. Off by default; when on,
    # the number of skipped customers is printed, and benchmarks never skip.
    RESUME: bool = False
    LIMIT_CUSTOMERS: Optional[int] = None
    MAX_CONSECUTIVE_ERRORS: int = 3
    PROCESS_SECONDARY_DOCUMENTS: bool = True

    def dirs(self) -> Dict[str, Path]:
        d = {k: self.OUTPUT_DIR / v for k, v in {
            "pdfs": "00_selected_pdfs", "results": "01_results", "reports": "02_reports",
            "debug": "debug_images", "logs": "04_logs"}.items()}
        for p in d.values():
            p.mkdir(parents=True, exist_ok=True)
        return d


CFG = Config()
DIRS = CFG.dirs()

IDENTITY_DOC = "JUSTIFICATIF IDENTITE.PDF"
DOMICILE_DOC = "JUSTIFICATIF DOMICILE.PDF"
CONVENTION_DOC = "CONVENTION COMPTE.PDF"
FATCA_DOC = "FATCA.PDF"
SIGNATURE_DOC = "CARTON SIGNATURE.PDF"
DOCUMENTS_REQUIRED = [IDENTITY_DOC, DOMICILE_DOC, CONVENTION_DOC, FATCA_DOC, SIGNATURE_DOC]
PRESENCE_COLUMNS = OrderedDict([
    (IDENTITY_DOC, "identity_document_exists"),
    (DOMICILE_DOC, "domicile_document_exists"),
    (CONVENTION_DOC, "convention_compte_exists"),
    (FATCA_DOC, "fatca_exists"),
    (SIGNATURE_DOC, "signature_card_exists"),
])

IDENTITY_FIELDS = ["document_type", "surname_latin", "given_names_latin", "date_of_birth",
                   "place_of_birth", "nationality", "sex", "document_number", "issue_date",
                   "expiry_date", "issuing_authority", "personal_number", "mrz"]
# Mandatory when visually available (section 12)
MANDATORY_FIELDS = ["surname_latin", "given_names_latin", "date_of_birth", "expiry_date"]
NAME_FIELDS = ["surname_latin", "given_names_latin"]

print("model   :", CFG.MODEL_PATH)
print("zip     :", CFG.ZIP_PATH, "| exists:", CFG.ZIP_PATH.exists())
print("database:", CFG.CUSTOMER_DATABASE_PATH)
print("output  :", CFG.OUTPUT_DIR)

In [ ]:
# =========================================================================
# CELL 5 — DEPENDENCY CHECKS
# =========================================================================
def check_dependencies(cfg: Config = CFG) -> Dict[str, Any]:
    """Report what is present. The notebook degrades rather than crashing, and says how."""
    status: Dict[str, Any] = {}
    problems: List[str] = []

    status["pymupdf"] = pymupdf is not None
    if pymupdf is None:
        problems.append("pymupdf MISSING -- mandatory: PDF rendering and ROI re-rendering")
    status["opencv"] = cv2 is not None
    if cv2 is None:
        problems.append("opencv MISSING -- MRZ localisation, orientation and quality metrics "
                        "are disabled; OCR still runs with reduced recall")
    status["torch"] = torch is not None
    if torch is None and not cfg.MOCK_MODEL:
        problems.append("torch MISSING -- mandatory for inference (or set CFG.MOCK_MODEL=True)")
    status["tesseract"] = pytesseract is not None       # optional, orientation only
    status["cuda"] = bool(torch is not None and torch.cuda.is_available())

    try:
        import transformers
        status["transformers"] = transformers.__version__
    except Exception:
        status["transformers"] = None
        problems.append("transformers MISSING -- mandatory")

    status["model_path_exists"] = Path(cfg.MODEL_PATH).exists()
    if not status["model_path_exists"]:
        problems.append(f"MODEL_PATH does not exist: {cfg.MODEL_PATH}")
    status["zip_exists"] = Path(cfg.ZIP_PATH).exists()
    if not status["zip_exists"]:
        problems.append(f"ZIP_PATH does not exist: {cfg.ZIP_PATH}")
    status["database_exists"] = bool(cfg.CUSTOMER_DATABASE_PATH and
                                     Path(cfg.CUSTOMER_DATABASE_PATH).exists())
    if not status["database_exists"]:
        problems.append("customer database not found -- database verification will report "
                        "NOT_AVAILABLE for every customer (extraction is unaffected)")

    for k, v in status.items():
        print(f"  {k:<20}: {v}")
    if problems:
        print("\nATTENTION")
        for p in problems:
            print("  -", p)
    else:
        print("\nall checks passed")
    return status


DEP_STATUS = check_dependencies(CFG)

In [ ]:
# =========================================================================
# CELL 6 — GPU AND ATTENTION DIAGNOSTICS
# =========================================================================
def gpu_memory() -> Dict[str, float]:
    if torch is None or not torch.cuda.is_available():
        return {}
    out = {}
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        out[f"gpu{i}_total_gib"] = round(total / 2**30, 1)
        out[f"gpu{i}_free_gib"] = round(free / 2**30, 1)
        out[f"gpu{i}_allocated_gib"] = round(torch.cuda.memory_allocated(i) / 2**30, 1)
    return out


def gpu_report() -> None:
    """This process's memory AND other processes on the card. A second PID here explains most
    'OOM on every call' reports."""
    if torch is None or not torch.cuda.is_available():
        print("no CUDA device visible -- set CFG.MOCK_MODEL = True to rehearse")
        return
    print(f"device            : {torch.cuda.get_device_name(0)}")
    for k, v in gpu_memory().items():
        print(f"{k:<18}: {v} GiB")
    try:
        out = subprocess.run(["nvidia-smi", "--query-compute-apps=pid,process_name,used_memory",
                              "--format=csv,noheader"], capture_output=True, text=True,
                             timeout=15).stdout.strip()
        print("processes on GPU  :", out or "(none)")
        print("this pid          :", os.getpid())
    except Exception as exc:
        print("nvidia-smi unavailable:", exc)


def select_attention_implementation(cfg: Config = CFG) -> str:
    """Choose a NATIVE PyTorch attention backend.

    flash_attn is never imported, probed or installed: the Domino image cannot build it, and a
    missing optional import must never be a failure path. SDPA is PyTorch's own fused attention
    (memory-efficient / flash kernels where the hardware supports them) and needs no extra
    package. If the installed transformers build rejects sdpa for this architecture, fall back to
    eager, which always works."""
    requested = (cfg.ATTENTION_IMPLEMENTATION or "sdpa").lower()
    if requested == "flash_attention_2":
        log.warning("ATTENTION_IMPLEMENTATION=flash_attention_2 requested but flash_attn is not "
                    "available in this environment; using sdpa instead")
        requested = "sdpa"
    if requested == "sdpa":
        if torch is None:
            return "eager"
        if not hasattr(torch.nn.functional, "scaled_dot_product_attention"):
            log.warning("torch has no scaled_dot_product_attention (torch<2.0) -> eager")
            return "eager"
    return requested


def release_cuda_cache() -> None:
    """NOT called per page: emptying the cache forces re-allocation and costs performance.
    Reserved for OOM recovery and teardown."""
    gc.collect()
    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()


gpu_report()
ATTENTION_IMPL = select_attention_implementation(CFG)
print("attention backend :", ATTENTION_IMPL, "(flash_attn is never used)")

In [ ]:
# =========================================================================
# CELL — ATTENTION BACKEND AND flash-linear-attention COMPATIBILITY
# =========================================================================
# flash-linear-attention 0.5.2 is installed, so it is CHECKED rather than assumed. The honest
# answer matters more than using the new package: FLA implements *linear-attention* kernels
# (GLA, RetNet, Mamba-style, RWKV, DeltaNet ...). Qwen-VL is a standard softmax-attention
# transformer. A linear-attention kernel is not a drop-in replacement for softmax attention --
# it computes a different function, so substituting it would silently change model outputs.
# `attn_implementation` in transformers also accepts only the backends the architecture
# registers ("eager", "sdpa", "flash_attention_2", ...); "fla" is not one of them.
FLA_REPORT: Dict[str, Any] = {"installed": False, "version": None, "compatible_with_model": False,
                              "used": False, "reason": ""}


def check_flash_linear_attention(cfg: Config = CFG) -> Dict[str, Any]:
    try:
        import fla  # noqa: F401
        FLA_REPORT["installed"] = True
        FLA_REPORT["version"] = getattr(fla, "__version__", "unknown")
    except Exception as exc:
        FLA_REPORT["reason"] = f"not importable ({type(exc).__name__})"
        return FLA_REPORT

    arch = " ".join(getattr(globals().get("ENGINE", None), "loader", "") or "")
    model_type = ""
    try:
        cfgj = json.loads((Path(cfg.MODEL_PATH) / "config.json").read_text())
        model_type = str(cfgj.get("model_type", ""))
        arch = " ".join(cfgj.get("architectures", []) or [])
    except Exception:
        pass
    FLA_REPORT["model_type"] = model_type
    FLA_REPORT["architectures"] = arch

    # FLA targets linear-attention families. Ask the package itself what it supports.
    supported = []
    try:
        import fla.models as fm
        supported = [n for n in dir(fm) if not n.startswith("_")]
    except Exception:
        pass
    FLA_REPORT["fla_supported_families"] = supported[:20]
    qwen_linear = any("qwen" in s.lower() for s in supported)
    FLA_REPORT["compatible_with_model"] = bool(qwen_linear)
    FLA_REPORT["reason"] = (
        "fla exposes a Qwen family; verify it is the SAME architecture and not a linear-attention "
        "variant before enabling" if qwen_linear else
        "Qwen3.8-VL uses softmax attention; flash-linear-attention provides linear-attention "
        "kernels, which compute a different function. Not a valid substitute.")
    return FLA_REPORT


def resolve_attention(cfg: Config = CFG) -> str:
    """Return the attention implementation that will ACTUALLY be used, and say why."""
    check_flash_linear_attention(cfg)
    requested = (cfg.ATTENTION_IMPLEMENTATION or "sdpa").lower()
    if requested == "flash_attention_2":
        log.warning("flash_attention_2 requested but flash-attn is not available here; using sdpa")
        requested = "sdpa"
    if requested in ("fla", "flash_linear_attention"):
        if not FLA_REPORT["compatible_with_model"]:
            log.warning("flash-linear-attention is installed but NOT compatible with this "
                        "architecture: %s -- falling back to sdpa", FLA_REPORT["reason"])
            requested = "sdpa"
    if requested == "sdpa" and torch is not None and \
            not hasattr(torch.nn.functional, "scaled_dot_product_attention"):
        log.warning("torch has no scaled_dot_product_attention -> eager")
        requested = "eager"
    return requested


print(f"{'transformers':<28}: {__import__('transformers').__version__ if 'transformers' in sys.modules or True else '?'}"
      if False else "")
_fla = check_flash_linear_attention(CFG)
print(f"{'flash-linear-attention':<28}: installed={_fla['installed']} version={_fla['version']}")
print(f"{'FLA compatible with model':<28}: {_fla['compatible_with_model']}")
print(f"{'FLA reason':<28}: {_fla['reason']}")
print(f"{'FLA actually used':<28}: {_fla['used']}")
ATTENTION_IMPL = resolve_attention(CFG)
print(f"{'attention backend in use':<28}: {ATTENTION_IMPL}   (flash_attn is never imported)")

In [ ]:
# =========================================================================
# CELL 8 — TRANSFORMERS API CHECK AND FineGrainedFP8Config RESOLUTION
# =========================================================================
# IMPORTANT, verified against the installed package rather than assumed:
#
#   from transformers.integrations.finegrained_fp8 import FineGrainedFP8Config   -> ImportError
#
# That module defines the RUNTIME pieces (FP8Linear, replace_with_fp8_linear, the Triton
# w8a8 block matmul). The CONFIG class lives in transformers.utils.quantization_config and is
# re-exported at the transformers top level. This resolver tries the path named in the spec
# first, then the ones that actually exist, and reports which succeeded -- so a future
# transformers release that does move the class keeps working.
import inspect

FP8_IMPORT_CANDIDATES = [
    ("transformers.integrations.finegrained_fp8", "FineGrainedFP8Config"),   # as specified
    ("transformers", "FineGrainedFP8Config"),                                # actual export
    ("transformers.utils.quantization_config", "FineGrainedFP8Config"),      # definition site
]


def resolve_fp8_config_class() -> Tuple[Any, str, List[str]]:
    """Return (class, import_path_used, constructor parameter names)."""
    import importlib
    errors = []
    for module_name, attr in FP8_IMPORT_CANDIDATES:
        try:
            mod = importlib.import_module(module_name)
            cls = getattr(mod, attr)
            params = [p for p in inspect.signature(cls.__init__).parameters
                      if p not in ("self", "args", "kwargs")]
            return cls, f"{module_name}.{attr}", params
        except Exception as exc:
            errors.append(f"{module_name}: {type(exc).__name__}: {exc}")
    raise ImportError("FineGrainedFP8Config not found. Tried:\n  " + "\n  ".join(errors) +
                      "\nFP8 quantisation is required by this pipeline; it will not silently "
                      "fall back to the non-quantised model.")


import transformers as _tf
print(f"{'transformers':<24}: {_tf.__version__}")
print(f"{'torch':<24}: {torch.__version__ if torch else 'MISSING'}")

FP8_CONFIG_CLASS, FP8_IMPORT_PATH, FP8_PARAMS = resolve_fp8_config_class()
print(f"{'FineGrainedFP8Config':<24}: FOUND via {FP8_IMPORT_PATH}")
print(f"{'constructor parameters':<24}: {FP8_PARAMS}")
print(f"{'docstring':<24}: {' '.join((FP8_CONFIG_CLASS.__doc__ or '').split())[:160]}...")

# The FP8 runtime pieces are in the integrations module; confirm they are importable.
try:
    from transformers.integrations.finegrained_fp8 import FP8Linear
    print(f"{'FP8Linear runtime':<24}: available")
except Exception as exc:
    FP8Linear = None
    print(f"{'FP8Linear runtime':<24}: NOT available ({exc})")

if torch is not None and torch.cuda.is_available():
    _cap = torch.cuda.get_device_capability(0)
    print(f"{'compute capability':<24}: sm_{_cap[0]}{_cap[1]} "
          f"({'FP8 tensor cores present' if _cap >= (8, 9) else 'NO native FP8 (needs sm_89+)'})")

In [ ]:
# =========================================================================
# CELL 9 — FP8 CONFIGURATION
# =========================================================================
def build_fp8_config(cfg: Config = CFG) -> Tuple[Any, Dict[str, Any]]:
    """Construct FineGrainedFP8Config using ONLY parameters the installed class accepts."""
    kwargs: Dict[str, Any] = {}
    if "activation_scheme" in FP8_PARAMS:
        kwargs["activation_scheme"] = cfg.FP8_ACTIVATION_SCHEME
    if "weight_block_size" in FP8_PARAMS and cfg.FP8_WEIGHT_BLOCK_SIZE:
        kwargs["weight_block_size"] = tuple(cfg.FP8_WEIGHT_BLOCK_SIZE)
    if "modules_to_not_convert" in FP8_PARAMS and cfg.FP8_SKIP_MODULES:
        # Vision tower and LM head stay out of FP8: the encoder is what resolves MRZ glyphs and
        # handwriting strokes, and it is a small share of the parameters.
        kwargs["modules_to_not_convert"] = list(cfg.FP8_SKIP_MODULES)
    quant_config = FP8_CONFIG_CLASS(**kwargs)
    print("FineGrainedFP8Config constructed with:", json.dumps(kwargs, default=str))
    return quant_config, kwargs


def checkpoint_quantization(cfg: Config = CFG) -> Optional[Dict[str, Any]]:
    """Read any quantization_config already present in the checkpoint's config.json."""
    p = Path(cfg.MODEL_PATH) / "config.json"
    if not p.exists():
        return None
    try:
        return json.loads(p.read_text()).get("quantization_config")
    except Exception:
        return None


FP8_QUANT_CONFIG, FP8_KWARGS = (build_fp8_config(CFG) if CFG.USE_FP8 else (None, {}))
CKPT_QUANT = checkpoint_quantization(CFG)
print("checkpoint quantization_config:", json.dumps(CKPT_QUANT) if CKPT_QUANT else "none declared")
if CKPT_QUANT and str(CKPT_QUANT.get("quant_method", "")).lower() not in ("fp8", "finegrained_fp8"):
    log.warning("checkpoint declares quant_method=%r, which is not FP8",
                CKPT_QUANT.get("quant_method"))

In [ ]:
# =========================================================================
# CELL 10 — MODEL AND PROCESSOR LOADING  (exactly once per kernel)
# =========================================================================
# Processor and chat-template mechanism are unchanged from the previous working notebook:
#   AutoProcessor (one instance, min_pixels/max_pixels)
#   processor.apply_chat_template(messages, add_generation_prompt=True) with content
#     [{"type": "image"}, {"type": "text", "text": ...}]
#   processor(text=[...], images=[PIL.Image], return_tensors="pt")
#   model.generate(...) inside torch.inference_mode()
# What changed: the weights are FP8 via FineGrainedFP8Config, and attention is sdpa.
VISION_HINTS = ("vl", "vision", "image_text", "qwen2_vl", "qwen2_5_vl", "qwen3_vl")


class QwenFP8Engine:
    _instance: Optional["QwenFP8Engine"] = None

    def __init__(self, cfg: Config = CFG):
        from transformers import AutoConfig, AutoProcessor
        import transformers as tf
        self.cfg = cfg
        assert torch is not None, "PyTorch is required"
        assert Path(cfg.MODEL_PATH).exists(), f"model path not found: {cfg.MODEL_PATH}"

        self.mem_before = gpu_memory()
        busy = max((v for k, v in self.mem_before.items() if k.endswith("allocated_gib")),
                   default=0.0)
        if busy > 1.0:
            log.warning("%.1f GiB already allocated on this GPU. If you re-ran this cell, call "
                        "free_model() or restart the kernel: two resident copies of the weights "
                        "will OOM on every generate().", busy)

        self.hf_config = AutoConfig.from_pretrained(cfg.MODEL_PATH, trust_remote_code=True,
                                                    local_files_only=True)
        archs = list(getattr(self.hf_config, "architectures", []) or [])
        mtype = str(getattr(self.hf_config, "model_type", "")).lower()
        if not (hasattr(self.hf_config, "vision_config") or
                any(h in " ".join(archs).lower() or h in mtype for h in VISION_HINTS)):
            raise RuntimeError(
                f"{cfg.MODEL_PATH}\n  model_type={mtype!r} architectures={archs}\n"
                "  No vision tower: this checkpoint cannot read an image.")

        # ---- processor: ONE instance, visual-token budget baked in --------
        px = cfg.MAX_VISUAL_TOKENS * 28 * 28      # Qwen-VL: ~1 visual token per 28x28 px block
        t0 = time.perf_counter()
        try:
            self.processor = AutoProcessor.from_pretrained(
                cfg.MODEL_PATH, trust_remote_code=True, local_files_only=True,
                min_pixels=256 * 28 * 28, max_pixels=px)
        except TypeError:
            self.processor = AutoProcessor.from_pretrained(cfg.MODEL_PATH, trust_remote_code=True,
                                                           local_files_only=True)
        self.processor_load_s = round(time.perf_counter() - t0, 2)
        self.tokenizer = getattr(self.processor, "tokenizer", self.processor)
        try:
            self.tokenizer.padding_side = "left"
        except Exception:
            pass

        kwargs: Dict[str, Any] = dict(
            torch_dtype=getattr(torch, cfg.TORCH_DTYPE), device_map=cfg.DEVICE_MAP,
            trust_remote_code=True, local_files_only=True, low_cpu_mem_usage=True,
            attn_implementation=ATTENTION_IMPL)
        if cfg.USE_FP8 and FP8_QUANT_CONFIG is not None:
            kwargs["quantization_config"] = FP8_QUANT_CONFIG

        last, self.model, self.fp8_path = None, None, None
        for cls_name in archs + ["AutoModelForImageTextToText", "AutoModelForVision2Seq"]:
            if not hasattr(tf, cls_name):
                continue
            for attempt in ("explicit_fp8_config", "checkpoint_fp8_config"):
                try:
                    t0 = time.perf_counter()
                    self.model = getattr(tf, cls_name).from_pretrained(cfg.MODEL_PATH, **kwargs)
                    self.loader = cls_name
                    self.load_s = round(time.perf_counter() - t0, 1)
                    self.fp8_path = attempt
                    break
                except Exception as exc:
                    msg = str(exc).lower()
                    last = f"{cls_name}/{attempt}: {type(exc).__name__}: {exc}"
                    release_cuda_cache()
                    # A pre-quantised checkpoint can refuse an explicit quantization_config. That
                    # is NOT a fallback to the non-quantised model: the checkpoint's own
                    # FineGrainedFP8Config applies, and it is verified below.
                    if attempt == "explicit_fp8_config" and "quantization_config" in kwargs and \
                            ("already" in msg or "quantiz" in msg):
                        log.warning("checkpoint rejected an explicit quantization_config (%s); "
                                    "loading with the checkpoint's own FP8 config", type(exc).__name__)
                        kwargs.pop("quantization_config", None)
                        continue
                    if "attention" in msg and kwargs.get("attn_implementation") != "eager":
                        log.warning("attn_implementation=%s rejected; retrying with eager",
                                    kwargs["attn_implementation"])
                        kwargs["attn_implementation"] = "eager"
                        continue
                    break
            if self.model is not None:
                break
        if self.model is None:
            raise RuntimeError(f"could not load the FP8 model. last error: {last}")

        self.attn = kwargs.get("attn_implementation", ATTENTION_IMPL)
        self.model.eval()
        gen_cfg = getattr(self.model, "generation_config", None)
        if gen_cfg is not None:        # neutralise any sampling defaults in the checkpoint
            gen_cfg.do_sample = False
            gen_cfg.temperature = gen_cfg.top_p = gen_cfg.top_k = None

        # ---- verify FP8 is ACTUALLY applied; never accept a silent bf16 load ----
        self.quantization = self._describe_quantization()
        if cfg.USE_FP8 and not self.quantization["is_fp8"]:
            raise RuntimeError(
                "FP8 was requested but the loaded model is not FP8 quantised "
                f"({self.quantization}). Refusing to continue on the non-quantised path.")
        self.mem_after = gpu_memory()
        self.name = f"{Path(cfg.MODEL_PATH).parent.name} [{self.loader}]"
        self.consecutive_ooms = 0

    def _describe_quantization(self) -> Dict[str, Any]:
        qc = getattr(self.model.config, "quantization_config", None)
        method = None
        if qc is not None:
            method = getattr(qc, "quant_method", None) or (
                qc.get("quant_method") if isinstance(qc, dict) else None)
            method = getattr(method, "value", method)
        n_fp8, n_linear, dtypes = 0, 0, {}
        for _, module in self.model.named_modules():
            cname = type(module).__name__
            if cname == "FP8Linear":
                n_fp8 += 1
            elif cname == "Linear":
                n_linear += 1
        for _, p in list(self.model.named_parameters())[:400]:
            dtypes[str(p.dtype)] = dtypes.get(str(p.dtype), 0) + 1
        return {"quant_method": str(method) if method else None,
                "n_fp8_linear_modules": n_fp8, "n_plain_linear_modules": n_linear,
                "parameter_dtypes_sample": dtypes,
                "is_fp8": bool(n_fp8 > 0 or (method and "fp8" in str(method).lower()))}

    @classmethod
    def get(cls, cfg: Config = CFG) -> "QwenFP8Engine":
        if cls._instance is None:
            cls._instance = cls(cfg)
        return cls._instance


class MockEngine:
    """Rehearsal without a GPU. Transcribes nothing: it invents nothing."""
    name, attn, consecutive_ooms = "MOCK", "n/a", 0
    quantization = {"is_fp8": False, "quant_method": "mock"}
    load_s = processor_load_s = 0.0
    mem_before = mem_after = {}


def free_model() -> None:
    inst = QwenFP8Engine._instance
    if inst is not None:
        try:
            inst.model.to("meta")
        except Exception:
            pass
        del inst.model
        QwenFP8Engine._instance = None
    release_cuda_cache()
    log.info("model freed. GPU: %s", gpu_memory() or "no CUDA")


def load_model(cfg: Config = CFG):
    """Load model AND processor exactly once. Never call this inside a loop."""
    return MockEngine() if cfg.MOCK_MODEL else QwenFP8Engine.get(cfg)


ENGINE = load_model(CFG)
print("engine        :", ENGINE.name)
print("attention     :", ENGINE.attn)
print("quantization  :", json.dumps(ENGINE.quantization, default=str))
print("fp8 config via:", getattr(ENGINE, "fp8_path", "n/a"))
print("load times    : model %.1fs | processor %.1fs" %
      (getattr(ENGINE, "load_s", 0.0), getattr(ENGINE, "processor_load_s", 0.0)))

In [ ]:
# =========================================================================
# CELL 11 — MODEL MEMORY DIAGNOSTICS AND WARM-UP
# =========================================================================
# Warm-up matters for honest benchmarking: the first call pays for CUDA context creation, kernel
# autotuning and lazy weight materialisation. It is measured separately and EXCLUDED from the
# steady-state figures reported in the benchmark cell.
MEMORY_TRACE: List[Dict[str, Any]] = []


def record_memory(stage: str) -> Dict[str, Any]:
    rec = {"stage": stage, "timestamp": utcnow()}
    if torch is not None and torch.cuda.is_available():
        rec.update({
            "allocated_gib": round(torch.cuda.memory_allocated() / 2**30, 2),
            "reserved_gib": round(torch.cuda.memory_reserved() / 2**30, 2),
            "max_allocated_gib": round(torch.cuda.max_memory_allocated() / 2**30, 2),
            "free_gib": round(torch.cuda.mem_get_info()[0] / 2**30, 2)})
    MEMORY_TRACE.append(rec)
    return rec


print("before load :", getattr(ENGINE, "mem_before", {}))
print("after load  :", record_memory("after_model_load"))

WARMUP_STATS: Dict[str, Any] = {"ran": False}


def warm_up(engine=None, cfg: Config = CFG) -> Dict[str, Any]:
    """One throwaway inference on a synthetic image, timed separately from real work."""
    engine = engine or ENGINE
    if isinstance(engine, MockEngine):
        return {"ran": False, "reason": "mock engine"}
    img = Image.new("RGB", (768, 512), "white")
    t0 = time.perf_counter()
    try:
        _ = run_qwen_ocr(img, "Return exactly: {\"ok\": true}", 16, engine, cfg=cfg)
        out = {"ran": True, "warmup_s": round(time.perf_counter() - t0, 2)}
    except Exception as exc:
        out = {"ran": False, "error": f"{type(exc).__name__}: {exc}"}
    out["memory"] = record_memory("after_warmup")
    return out


# warm_up() is called from the driver, AFTER run_qwen_ocr is defined.
print("CELL 11 ready: record_memory(), warm_up()")

In [ ]:
# =========================================================================
# CELL 12 — PERFORMANCE PROFILER
# =========================================================================
# Every stage writes here. CUDA is asynchronous, so GPU timings are only trustworthy with a
# synchronize() around them -- which itself costs time. PROFILE_CUDA_SYNC enables that ONLY for
# benchmarking; the production path never inserts synchronisation.
PERF_ROWS: List[Dict[str, Any]] = []
STAGE_TOTALS: Dict[str, float] = defaultdict(float)
STAGE_CALLS: Dict[str, int] = defaultdict(int)


class Profiler:
    """Context manager + accumulator. Cheap: one perf_counter pair per stage."""

    def __init__(self, cfg: Config = CFG):
        self.cfg = cfg

    @staticmethod
    def sync(cfg: Config = CFG) -> None:
        if cfg.PROFILE_CUDA_SYNC and torch is not None and torch.cuda.is_available():
            torch.cuda.synchronize()

    class _Timer:
        def __init__(self, stage: str, sink: Optional[Dict[str, float]], cfg: Config):
            self.stage, self.sink, self.cfg = stage, sink, cfg

        def __enter__(self):
            Profiler.sync(self.cfg)
            self.t0 = time.perf_counter()
            return self

        def __exit__(self, *exc):
            Profiler.sync(self.cfg)
            dt = time.perf_counter() - self.t0
            self.elapsed = dt
            STAGE_TOTALS[self.stage] += dt
            STAGE_CALLS[self.stage] += 1
            if self.sink is not None:
                self.sink[self.stage] = self.sink.get(self.stage, 0.0) + dt
            return False

    def stage(self, name: str, sink: Optional[Dict[str, float]] = None):
        return Profiler._Timer(name, sink, self.cfg)


PROF = Profiler(CFG)


def record_perf(**kw) -> None:
    """One row per inference task (section 41 columns)."""
    PERF_ROWS.append({"timestamp": utcnow(), **kw})


def stage_report() -> pd.DataFrame:
    total = sum(STAGE_TOTALS.values()) or 1.0
    rows = [{"stage": k, "total_time_s": round(v, 3), "calls": STAGE_CALLS[k],
             "avg_time_s": round(v / max(1, STAGE_CALLS[k]), 4),
             "pct_of_measured": round(100 * v / total, 1)}
            for k, v in sorted(STAGE_TOTALS.items(), key=lambda x: -x[1])]
    return pd.DataFrame(rows)


def gpu_snapshot(tag: str = "") -> Dict[str, Any]:
    rec = {"tag": tag, "timestamp": utcnow()}
    if torch is not None and torch.cuda.is_available():
        rec.update({"allocated_gib": round(torch.cuda.memory_allocated() / 2**30, 2),
                    "reserved_gib": round(torch.cuda.memory_reserved() / 2**30, 2),
                    "max_allocated_gib": round(torch.cuda.max_memory_allocated() / 2**30, 2),
                    "free_gib": round(torch.cuda.mem_get_info()[0] / 2**30, 2)})
        try:
            out = subprocess.run(["nvidia-smi",
                                  "--query-gpu=utilization.gpu,utilization.memory",
                                  "--format=csv,noheader,nounits"],
                                 capture_output=True, text=True, timeout=5).stdout.strip()
            u = out.split("\n")[0].split(",")
            rec["gpu_util_pct"] = int(u[0])
            rec["mem_util_pct"] = int(u[1])
        except Exception:
            pass
    MEMORY_TRACE.append(rec)
    return rec


MEMORY_TRACE: List[Dict[str, Any]] = []
print("CELL 12 ready: PROF.stage(), record_perf(), stage_report(), gpu_snapshot()")

In [ ]:
# =========================================================================
# CELL 13 — CUSTOMER DATABASE  (verification reference only; exactly four columns)
# =========================================================================
# The database is NEVER an OCR source and can never complete an unreadable value. Only these
# four columns are read; every other column in the file is ignored by construction.
def strip_accents(text: str) -> str:
    return "".join(c for c in unicodedata.normalize("NFKD", str(text))
                   if not unicodedata.combining(c))


def _find_column(columns: Sequence[str], wanted: str) -> Optional[str]:
    """Locate a column tolerating accent/case/whitespace differences in the export."""
    def key(s):
        return re.sub(r"[^a-z0-9]", "", strip_accents(str(s)).lower())
    target = key(wanted)
    for c in columns:
        if key(c) == target:
            return c
    for c in columns:                       # e.g. "Date d'expiration du Document (JJ/MM/AAAA)"
        if target and target in key(c):
            return c
    return None


def load_database(path: Optional[Path] = None, cfg: Config = CFG) -> Optional[pd.DataFrame]:
    path = Path(path or cfg.CUSTOMER_DATABASE_PATH) if (path or cfg.CUSTOMER_DATABASE_PATH) \
        else None
    if path is None or not path.exists():
        log.warning("customer database not found (%s): database verification -> NOT_AVAILABLE",
                    path)
        return None
    raw = pd.read_csv(path, dtype=str, sep=None, engine="python").fillna("") \
        if path.suffix.lower() == ".csv" else pd.read_excel(path, dtype=str).fillna("")

    mapping = {}
    for logical, wanted in [("customer_id", cfg.DB_COL_ID), ("name", cfg.DB_COL_NAME),
                            ("date_of_birth", cfg.DB_COL_DOB), ("expiry_date", cfg.DB_COL_EXPIRY)]:
        col = _find_column(raw.columns, wanted)
        if col is None:
            log.error("database column %r not found; available: %s", wanted, list(raw.columns))
            return None
        mapping[logical] = col
    log.info("database columns in use (and ONLY these): %s", mapping)

    df = raw[[mapping[k] for k in ("customer_id", "name", "date_of_birth", "expiry_date")]].copy()
    df.columns = ["customer_id", "name", "date_of_birth", "expiry_date"]
    df["customer_id"] = df["customer_id"].astype(str).str.strip()
    df = df.set_index("customer_id", drop=False)
    log.info("customer database: %d records", len(df))
    return df


def database_record(db: Optional[pd.DataFrame], customer_id: str) -> Optional[Dict[str, str]]:
    if db is None:
        return None
    cid = str(customer_id).strip()
    if cid not in db.index:
        return None
    row = db.loc[cid]
    if isinstance(row, pd.DataFrame):       # duplicate ids: refuse to pick one silently
        log.warning("customer %s appears %d times in the database; skipping comparison",
                    cid, len(row))
        return None
    return {k: str(v).strip() for k, v in row.to_dict().items()}


DB = load_database(cfg=CFG)
print("CELL 13 ready:", "database loaded" if DB is not None else "no database")

In [ ]:
# =========================================================================
# CELL 14 — ZIP EXTRACTION
# =========================================================================
def norm_name(text: str) -> str:
    """Accent-free, upper-case, punctuation-collapsed. Filename matching only."""
    s = re.sub(r"[^A-Z0-9]+", " ", strip_accents(text).upper())
    return re.sub(r"\s+", " ", s).strip()


DOC_ALIASES: "OrderedDict[str, List[str]]" = OrderedDict([
    (IDENTITY_DOC, ["JUSTIFICATIF IDENTITE", "JUSTIFICATIF D IDENTITE", "JUSTIF IDENTITE",
                    "PIECE IDENTITE", "PIECE D IDENTITE", "IDENTITE"]),
    (DOMICILE_DOC, ["JUSTIFICATIF DOMICILE", "JUSTIFICATIF DE DOMICILE", "JUSTIF DOMICILE",
                    "PREUVE DE DOMICILE", "DOMICILE"]),
    (CONVENTION_DOC, ["CONVENTION COMPTE", "CONVENTION DE COMPTE", "CONVENTION DU COMPTE",
                      "CONVENTION OUVERTURE COMPTE"]),
    (FATCA_DOC, ["FATCA", "FORMULAIRE FATCA", "FATCA CRS", "AUTOCERTIFICATION FATCA"]),
    # Earlier specifications spelled it SIGNATUTE; both are accepted so a folder matches
    # whichever way the file was actually named.
    (SIGNATURE_DOC, ["CARTON SIGNATURE", "CARTON SIGNATUTE", "CARTON DE SIGNATURE",
                     "SPECIMEN SIGNATURE", "SPECIMEN DE SIGNATURE", "SPICIMEN DE SIGNATURE"]),
])
DOC_NORM = {d: sorted({norm_name(a) for a in [d] + al}, key=len, reverse=True)
            for d, al in DOC_ALIASES.items()}
DOC_SLUG = {d: norm_name(d).replace(" ", "_") for d in DOC_ALIASES}


def match_document(filename: str, fuzzy_threshold: float = 0.88
                   ) -> Tuple[Optional[str], str, float]:
    """(canonical_name, match_type, score). Only capitalisation/whitespace/accents are treated as
    insignificant; an unrelated PDF must not be mapped, so fuzzy hits are flagged for review."""
    p = Path(str(filename))
    if p.suffix.lower() != ".pdf":
        return None, "not_pdf", 0.0
    stem = re.sub(r"\s+\d+$", "", norm_name(p.stem)).strip()
    for doc, aliases in DOC_NORM.items():
        if stem in aliases:
            return doc, "exact", 1.0
    for doc, aliases in DOC_NORM.items():
        for a in aliases:
            if len(a) >= 5 and a in stem:
                return doc, "contains", 0.95
    best_doc, best = None, 0.0
    for doc, aliases in DOC_NORM.items():
        for a in aliases:
            r = difflib.SequenceMatcher(None, stem, a).ratio()
            if r > best:
                best_doc, best = doc, r
    return (best_doc, "fuzzy", round(best, 3)) if best >= fuzzy_threshold \
        else (None, "no_match", 0.0)


def _safe_parts(member: str) -> Optional[List[str]]:
    name = member.replace("\\", "/")
    if name.startswith("/") or re.match(r"^[A-Za-z]:", name):
        return None
    parts = [p for p in name.split("/") if p not in ("", ".")]
    return None if any(p == ".." for p in parts) else parts


def extract_zip(zip_path: Path = None, cfg: Config = CFG) -> pd.DataFrame:
    """Unzip, keeping only the five required PDFs. The folder name is the customer_id and is
    carried through every downstream record."""
    zip_path = Path(zip_path or cfg.ZIP_PATH)
    assert zip_path.exists(), f"ZIP not found: {zip_path}"
    rows, seen = [], {}
    with zipfile.ZipFile(zip_path) as zf:
        infos = [i for i in zf.infolist() if not i.is_dir()]
        parts_all = [_safe_parts(i.filename) for i in infos]
        roots = {p[0] for p in parts_all if p and len(p) > 1}
        root = roots.pop() if len(roots) == 1 and any(p and len(p) > 2 for p in parts_all) else None
        for info, parts in zip(infos, parts_all):
            if parts is None:
                log.warning("unsafe zip member skipped: %s", info.filename)
                continue
            if root and parts[0] == root:
                parts = parts[1:]
            if not parts:
                continue
            customer_id = parts[0] if len(parts) > 1 else "_ARCHIVE_ROOT_"
            fname = parts[-1]
            doc, mtype, score = match_document(fname)
            row = dict(customer_id=customer_id, member="/".join(parts), filename=fname,
                       matched_document=doc or "", match_type=mtype, match_score=score,
                       size_bytes=info.file_size, extracted_path="")
            if doc is not None:
                dest_dir = DIRS["pdfs"] / customer_id
                dest_dir.mkdir(parents=True, exist_ok=True)
                n = seen.get((customer_id, doc), 0)
                seen[(customer_id, doc)] = n + 1
                dest = dest_dir / (f"{DOC_SLUG[doc]}.pdf" if n == 0
                                   else f"{DOC_SLUG[doc]}__dup{n+1}.pdf")
                dest.write_bytes(zf.read(info))
                row["extracted_path"] = str(dest)
            rows.append(row)
    catalog = pd.DataFrame(rows)
    log.info("zip: %d members, %d customer folders, %d required PDFs kept",
             len(catalog), catalog["customer_id"].nunique(),
             int((catalog["matched_document"] != "").sum()))
    return catalog


print("CELL 14 ready: extract_zip()")

In [ ]:
# =========================================================================
# CELL 15 — CUSTOMER / DOCUMENT DISCOVERY
# =========================================================================
@dataclass
class CustomerDocs:
    customer_id: str
    documents: Dict[str, Optional[str]]      # ONE selected PDF per logical type
    n_files: int
    duplicates: Dict[str, List[str]] = field(default_factory=dict)   # recorded, not processed

    @property
    def has_identity(self) -> bool:
        return bool(self.documents.get(IDENTITY_DOC))

    def path(self, doc: str) -> Optional[Path]:
        p = self.documents.get(doc)
        return Path(p) if p else None


def discover_documents(catalog: pd.DataFrame) -> List[CustomerDocs]:
    out = []
    for cid, grp in catalog.groupby("customer_id", sort=True):
        docs = {}
        for doc in DOCUMENTS_REQUIRED:
            hit = grp[(grp["matched_document"] == doc) & (grp["extracted_path"] != "")]
            docs[doc] = hit.iloc[0]["extracted_path"] if len(hit) else None
        out.append(CustomerDocs(str(cid), docs, len(grp)))
    return out


def select_targets(customers: List[CustomerDocs], cfg: Config = CFG) -> List[CustomerDocs]:
    """Only folders holding the identity document reach the model."""
    targets = [c for c in customers if c.has_identity]
    if cfg.LIMIT_CUSTOMERS:
        targets = targets[:cfg.LIMIT_CUSTOMERS]
    log.info("discovery: %d customers, %d with %s -> %d to process",
             len(customers), sum(c.has_identity for c in customers), IDENTITY_DOC, len(targets))
    return targets


print("CELL 15 ready: discover_documents()")

In [ ]:
# =========================================================================
# CELL 16 — DOCUMENT PRESENCE REPORT
# =========================================================================
def build_presence_report(customers: List[CustomerDocs], catalog: pd.DataFrame,
                          cfg: Config = CFG) -> pd.DataFrame:
    rows = []
    for c in customers:
        row: Dict[str, Any] = {"customer_id": c.customer_id}
        for doc, col in PRESENCE_COLUMNS.items():
            row[col] = "TRUE" if c.documents.get(doc) else "FALSE"
        grp = catalog[catalog["customer_id"] == c.customer_id]
        fuzzy = grp[grp["match_type"] == "fuzzy"]["filename"].tolist()
        row.update({"n_files_in_folder": c.n_files,
                    "n_required_present": sum(bool(c.documents.get(d))
                                              for d in DOCUMENTS_REQUIRED),
                    "will_be_processed": c.has_identity,
                    "fuzzy_matched_filenames": ";".join(fuzzy),
                    "needs_filename_review": bool(fuzzy)})
        rows.append(row)
    df = pd.DataFrame(rows)
    out = DIRS["reports"] / "document_presence_report.csv"
    df.to_csv(out, index=False, encoding="utf-8-sig")     # BOM: Excel keeps accents and Arabic
    log.info("presence report -> %s", out)
    summary = pd.DataFrame({
        "present": [int((df[c] == "TRUE").sum()) for c in PRESENCE_COLUMNS.values()],
        "missing": [int((df[c] == "FALSE").sum()) for c in PRESENCE_COLUMNS.values()]},
        index=list(PRESENCE_COLUMNS))
    print(summary.to_string())
    return df


print("CELL 16 ready: build_presence_report()")

In [ ]:
# =========================================================================
# CELL — DUPLICATE DOCUMENT SELECTION  (one PDF per logical type)
# =========================================================================
# A folder may hold FATCA.PDF, FATCA (1).PDF, FATCA (2).PDF. Exactly one is processed; the rest
# are recorded. Selection is deterministic: the canonical name first, then the shortest
# normalised name, then lexicographic -- never arbitrary.
DUP_SUFFIX = re.compile(r"\s*\((\d+)\)\s*$")


def _canonical_rank(filename: str, logical: str) -> Tuple[int, int, str]:
    stem = Path(filename).stem
    norm = norm_name(stem)
    m = DUP_SUFFIX.search(stem)
    copy_index = int(m.group(1)) if m else 0
    is_canonical = 0 if norm == norm_name(Path(logical).stem) else 1
    return (is_canonical, copy_index, norm)


def select_documents(catalog: pd.DataFrame) -> Tuple[List["CustomerDocs"], pd.DataFrame]:
    """Group files by logical type, pick one representative, record duplicates."""
    customers, inv_rows = [], []
    for cid, grp in catalog.groupby("customer_id", sort=True):
        docs: Dict[str, Optional[str]] = {}
        dups: Dict[str, List[str]] = {}
        for doc in DOCUMENTS_REQUIRED:
            hits = grp[(grp["matched_document"] == doc) & (grp["extracted_path"] != "")]
            if not len(hits):
                docs[doc] = None
                dups[doc] = []
                inv_rows.append({"customer_id": cid, "logical_document_type": doc,
                                 "selected_file": None, "duplicate_files": "",
                                 "n_candidates": 0})
                continue
            ranked = sorted(hits.to_dict("records"),
                            key=lambda r: _canonical_rank(r["filename"], doc))
            chosen = ranked[0]
            docs[doc] = chosen["extracted_path"]
            dups[doc] = [r["filename"] for r in ranked[1:]]
            if dups[doc]:
                log.info("%s / %s: %d duplicates ignored (%s)", cid, doc, len(dups[doc]),
                         ";".join(dups[doc][:3]))
            inv_rows.append({"customer_id": cid, "logical_document_type": doc,
                             "selected_file": chosen["filename"],
                             "duplicate_files": ";".join(dups[doc]),
                             "n_candidates": len(ranked)})
        c = CustomerDocs(str(cid), docs, len(grp))
        c.duplicates = dups                       # kept for the presence report
        customers.append(c)
    inventory = pd.DataFrame(inv_rows)
    return customers, inventory


print("CELL 17 ready: select_documents() -- one PDF per logical type, duplicates recorded")

In [ ]:
# =========================================================================
# CELL 18 — PDF RENDERING WITH PYMUPDF
# =========================================================================
# Every page is rendered explicitly and KEPT at full resolution. The model never receives a PDF;
# it receives images this code produced, so "page 2 was processed" is verifiable, not assumed.
@dataclass
class RenderedPage:
    page_number: int
    image: Image.Image                 # full-resolution render, never downscaled in place
    width: int
    height: int
    render_dpi: int
    render_time: float
    pdf_rect: Tuple[float, float, float, float]
    native_image_px: Optional[Tuple[int, int]] = None


def open_pdf(pdf_path: Path):
    if pymupdf is None:
        raise RuntimeError("pymupdf is required (note: the deprecated `fitz` alias is not used)")
    doc = pymupdf.open(str(pdf_path))
    if doc.page_count == 0:
        doc.close()
        raise ValueError(f"PDF has zero pages: {pdf_path}")
    return doc


def _largest_embedded_image(doc, index: int) -> Optional[Tuple[int, int]]:
    """Native size of the biggest raster on the page.

    A scanned PDF is usually one full-page JPEG. If that JPEG is 900 px wide, rendering at 600 dpi
    only interpolates -- this tells the diagnostics whether more DPI could ever help."""
    try:
        infos = doc[index].get_images(full=True)
        return max(((i[2], i[3]) for i in infos), key=lambda wh: wh[0] * wh[1]) if infos else None
    except Exception:
        return None


def render_pdf_pages(pdf_path: Path, cfg: Config = CFG) -> List[RenderedPage]:
    """Kept for ad-hoc use. The pipeline itself goes through PageCache, which renders once."""
    """Render EVERY page independently. No resizing happens here."""
    doc = open_pdf(pdf_path)
    pages: List[RenderedPage] = []
    try:
        n = min(doc.page_count, cfg.MAX_PAGES_PER_PDF)
        if doc.page_count > cfg.MAX_PAGES_PER_PDF:
            log.warning("%s has %d pages; capped at %d", pdf_path.name, doc.page_count, n)
        for i in range(n):
            t0 = time.perf_counter()
            page = doc[i]
            pix = page.get_pixmap(dpi=cfg.PDF_RENDER_DPI, colorspace=pymupdf.csRGB, alpha=False)
            img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples).copy()
            r = page.rect
            pages.append(RenderedPage(i + 1, img, pix.width, pix.height, cfg.PDF_RENDER_DPI,
                                      round(time.perf_counter() - t0, 3),
                                      (r.x0, r.y0, r.x1, r.y1),
                                      _largest_embedded_image(doc, i)))
    finally:
        doc.close()
    log.info("%s: %d pages rendered (%s)", pdf_path.name, len(pages),
             ", ".join(f"p{p.page_number} {p.width}x{p.height} {p.render_time}s" for p in pages))
    return pages


def render_pdf_clip(pdf_path: Path, page_number: int,
                    clip_rect: Tuple[float, float, float, float],
                    dpi: int) -> Optional[Image.Image]:
    """Re-render ONE RECTANGLE of a page at high DPI, straight from the PDF.

    This is the core resolution fix. Upscaling a downscaled crop interpolates pixels that were
    already discarded; re-rendering the clip recovers real detail from the source, and a narrow
    strip costs a fraction of the visual tokens a full high-DPI page would."""
    if pymupdf is None:
        return None
    doc = None
    try:
        doc = pymupdf.open(str(pdf_path))
        page = doc[page_number - 1]
        rect = pymupdf.Rect(*clip_rect) & page.rect
        if rect.is_empty or rect.width < 1 or rect.height < 1:
            return None
        zoom = dpi / 72.0
        pix = page.get_pixmap(matrix=pymupdf.Matrix(zoom, zoom), clip=rect,
                              colorspace=pymupdf.csRGB, alpha=False)
        return Image.frombytes("RGB", (pix.width, pix.height), pix.samples).copy()
    except Exception as exc:
        log.warning("clip re-render failed p%d: %s", page_number, exc)
        return None
    finally:
        if doc is not None:
            doc.close()


print("CELL 18 ready: render_pdf_pages(), render_pdf_clip()  [pymupdf]")

In [ ]:
# =========================================================================
# CELL 19 — ORIENTATION CORRECTION
# =========================================================================
# A VLM call just to ask "which way up?" costs a full image prefill for four output tokens.
# This cascade costs ~20 ms: tesseract OSD if installed -> projection-profile axis test ->
# baseline-asymmetry test for the 180 degree case.
WORK_SIDE = 1000       # fixed measurement scale: blur variance is meaningless across sizes


def _work_gray(img: Image.Image, side: int = WORK_SIDE) -> np.ndarray:
    g = img.convert("L")
    if max(g.size) > side:
        s = side / max(g.size)
        g = g.resize((max(1, int(g.width * s)), max(1, int(g.height * s))), Image.BILINEAR)
    return np.asarray(g, dtype=np.uint8)


def _axis_score(binv: np.ndarray) -> float:
    """>1 = text lines run horizontally (0/180); <1 = vertically (90/270)."""
    row, col = binv.sum(axis=1).astype(np.float32), binv.sum(axis=0).astype(np.float32)
    rv = row.var() / (row.mean() ** 2 + 1e-6)
    cvv = col.var() / (col.mean() ** 2 + 1e-6)
    return float((rv + 1e-6) / (cvv + 1e-6))


def _updown_score(binv: np.ndarray) -> float:
    """Ink-mass asymmetry within text bands. Positive = upright.

    Capitals and ascenders outnumber descenders in Latin and Cyrillic, and Arabic also carries
    most of its mass above the baseline, so a band's ink centroid sits above its geometric
    centre when the page is upright."""
    row = binv.sum(axis=1).astype(np.float32)
    if row.max() <= 0:
        return 0.0
    thr = row.mean() + 0.3 * row.std()
    bands, start = [], None
    for i, v in enumerate(row):
        if v > thr and start is None:
            start = i
        elif v <= thr and start is not None:
            if i - start >= 3:
                bands.append((start, i))
            start = None
    if start is not None and len(row) - start >= 3:
        bands.append((start, len(row)))
    if len(bands) < 3:
        return 0.0
    scores = []
    for a, b in bands[:60]:
        seg = row[a:b]
        if seg.sum() <= 0:
            continue
        idx = np.arange(len(seg), dtype=np.float32)
        scores.append(0.5 - float((seg * idx).sum() / seg.sum()) / max(1, len(seg) - 1))
    return float(np.mean(scores) * 2.0) if scores else 0.0


def detect_orientation(img: Image.Image, cfg: Config = CFG) -> Dict[str, Any]:
    out = {"rotation": 0, "method": "disabled", "margin": 1.0, "osd_conf": None}
    if not cfg.ENABLE_ORIENTATION_CORRECTION or cv2 is None:
        return out
    if pytesseract is not None:
        try:
            small = img.copy()
            small.thumbnail((1000, 1000))
            osd = pytesseract.image_to_osd(small, output_type=TessOutput.DICT, config="--psm 0")
            conf = float(osd.get("orientation_conf", 0.0))
            out["osd_conf"] = conf
            if conf >= 2.0:
                return {**out, "rotation": int(osd.get("rotate", 0)) % 360,
                        "method": "tesseract_osd"}
        except Exception:
            pass
    gray = _work_gray(img, 800)
    binv = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    axis = _axis_score(binv)
    candidates = (0, 180) if axis >= 1.0 else (90, 270)
    scored = {d: _updown_score(np.ascontiguousarray(np.rot90(binv, k=d // 90) if d else binv))
              for d in candidates}
    best = max(scored, key=lambda d: scored[d])
    out.update({"rotation": int(best), "method": "cv_projection",
                "margin": round(abs(scored[candidates[0]] - scored[candidates[1]]), 4),
                "axis_score": round(float(axis), 3)})
    return out


def correct_orientation(img: Image.Image, cfg: Config = CFG) -> Tuple[Image.Image, Dict[str, Any]]:
    info = detect_orientation(img, cfg)
    if info["rotation"]:
        img = img.rotate(info["rotation"], expand=True)      # PIL rotates counter-clockwise
    return img, info


print("CELL 19 ready: detect_orientation(), correct_orientation()")

In [ ]:
# =========================================================================
# CELL 20 — IMAGE PREPROCESSING  (modular; applied only when measured as needed)
# =========================================================================
def estimate_noise_sigma(gray: np.ndarray) -> float:
    """Immerkaer estimator: one 3x3 convolution; separates sensor noise from real structure."""
    if cv2 is None or gray.size == 0 or min(gray.shape) < 5:
        return 0.0
    M = np.array([[1, -2, 1], [-2, 4, -2], [1, -2, 1]], dtype=np.float32)
    conv = cv2.filter2D(gray.astype(np.float32), -1, M)
    h, w = gray.shape
    return round(float(np.abs(conv).sum() * math.sqrt(0.5 * math.pi) /
                       (6.0 * (w - 2) * (h - 2))), 2)


def estimate_text_height_px(gray: np.ndarray) -> float:
    """Median height of text-like connected components -- the metric that actually predicts OCR
    failure. Nominal DPI says nothing about how large the glyphs are."""
    if cv2 is None or gray.size == 0:
        return 0.0
    binv = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    n, _, stats, _ = cv2.connectedComponentsWithStats(binv, connectivity=8)
    H = gray.shape[0]
    hs = [stats[i][3] for i in range(1, n)
          if 3 <= stats[i][3] <= max(8, H * 0.08) and 1 <= stats[i][2] <= H * 0.15
          and stats[i][4] >= 6]
    return round(float(np.median(hs)), 1) if len(hs) >= 8 else 0.0


def analyze_page_quality(rendered: RenderedPage, cfg: Config = CFG) -> Dict[str, Any]:
    """Measure the page. No GPU, ~30 ms. Drives preprocessing, tiling and diagnostics."""
    img = rendered.image
    gray = _work_gray(img)
    scale_back = max(img.size) / max(gray.shape) if max(gray.shape) else 1.0
    lap = float(cv2.Laplacian(gray, cv2.CV_64F).var()) if cv2 is not None else \
        float(np.diff(gray.astype(np.float32), axis=1).var())
    p5, p95 = np.percentile(gray, [5, 95])
    contrast = float((p95 - p5) / 255.0)
    noise = estimate_noise_sigma(gray)
    text_h_full = round(estimate_text_height_px(gray) * scale_back, 1)
    ink = 0.0
    if cv2 is not None:
        binv = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
        ink = round(float((binv > 0).mean()), 4)
    g32 = gray.astype(np.float32)
    bh, bw = max(1, gray.shape[0] // 16), max(1, gray.shape[1] // 16)
    blocks = [g32[i:i + bh, j:j + bw].mean() for i in range(0, gray.shape[0] - bh + 1, bh)
              for j in range(0, gray.shape[1] - bw + 1, bw)]
    uniformity = round(float(np.std(blocks) / 255.0), 4) if blocks else 0.0

    blur_score = float(np.clip((lap - cfg.BLUR_VAR_POOR) /
                               max(1e-6, cfg.BLUR_VAR_GOOD - cfg.BLUR_VAR_POOR), 0, 1))
    contrast_score = float(np.clip(contrast / cfg.CONTRAST_LOW, 0, 1))
    noise_score = float(np.clip(1 - noise / max(1e-6, 2 * cfg.NOISE_SIGMA_HIGH), 0, 1))

    # Text height AFTER the page is scaled to MAX_IMAGE_DIMENSION: this is what the model
    # actually sees, and it is the number that decides whether tiling is required.
    view_scale = min(1.0, cfg.MAX_IMAGE_DIMENSION / max(rendered.width, rendered.height))
    text_h_view = round(text_h_full * view_scale, 1)

    q = {"page_number": rendered.page_number, "width": rendered.width, "height": rendered.height,
         "render_dpi": rendered.render_dpi,
         "native_image_px": list(rendered.native_image_px) if rendered.native_image_px else None,
         "text_height_px_full": text_h_full, "text_height_px_in_view": text_h_view,
         "blur_laplacian_var": round(lap, 1), "blur_score": round(blur_score, 3),
         "contrast_raw": round(contrast, 3), "contrast_score": round(contrast_score, 3),
         "brightness": round(float(gray.mean() / 255.0), 3),
         "noise_sigma": noise, "noise_score": round(noise_score, 3),
         "illumination_uniformity": uniformity, "ink_ratio": ink,
         "visual_clarity": round(0.4 * blur_score + 0.3 * contrast_score + 0.3 * noise_score, 3)}
    # Both conditions required: a small ID card on white A4 has a genuinely tiny ink ratio.
    q["is_blank"] = bool(ink < cfg.INK_RATIO_BLANK and text_h_full <= 0)
    q["needs_contrast_enhancement"] = bool(cfg.ENABLE_CONTRAST_ENHANCEMENT and
                                           (contrast < cfg.CONTRAST_LOW or
                                            uniformity > cfg.ILLUMINATION_POOR))
    q["needs_denoise"] = bool(cfg.ENABLE_DENOISE and noise > cfg.NOISE_SIGMA_HIGH)
    q["needs_tiling"] = bool(0 < text_h_view < cfg.MIN_TEXT_HEIGHT)
    q["quality"] = ("blank" if q["is_blank"] else
                    "good" if q["visual_clarity"] >= 0.75 else
                    "fair" if q["visual_clarity"] >= 0.45 else "poor")
    return q


def _clahe(img: Image.Image, clip: float = 2.0, tiles: Tuple[int, int] = (8, 8)) -> Image.Image:
    g = np.asarray(img.convert("L"))
    return Image.fromarray(cv2.createCLAHE(clipLimit=clip,
                                           tileGridSize=tiles).apply(g)).convert("RGB")


def _flatten_illumination(img: Image.Image) -> Image.Image:
    """Divide out a heavily blurred copy: removes shadows and vignetting without touching strokes."""
    g = np.asarray(img.convert("L"), dtype=np.float32)
    bg = cv2.GaussianBlur(g, (0, 0), sigmaX=max(g.shape) / 30.0)
    return Image.fromarray(np.clip(g / (bg + 1e-3) * float(np.median(bg)), 0, 255)
                           .astype(np.uint8)).convert("RGB")


def _unsharp(img: Image.Image, amount: float = 1.5) -> Image.Image:
    """Mild unsharp mask: raises edge contrast on an already high-resolution crop. It is a linear
    filter -- it does not manufacture glyph shapes, and it is never used to make an unreadable
    character 'readable'. Every applied operation is recorded with the result."""
    g = np.asarray(img.convert("L"))
    blur = cv2.GaussianBlur(g, (0, 0), 1.2)
    return Image.fromarray(cv2.addWeighted(g, amount, blur, 1 - amount, 0)).convert("RGB")


def _denoise(img: Image.Image) -> Image.Image:
    """Edge preserving ONLY. Median/Gaussian blur eats thin strokes and Arabic diacritics."""
    return Image.fromarray(cv2.bilateralFilter(np.asarray(img.convert("L")), 5, 45, 45)
                           ).convert("RGB")


def _resize_max(img: Image.Image, max_side: int) -> Image.Image:
    if max(img.size) <= max_side:
        return img
    s = max_side / max(img.size)
    return img.resize((max(1, int(img.width * s)), max(1, int(img.height * s))), Image.LANCZOS)


def preprocess_page(img: Image.Image, quality: Dict[str, Any], cfg: Config = CFG,
                    max_dimension: Optional[int] = None,
                    already_oriented: bool = False) -> Tuple[Image.Image, Dict[str, Any]]:
    """Geometry -> resize -> photometry ON THE SMALL IMAGE.

    Order matters for speed: filtering at full 8.7 MP and then downscaling spends seconds on
    pixels that are about to be discarded."""
    max_dimension = max_dimension or cfg.MAX_IMAGE_DIMENSION
    t0 = time.perf_counter()
    applied: List[str] = []
    meta: Dict[str, Any] = {"input_size": list(img.size), "rotation": 0, "skew_deg": 0.0}

    if not already_oriented:
        img, oinfo = correct_orientation(img, cfg)
        meta["orientation"] = oinfo
        meta["rotation"] = oinfo["rotation"]
        if oinfo["rotation"]:
            applied.append(f"rotate_{oinfo['rotation']}")

    if cfg.ENABLE_DESKEW and cv2 is not None:
        gray = _work_gray(img, 1000)
        binv = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
        binv = cv2.dilate(binv, cv2.getStructuringElement(cv2.MORPH_RECT, (25, 3)))
        coords = cv2.findNonZero(binv)
        if coords is not None and len(coords) >= 80:
            ang = cv2.minAreaRect(coords)[-1]
            ang = ang + 90 if ang < -45 else (ang - 90 if ang > 45 else ang)
            if cfg.DESKEW_MIN_DEG <= abs(ang) <= cfg.DESKEW_MAX_DEG:
                a = np.asarray(img)
                h, w = a.shape[:2]
                M = cv2.getRotationMatrix2D((w / 2, h / 2), ang, 1.0)
                cos, sin = abs(M[0, 0]), abs(M[0, 1])
                nw, nh = int(h * sin + w * cos), int(h * cos + w * sin)
                M[0, 2] += nw / 2 - w / 2
                M[1, 2] += nh / 2 - h / 2
                img = Image.fromarray(cv2.warpAffine(a, M, (nw, nh), flags=cv2.INTER_CUBIC,
                                                     borderMode=cv2.BORDER_REPLICATE))
                meta["skew_deg"] = round(float(ang), 2)
                applied.append(f"deskew_{ang:+.2f}")

    img = _resize_max(img, max_dimension)
    applied.append(f"resize_max_{max_dimension}")

    if cv2 is not None:
        if quality.get("needs_contrast_enhancement"):
            if quality.get("illumination_uniformity", 0) > cfg.ILLUMINATION_POOR:
                img = _flatten_illumination(img)
                applied.append("flatten_illumination")
            img = _clahe(img)
            applied.append("clahe")
        if quality.get("needs_denoise"):
            img = _denoise(img)
            applied.append("denoise")

    meta.update({"applied": applied, "output_size": list(img.size),
                 "approx_visual_tokens": int(img.width * img.height / 784),
                 "preprocessing_time": round(time.perf_counter() - t0, 3)})
    return img, meta


print("CELL 20 ready: analyze_page_quality(), preprocess_page()")

In [ ]:
# =========================================================================
# CELL 21 — PAGE CACHE  (PDF opened once, pages rendered once, handle reused)
# =========================================================================
# The previous version called pymupdf.open() again for every ROI clip: once per title, once per
# tile retry, once per MRZ attempt. Each open re-parses the document. Here one PageCache per PDF
# holds the open handle and the rendered pages, and every downstream consumer reuses them.
@dataclass
class CachedPage:
    page_number: int
    image: Image.Image                     # full-resolution render, rendered exactly once
    width: int
    height: int
    render_dpi: int
    render_time: float
    pdf_rect: Tuple[float, float, float, float]
    native_image_px: Optional[Tuple[int, int]] = None
    text_layer: str = ""                   # embedded text, empty for scanned PDFs
    has_text_layer: bool = False
    view: Optional[Image.Image] = None     # preprocessed view, computed once
    view_meta: Optional[Dict[str, Any]] = None
    quality: Optional[Dict[str, Any]] = None
    mrz_candidates: List[Any] = field(default_factory=list)   # cached by the planner


class PageCache:
    """Open a PDF once; render, preprocess and analyse each page once; reuse everywhere.

    Also exposes clip re-rendering through the SAME open handle, which is how MRZ, title and
    handwriting crops get real resolution without reopening the file."""

    def __init__(self, pdf_path: Path, cfg: Config = CFG):
        self.pdf_path = Path(pdf_path)
        self.cfg = cfg
        self.doc = None
        self.pages: List[CachedPage] = []
        self._preproc_variants: Dict[Tuple, Image.Image] = {}

    def __enter__(self) -> "PageCache":
        if pymupdf is None:
            raise RuntimeError("pymupdf is required")
        self.doc = pymupdf.open(str(self.pdf_path))
        n = min(self.doc.page_count, self.cfg.MAX_PAGES_PER_PDF)
        for i in range(n):
            with PROF.stage("pdf_render"):
                page = self.doc[i]
                t0 = time.perf_counter()
                pix = page.get_pixmap(dpi=self.cfg.PDF_RENDER_DPI, colorspace=pymupdf.csRGB,
                                      alpha=False)
                img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples).copy()
                rt = time.perf_counter() - t0
                r = page.rect
                text = ""
                if self.cfg.USE_PDF_TEXT_LAYER:
                    try:
                        text = page.get_text("text") or ""
                    except Exception:
                        text = ""
                imgs = page.get_images(full=True)
                native = max(((x[2], x[3]) for x in imgs), key=lambda wh: wh[0] * wh[1]) \
                    if imgs else None
                self.pages.append(CachedPage(
                    page_number=i + 1, image=img, width=pix.width, height=pix.height,
                    render_dpi=self.cfg.PDF_RENDER_DPI, render_time=round(rt, 3),
                    pdf_rect=(r.x0, r.y0, r.x1, r.y1), native_image_px=native,
                    text_layer=text, has_text_layer=len(text.strip()) >= 40))
        return self

    def __exit__(self, *exc):
        if self.doc is not None:
            self.doc.close()
            self.doc = None
        self._preproc_variants.clear()
        return False

    # ------------------------------------------------------------------
    def view(self, page: CachedPage) -> Tuple[Image.Image, Dict[str, Any], Dict[str, Any]]:
        """Quality + preprocessing, computed ONCE per page and cached on the page object."""
        if page.view is None:
            with PROF.stage("page_analysis"):
                rendered_like = RenderedPage(page.page_number, page.image, page.width,
                                             page.height, page.render_dpi, page.render_time,
                                             page.pdf_rect, page.native_image_px)
                page.quality = analyze_page_quality(rendered_like, self.cfg)
            with PROF.stage("preprocess"):
                page.view, page.view_meta = preprocess_page(page.image, page.quality, self.cfg)
        return page.view, page.view_meta, page.quality

    def clip(self, page_number: int, rect: Tuple[float, float, float, float],
             dpi: int) -> Optional[Image.Image]:
        """Re-render a rectangle at high DPI through the ALREADY OPEN handle."""
        if self.doc is None:
            return None
        try:
            with PROF.stage("roi_render"):
                page = self.doc[page_number - 1]
                r = pymupdf.Rect(*rect) & page.rect
                if r.is_empty or r.width < 1 or r.height < 1:
                    return None
                zoom = dpi / 72.0
                pix = page.get_pixmap(matrix=pymupdf.Matrix(zoom, zoom), clip=r,
                                      colorspace=pymupdf.csRGB, alpha=False)
                return Image.frombytes("RGB", (pix.width, pix.height), pix.samples).copy()
        except Exception as exc:
            log.debug("clip failed p%s: %s", page_number, exc)
            return None

    def region(self, page: CachedPage, box: Tuple[int, int, int, int], dpi: int,
               max_width: int, pad: float = 0.0) -> Dict[str, Any]:
        """Highest-resolution crop for a region, via the open handle; pixel upscale as fallback."""
        view, meta, _ = self.view(page)
        W, H = view.size
        if pad:
            px, py = int(pad * W), int(pad * H)
            box = (max(0, box[0] - px), max(0, box[1] - py),
                   min(W, box[2] + px), min(H, box[3] + py))
        box = (max(0, box[0]), max(0, box[1]), min(W, box[2]), min(H, box[3]))
        img, source = None, None
        deskewed = abs(meta.get("skew_deg", 0.0)) >= self.cfg.DESKEW_MIN_DEG
        if not deskewed:
            rect = _box_to_pdf_rect(box, (W, H), page.pdf_rect, meta.get("rotation", 0))
            if rect:
                clip = self.clip(page.page_number, rect, dpi)
                if clip is not None and clip.width >= 150:
                    rot = meta.get("rotation", 0) % 360
                    if rot:
                        clip = clip.rotate(rot, expand=True)
                    img, source = clip, "pdf_clip_rerender"
        if img is None:
            crop = view.crop(box)
            f = max(1.0, min(4.0, max_width / max(1, crop.width)))
            img = crop.resize((int(crop.width * f), int(crop.height * f)), Image.LANCZOS)
            source = "pixel_upscale"
        if img.width > max_width:
            s = max_width / img.width
            img = img.resize((max_width, max(1, int(img.height * s))), Image.LANCZOS)
        return {"image": img, "source": source, "bbox": list(box),
                "dimensions": [img.width, img.height],
                "approx_visual_tokens": int(img.width * img.height / 784)}


def text_layer_title(page: CachedPage, max_lines: int = 12) -> Optional[str]:
    """Heading text from the embedded text layer, when the PDF has one.

    Used ONLY for document classification and region hints (section 19). It never substitutes for
    visual OCR of a field value, and a scanned page simply has no text layer, so the visual path
    runs as before."""
    if not page.has_text_layer:
        return None
    lines = [l.strip() for l in page.text_layer.splitlines() if l.strip()][:max_lines]
    return " | ".join(lines) if lines else None


def text_layer_mrz_hint(page: CachedPage) -> Optional[str]:
    """MRZ-looking lines in the text layer: a free hint about WHICH page carries the MRZ.
    The value is still read visually from a high-resolution crop -- this only guides detection."""
    if not page.has_text_layer:
        return None
    lines = [l.strip().replace(" ", "") for l in page.text_layer.splitlines() if l.strip()]
    hits = [l for l in lines if MRZ_LINE_RX.match(l) and l.count("<") >= 3]
    return "\n".join(hits[:3]) if len(hits) >= 2 else None


print("CELL 21 ready: PageCache (open once, render once, clip through the same handle)")

In [ ]:
# =========================================================================
# CELL 22 — REGION OF INTEREST EXTRACTION (shared by MRZ, names, titles)
# =========================================================================
# One mechanism serves every targeted crop: map the region back to PDF coordinates and
# RE-RENDER it from the vector source at high DPI. Upscaling a crop of an already-downscaled
# page interpolates pixels that were discarded; re-rendering recovers real detail.
def _box_to_pdf_rect(box: Tuple[int, int, int, int], view_size: Tuple[int, int],
                     pdf_rect: Tuple[float, float, float, float],
                     rotation: int) -> Optional[Tuple[float, float, float, float]]:
    """Map a box in the ORIENTED view back to PDF points. Exact only for 0/90/180/270."""
    W, H = view_size
    x0, y0, x1, y1 = box
    fx0, fy0, fx1, fy1 = x0 / W, y0 / H, x1 / W, y1 / H
    r = rotation % 360
    if r == 0:
        a, b, c, d = fx0, fy0, fx1, fy1
    elif r == 90:
        a, b, c, d = fy0, 1 - fx1, fy1, 1 - fx0
    elif r == 180:
        a, b, c, d = 1 - fx1, 1 - fy1, 1 - fx0, 1 - fy0
    elif r == 270:
        a, b, c, d = 1 - fy1, fx0, 1 - fy0, fx1
    else:
        return None
    px0, py0, px1, py1 = pdf_rect
    pw, ph = px1 - px0, py1 - py0
    return (px0 + a * pw, py0 + b * ph, px0 + c * pw, py0 + d * ph)


def extract_region(box: Tuple[int, int, int, int], view_img: Image.Image,
                   rendered: "RenderedPage", pdf_path: Path, view_meta: Dict[str, Any],
                   dpi: int, max_width: int, pad: float = 0.0,
                   cfg: Config = CFG) -> Dict[str, Any]:
    """Highest-resolution image obtainable for a region, plus where it came from."""
    W, H = view_img.size
    if pad:
        px, py = int(pad * W), int(pad * H)
        box = (max(0, box[0] - px), max(0, box[1] - py),
               min(W, box[2] + px), min(H, box[3] + py))
    box = (max(0, box[0]), max(0, box[1]), min(W, box[2]), min(H, box[3]))
    img, source = None, None

    # A deskew warp is not a rectangle mapping, so the clip path is only exact without it.
    deskewed = abs(view_meta.get("skew_deg", 0.0)) >= cfg.DESKEW_MIN_DEG
    if not deskewed and pymupdf is not None:
        rect = _box_to_pdf_rect(box, (W, H), rendered.pdf_rect, view_meta.get("rotation", 0))
        if rect:
            clip = render_pdf_clip(pdf_path, rendered.page_number, rect, dpi)
            if clip is not None and clip.width >= 150:
                rot = view_meta.get("rotation", 0) % 360
                if rot:
                    clip = clip.rotate(rot, expand=True)
                img, source = clip, "pdf_clip_rerender"
    if img is None:
        crop = view_img.crop(box)
        f = max(1.0, min(4.0, max_width / max(1, crop.width)))
        img = crop.resize((int(crop.width * f), int(crop.height * f)), Image.LANCZOS)
        source = "pixel_upscale"
    if img.width > max_width:
        s = max_width / img.width
        img = img.resize((max_width, max(1, int(img.height * s))), Image.LANCZOS)
    return {"image": img, "source": source, "bbox": list(box),
            "dimensions": [img.width, img.height],
            "approx_visual_tokens": int(img.width * img.height / 784)}


def page_tiles(view_img: Image.Image, n: int = 2) -> List[Tuple[int, int, int, int]]:
    """Horizontal tiles with a small overlap. A half page re-rendered at the same max dimension
    carries ~2x the linear resolution: this is how small print is recovered without sending an
    enormous full-page image. Layout-agnostic: no hard-coded field coordinates."""
    W, H = view_img.size
    out, step = [], H / max(1, n)
    for i in range(n):
        y0 = max(0, int(i * step - 0.06 * step))
        y1 = min(H, int((i + 1) * step + 0.06 * step))
        out.append((0, y0, W, y1))
    return out


def title_band(view_img: Image.Image, fraction: float = 0.34) -> Tuple[int, int, int, int]:
    W, H = view_img.size
    return (0, 0, W, int(H * fraction))


print("CELL 22 ready: extract_region(), page_tiles(), title_band()")

In [ ]:
# =========================================================================
# CELL 23 — MRZ CANDIDATE DETECTION  (runs on EVERY page, never assumes page 2)
# =========================================================================
# The MRZ is located SPATIALLY first, then read. Detection is texture-based, so it survives
# different passport layouts, scanners, margins and small positional shifts.
#
# Pipeline: blackhat (dark text on light ground) -> Sobel-x (MRZ glyphs are stroke-dense)
#           -> closing (characters into lines) -> closing (lines into one block) -> contours.
#
# What must NOT be accepted as an MRZ, and the measurement that rejects it:
#   photograph / portrait      -> colour saturation and edge-orientation entropy are high,
#                                 horizontal line structure is absent  (saturation, line_count)
#   Arabic identity text       -> cursive with dots above/below gives high vertical-extent
#                                 variance and a ragged right edge     (height_uniformity, fill)
#   normal printed fields      -> "NOM", "DATE DE NAISSANCE" and their values are short and
#                                 interrupted by whitespace            (width_frac, fill)
#   a document number printed
#   elsewhere on the card      -> single short run, wrong aspect ratio (aspect, width_frac)
#   borders, holograms,
#   watermarks, guilloche      -> low fill after thresholding, no repeating glyph pitch
#                                 (fill, pitch_regularity)
# A candidate must pass ALL geometric gates and then clear MRZ_MIN_SCORE. The decisive
# confirmation is structural: the transcription must look like MRZ lines (CELL 19).
@dataclass
class MRZCandidate:
    page_number: int
    bbox: Tuple[int, int, int, int]
    score: float
    n_lines: int
    width_frac: float
    rel_y: float
    fill: float
    height_uniformity: float
    pitch_regularity: float
    saturation: float
    method: str
    rejected_reason: Optional[str] = None


def _glyph_stats(roi_bin: np.ndarray) -> Tuple[float, float]:
    """(height_uniformity, pitch_regularity) of the connected components inside a band.

    MRZ is OCR-B: one fixed glyph height and one fixed pitch. Arabic, proportional print and
    decorative elements all score low on at least one of these."""
    if cv2 is None or roi_bin.size == 0:
        return 0.0, 0.0
    n, _, stats, cents = cv2.connectedComponentsWithStats(roi_bin, connectivity=8)
    hs, xs = [], []
    for i in range(1, n):
        x, y, w, h, area = stats[i]
        if h >= 3 and area >= 4 and w <= roi_bin.shape[1] * 0.1:
            hs.append(h)
            xs.append(cents[i][0])
    if len(hs) < 8:
        return 0.0, 0.0
    hs = np.asarray(hs, dtype=np.float32)
    height_uniformity = float(np.clip(1.0 - hs.std() / (hs.mean() + 1e-6), 0, 1))
    xs = np.sort(np.asarray(xs, dtype=np.float32))
    gaps = np.diff(xs)
    gaps = gaps[(gaps > 1) & (gaps < roi_bin.shape[1] * 0.08)]
    pitch_regularity = 0.0 if gaps.size < 6 else \
        float(np.clip(1.0 - gaps.std() / (gaps.mean() + 1e-6), 0, 1))
    return round(height_uniformity, 3), round(pitch_regularity, 3)


def detect_mrz_candidates(view_img: Image.Image, page_number: int,
                          cfg: Config = CFG) -> List[MRZCandidate]:
    """Return ranked MRZ candidates for ONE page. Empty list is a legitimate answer."""
    if cv2 is None:
        return []
    W0, H0 = view_img.size
    scale = min(1.0, 1200 / W0)
    small = view_img.resize((max(1, int(W0 * scale)), max(1, int(H0 * scale))), Image.BILINEAR)
    gray = np.asarray(small.convert("L"), dtype=np.uint8)
    hsv_sat = np.asarray(small.convert("HSV"), dtype=np.uint8)[:, :, 1]
    H, W = gray.shape

    rect_k = cv2.getStructuringElement(cv2.MORPH_RECT, (max(9, W // 60), 5))
    sq_k = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    blackhat = cv2.morphologyEx(cv2.GaussianBlur(gray, (3, 3), 0), cv2.MORPH_BLACKHAT, rect_k)
    grad = np.absolute(cv2.Sobel(blackhat, cv2.CV_32F, 1, 0, ksize=-1))
    mn, mx = float(grad.min()), float(grad.max())
    grad = ((grad - mn) / (mx - mn + 1e-6) * 255).astype("uint8")
    grad = cv2.morphologyEx(grad, cv2.MORPH_CLOSE, rect_k)
    thresh = cv2.threshold(grad, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)[1]
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, sq_k)
    thresh = cv2.erode(thresh, None, iterations=2)
    thresh[:, :int(0.02 * W)] = 0                 # suppress page borders
    thresh[:, int(0.98 * W):] = 0

    ink = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    cnts, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cands: List[MRZCandidate] = []
    for c in cnts:
        x, y, w, h = cv2.boundingRect(c)
        reason = None
        ar, wf = w / float(max(1, h)), w / float(W)
        if h < 4:
            continue
        if wf < 0.45:
            reason = "too_narrow: a document number or a printed field, not a full-width MRZ"
        elif ar < 4.0:
            reason = "aspect_ratio: a block, not machine-readable lines"
        elif h > 0.30 * H:
            reason = "too_tall: a photo or a text block, not an MRZ band"
        roi_t = thresh[y:y + h, x:x + w]
        roi_ink = ink[y:y + h, x:x + w]
        fill = float((roi_t > 0).mean()) if roi_t.size else 0.0
        sat = float(hsv_sat[y:y + h, x:x + w].mean()) / 255.0 if roi_t.size else 0.0
        if reason is None and fill < 0.25:
            reason = "low_fill: decorative or security pattern, not dense glyph rows"
        if reason is None and sat > 0.35:
            reason = "colourful: portrait or security print, MRZ is monochrome"

        prof = (roi_t > 0).sum(axis=1).astype(np.float32)
        lines, run = 0, False
        for v in prof:
            hot = v > 0.35 * w
            if hot and not run:
                lines, run = lines + 1, True
            elif not hot:
                run = False
        hu, pr = _glyph_stats(roi_ink)
        if reason is None and lines > 4:
            reason = "too_many_lines: a paragraph of printed text, not an MRZ"
        if reason is None and hu < 0.45:
            reason = ("glyph heights not uniform: proportional or cursive script "
                      "(e.g. Arabic identity text), not OCR-B")

        rel_y = (y + h / 2) / H
        score = (min(ar / 20.0, 1.0) * 0.20 + wf * 0.20 + fill * 0.15 +
                 hu * 0.20 + pr * 0.15 +
                 (0.05 if lines in (2, 3) else 0.0) +
                 (0.05 if rel_y > 0.55 else 0.0))   # bottom is typical, never required
        cand = MRZCandidate(
            page_number=page_number,
            bbox=(int(x / scale), int(y / scale), int((x + w) / scale), int((y + h) / scale)),
            score=round(float(score), 3), n_lines=int(lines), width_frac=round(wf, 3),
            rel_y=round(float(rel_y), 3), fill=round(fill, 3), height_uniformity=hu,
            pitch_regularity=pr, saturation=round(sat, 3), method="morphology+glyph_stats",
            rejected_reason=reason)
        cands.append(cand)

    accepted = [c for c in cands if c.rejected_reason is None and c.score >= cfg.MRZ_MIN_SCORE]
    accepted.sort(key=lambda c: -c.score)
    return accepted[:cfg.MRZ_CANDIDATES_PER_PAGE]


def localize_mrz(all_candidates: List[MRZCandidate]) -> Optional[MRZCandidate]:
    """Choose the MRZ across ALL pages of the document.

    The MRZ may sit on page 1 (ID cards, many passports photographed as a single page) or on
    page 2 (a passport whose data page is scanned second). Selection is by evidence, not by
    page number."""
    if not all_candidates:
        return None
    return sorted(all_candidates, key=lambda c: -c.score)[0]


print("CELL 23 ready: detect_mrz_candidates() [every page], localize_mrz()")

In [ ]:
# =========================================================================
# CELL 24 — MRZ CROP, PREPROCESSING VARIANTS AND UPSCALING
# =========================================================================
# The MRZ gets its own preprocessing path, different from a normal page: the band is narrow, the
# glyphs are fixed-pitch, and horizontal resolution is what decides legibility.
def _mrz_deskew(img: Image.Image, cfg: Config = CFG) -> Tuple[Image.Image, float]:
    """Small-angle correction using the band's own text rows."""
    if cv2 is None:
        return img, 0.0
    g = np.asarray(img.convert("L"))
    binv = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    coords = cv2.findNonZero(cv2.dilate(binv, cv2.getStructuringElement(cv2.MORPH_RECT, (31, 3))))
    if coords is None or len(coords) < 60:
        return img, 0.0
    ang = cv2.minAreaRect(coords)[-1]
    ang = ang + 90 if ang < -45 else (ang - 90 if ang > 45 else ang)
    if not (0.3 <= abs(ang) <= 8.0):
        return img, 0.0
    a = np.asarray(img)
    h, w = a.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), ang, 1.0)
    return Image.fromarray(cv2.warpAffine(a, M, (w, h), flags=cv2.INTER_CUBIC,
                                          borderMode=cv2.BORDER_REPLICATE)), round(float(ang), 2)


def preprocess_mrz(img: Image.Image, variant: str, cfg: Config = CFG
                   ) -> Tuple[Image.Image, List[str]]:
    """Named preprocessing variants. Each retry uses a DIFFERENT one -- never the same input
    twice. The original crop is always preserved by the caller for comparison."""
    ops: List[str] = []
    if cv2 is None:
        return img, ops
    out = img
    if variant in ("standard", "aggressive", "binarised"):
        out, ang = _mrz_deskew(out, cfg)
        if ang:
            ops.append(f"deskew_{ang:+.2f}")
    g = np.asarray(out.convert("L"))
    if variant == "standard":
        g = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 4)).apply(g)
        ops.append("clahe")
    elif variant == "aggressive":
        bg = cv2.GaussianBlur(g.astype(np.float32), (0, 0), sigmaX=max(g.shape) / 12.0)
        g = np.clip(g / (bg + 1e-3) * float(np.median(bg)), 0, 255).astype(np.uint8)
        ops.append("flatten_illumination")
        g = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(16, 2)).apply(g)
        ops.append("clahe_strong")
        g = cv2.bilateralFilter(g, 5, 50, 50)
        ops.append("bilateral_denoise")
    elif variant == "binarised":
        g = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 4)).apply(g)
        g = cv2.adaptiveThreshold(g, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                  cv2.THRESH_BINARY, 31, 11)
        ops.append("adaptive_threshold")     # last resort: binarisation costs stroke detail
    if cfg.ENABLE_SHARPENING and variant != "binarised":
        blur = cv2.GaussianBlur(g, (0, 0), 1.2)
        g = cv2.addWeighted(g, 1.5, blur, -0.5, 0)
        ops.append("unsharp")                # linear filter: raises edge contrast, invents nothing
    return Image.fromarray(g).convert("RGB"), ops


def crop_mrz(candidate: MRZCandidate, view_img: Image.Image, rendered: "RenderedPage",
             pdf_path: Path, view_meta: Dict[str, Any], variant: str = "standard",
             cfg: Config = CFG) -> Dict[str, Any]:
    """Isolate the ENTIRE MRZ band and deliver it at OCR-grade resolution."""
    region = extract_region(candidate.bbox, view_img, rendered, pdf_path, view_meta,
                            cfg.REGION_RENDER_DPI, cfg.MRZ_MAX_WIDTH, pad=0.02, cfg=cfg)
    original = region["image"]

    # MRZ-specific upscaling: unlike a page, the target is a minimum CHARACTER height.
    n_lines = max(2, candidate.n_lines or 2)
    est_char = original.height / n_lines * 0.55
    img = original
    if est_char < cfg.MRZ_MIN_CHAR_HEIGHT:
        f = min(cfg.MRZ_UPSCALE_FACTOR, cfg.MRZ_MIN_CHAR_HEIGHT / max(1e-6, est_char))
        f = min(f, cfg.MRZ_MAX_WIDTH / max(1, img.width))
        if f > 1.02:
            img = img.resize((int(img.width * f), int(img.height * f)), Image.LANCZOS)
            est_char *= f
    processed, ops = preprocess_mrz(img, variant, cfg)
    return {"original": original, "image": processed, "source": region["source"],
            "bbox": region["bbox"], "variant": variant, "ops": ops,
            "crop_width": processed.width, "crop_height": processed.height,
            "estimated_char_height_px": round(est_char, 1),
            "sufficient_resolution": bool(est_char >= cfg.MRZ_MIN_CHAR_HEIGHT),
            "approx_visual_tokens": region["approx_visual_tokens"],
            "n_lines_assumed": n_lines}


print("CELL 24 ready: crop_mrz(), preprocess_mrz() [standard|aggressive|binarised]")

In [ ]:
# =========================================================================
# CELL 25 — HANDWRITING REGION PROCESSING
# =========================================================================
# Handwriting needs stroke continuity, not contrast extremes: binarisation breaks thin pen
# strokes and is never used here.
def preprocess_handwriting(img: Image.Image, variant: str = "standard",
                           cfg: Config = CFG) -> Tuple[Image.Image, List[str]]:
    ops: List[str] = []
    if cv2 is None:
        return img, ops
    g = np.asarray(img.convert("L"))
    if variant == "standard":
        g = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(g)
        ops.append("clahe")
    else:                                    # "contrast": form rules and shading removed harder
        bg = cv2.GaussianBlur(g.astype(np.float32), (0, 0), sigmaX=max(g.shape) / 16.0)
        g = np.clip(g / (bg + 1e-3) * float(np.median(bg)), 0, 255).astype(np.uint8)
        ops.append("flatten_illumination")
        g = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8)).apply(g)
        ops.append("clahe_strong")
    if cfg.ENABLE_SHARPENING:
        blur = cv2.GaussianBlur(g, (0, 0), 1.0)
        g = cv2.addWeighted(g, 1.3, blur, -0.3, 0)
        ops.append("mild_unsharp")
    return Image.fromarray(g).convert("RGB"), ops


def handwriting_region(box: Tuple[int, int, int, int], view_img: Image.Image,
                       rendered: "RenderedPage", pdf_path: Path, view_meta: Dict[str, Any],
                       variant: str = "standard", cfg: Config = CFG) -> Dict[str, Any]:
    """High-resolution crop for a handwritten field, with its own upscale factor."""
    region = extract_region(box, view_img, rendered, pdf_path, view_meta,
                            cfg.REGION_RENDER_DPI, cfg.REGION_MAX_WIDTH, pad=0.01, cfg=cfg)
    img = region["image"]
    if img.width < cfg.REGION_MAX_WIDTH:
        f = min(cfg.HANDWRITING_UPSCALE_FACTOR, cfg.REGION_MAX_WIDTH / max(1, img.width))
        if f > 1.02:
            img = img.resize((int(img.width * f), int(img.height * f)), Image.LANCZOS)
    processed, ops = preprocess_handwriting(img, variant, cfg)
    return {"original": region["image"], "image": processed, "source": region["source"],
            "bbox": region["bbox"], "variant": variant, "ops": ops,
            "dimensions": [processed.width, processed.height],
            "approx_visual_tokens": int(processed.width * processed.height / 784)}


def save_debug(customer_id: str, document: str, page: Optional[int], task: str,
               name: str, img: Image.Image, cfg: Config = CFG) -> Optional[str]:
    """debug_images/<customer_id>/<document>/page_<n>/<task>/<name>.png"""
    if not (cfg.SAVE_DEBUG_IMAGES or cfg.SAVE_DEBUG_ON_FAILURE):
        return None
    d = DIRS["debug"] / str(customer_id) / re.sub(r"[^A-Za-z0-9_]+", "_", document) / \
        (f"page_{page}" if page else "document") / task
    d.mkdir(parents=True, exist_ok=True)
    path = d / f"{name}.png"
    try:
        img.save(path)
        return str(path)
    except Exception as exc:
        log.warning("debug image save failed: %s", exc)
        return None


print("CELL 25 ready: handwriting_region(), preprocess_handwriting(), save_debug()")

In [ ]:
# =========================================================================
# CELL 26 — JSON PARSING  (raw -> parsed -> validated, kept separate)
# =========================================================================
CONF_BANDS = ("high", "medium", "low", "unreadable")
CONF_RANK = {"high": 3, "medium": 2, "low": 1, "unreadable": 0}
CONF_FLOAT = {"high": 0.92, "medium": 0.7, "low": 0.4, "unreadable": 0.0}
NULLISH = {"", "null", "none", "n/a", "na", "unreadable", "not visible", "-", "--"}


def _first_json_object(text: str) -> Optional[str]:
    if not text:
        return None
    t = re.sub(r"<think>.*?</think>", "", text, flags=re.S)
    t = re.sub(r"^```[a-zA-Z]*\s*|\s*```$", "", t.strip())
    start = t.find("{")
    if start < 0:
        return None
    depth, in_str, esc = 0, False, False
    for i in range(start, len(t)):
        c = t[i]
        if in_str:
            if esc:        esc = False
            elif c == "\\": esc = True
            elif c == '"': in_str = False
            continue
        if c == '"':   in_str = True
        elif c == "{": depth += 1
        elif c == "}":
            depth -= 1
            if depth == 0:
                return t[start:i + 1]
    return t[start:] + "}" * depth if depth > 0 else None      # tolerate a truncated tail


def _clean(v: Any, name: str = "") -> Optional[str]:
    if isinstance(v, (list, tuple)):
        v = ("\n" if name == "mrz" else " ").join(str(x) for x in v if x is not None)
    if v is None or isinstance(v, bool):
        return None
    if isinstance(v, (int, float)):
        v = str(v)
    if not isinstance(v, str):
        return None
    s = v.strip()
    return None if s.lower() in NULLISH else s


def parse_json_response(raw: str, fields: Sequence[str]) -> Tuple[Dict[str, Any], bool, Optional[str]]:
    """Coerce model output onto the requested keys. SUBTRACTIVE ONLY: a missing key becomes null,
    an unknown confidence is downgraded (never promoted), a null value is forced to 'unreadable'.
    Unparsable output fails CLOSED. The raw string is kept by the caller, untouched."""
    fields = list(fields)
    block = _first_json_object(raw)
    empty = {f: {"value": None, "confidence": "unreadable"} for f in fields}
    if block is None:
        return empty, False, "no_json_in_output"
    try:
        obj = json.loads(block)
    except json.JSONDecodeError as exc:
        try:
            obj = json.loads(re.sub(r",\s*([}\]])", r"\1", block))
        except Exception:
            return empty, False, f"json_decode_error: {exc}"
    if not isinstance(obj, dict):
        return empty, False, "not_an_object"

    lower = {str(k).strip().lower(): k for k in obj}
    out: Dict[str, Any] = {}
    for f in fields:
        key = lower.get(f) or (lower.get("machine_readable_zone") if f == "mrz" else None)
        node = obj.get(key) if key else None
        if isinstance(node, dict):
            value = _clean(node.get("value"), f)
            conf = str(node.get("confidence", "")).strip().lower()
            ttype = (str(node.get("text_type", "") or "").strip().lower() or None)
        else:
            value, conf, ttype = _clean(node, f), "", None
        if conf not in CONF_BANDS:
            conf = "low" if value is not None else "unreadable"
        if value is None:
            conf = "unreadable"
        entry: Dict[str, Any] = {"value": value, "confidence": conf}
        if ttype in ("printed", "handwritten", "mixed"):
            entry["text_type"] = ttype
        out[f] = entry
    return out, True, None


def parse_lines_response(raw: str) -> Tuple[List[str], Dict[str, Any], bool]:
    """Parse the line-transcription pass."""
    block = _first_json_object(raw)
    if block is None:
        return [], {}, False
    try:
        obj = json.loads(block)
    except Exception:
        try:
            obj = json.loads(re.sub(r",\s*([}\]])", r"\1", block))
        except Exception:
            return [], {}, False
    if not isinstance(obj, dict):
        return [], {}, False
    lines = obj.get("lines") or obj.get("text") or []
    if isinstance(lines, str):
        lines = lines.splitlines()
    lines = [str(l).strip() for l in lines if str(l).strip()]
    meta = {"scripts": obj.get("scripts"), "has_handwriting": obj.get("has_handwriting")}
    return lines, meta, True


print("CELL 26 ready: parse_json_response(), parse_lines_response()")

In [ ]:
# =========================================================================
# CELL 27 — FIELD MAPPING FROM TRANSCRIBED LINES + VALIDATION
# =========================================================================
# Mapping labelled lines to fields is deterministic and auditable. Agreement between this
# (from PASS A) and the model's own schema answer (PASS B) is an OBJECTIVE consistency signal --
# two independent reads of the same pixels -- rather than a self-reported confidence.
FIELD_LABELS: Dict[str, List[str]] = {
    "surname_latin": ["NOM", "SURNAME", "NOM DE FAMILLE", "FAMILY NAME", "APELLIDOS", "LAST NAME"],
    "given_names_latin": ["PRENOM", "PRENOMS", "GIVEN NAME", "GIVEN NAMES", "FORENAMES",
                          "FIRST NAME", "NOMBRE"],
    "date_of_birth": ["DATE DE NAISSANCE", "DATE OF BIRTH", "NE LE", "NEE LE", "GEBURTSDATUM",
                      "BIRTH DATE", "DOB"],
    "place_of_birth": ["LIEU DE NAISSANCE", "PLACE OF BIRTH", "BIRTH PLACE"],
    "nationality": ["NATIONALITE", "NATIONALITY", "NACIONALIDAD"],
    "sex": ["SEXE", "SEX", "GENDER"],
    "document_number": ["NUMERO DU DOCUMENT", "DOCUMENT NO", "PASSPORT NO", "NO DU PASSEPORT",
                        "NUMERO", "DOCUMENT NUMBER", "CARTE NO", "NO"],
    "issue_date": ["DATE DE DELIVRANCE", "DATE OF ISSUE", "DELIVRE LE", "ISSUED ON"],
    "expiry_date": ["DATE D EXPIRATION", "DATE OF EXPIRY", "EXPIRE LE", "VALABLE JUSQU AU",
                    "DATE D EXPIRATION DU DOCUMENT", "EXPIRY", "VALID UNTIL"],
    "issuing_authority": ["AUTORITE", "ISSUING AUTHORITY", "DELIVRE PAR", "AUTHORITY"],
    "personal_number": ["NUMERO PERSONNEL", "PERSONAL NO", "PERSONAL NUMBER", "CIN", "NIN",
                        "IDENTITY NUMBER"],
}
LABEL_LOOKUP = sorted(((lab, f) for f, labs in FIELD_LABELS.items() for lab in labs),
                      key=lambda x: -len(x[0]))
MRZ_LINE_RX = re.compile(r"^[A-Z0-9<]{25,50}$")


def _norm_label(s: str) -> str:
    return re.sub(r"\s+", " ", re.sub(r"[^A-Z0-9 ]+", " ", strip_accents(str(s)).upper())).strip()


def map_lines_to_fields(lines: Sequence[str]) -> Dict[str, Dict[str, Any]]:
    """Label-driven mapping. Handles 'Label: value' and a label whose value is on the next line."""
    out: Dict[str, Dict[str, Any]] = {}
    norm = [_norm_label(l) for l in lines]
    for i, (raw_line, nline) in enumerate(zip(lines, norm)):
        if not nline:
            continue
        for label, fieldname in LABEL_LOOKUP:
            if fieldname in out:
                continue
            if not nline.startswith(label):
                continue
            rest = raw_line
            m = re.search(r"[:\-]\s*", raw_line)
            if m:
                rest = raw_line[m.end():]
            else:
                rest = raw_line[len(label):] if len(raw_line) > len(label) else ""
            value = rest.strip(" .:-\t")
            if not value and i + 1 < len(lines):      # value printed on the following line
                nxt = lines[i + 1].strip()
                if nxt and not any(_norm_label(nxt).startswith(l) for l, _ in LABEL_LOOKUP):
                    value = nxt
            if value:
                out[fieldname] = {"value": value, "source_line": raw_line, "label": label}
            break
    mrz_lines = [l.strip().replace(" ", "") for l in lines
                 if MRZ_LINE_RX.match(l.strip().replace(" ", "")) and
                 l.count("<") >= 3]
    if len(mrz_lines) >= 2:
        out["mrz"] = {"value": "\n".join(mrz_lines[:3]), "source_line": "mrz_lines",
                      "label": "MRZ"}
    return out


# ------------------------------- validation --------------------------------
DATE_RX = [(re.compile(r"^(\d{2})[./\- ](\d{2})[./\- ](\d{4})$"), ("d", "m", "y")),
           (re.compile(r"^(\d{4})[./\- ](\d{2})[./\- ](\d{2})$"), ("y", "m", "d")),
           (re.compile(r"^(\d{2})[./\- ](\d{2})[./\- ](\d{2})$"), ("d", "m", "yy")),
           (re.compile(r"^(\d{2})\s?([A-Z]{3})\s?(\d{2,4})$"), ("d", "mon", "y"))]
MONTHS3 = {m: i + 1 for i, m in enumerate(["JAN", "FEB", "MAR", "APR", "MAY", "JUN", "JUL",
                                           "AUG", "SEP", "OCT", "NOV", "DEC"])}
MRZ_SPECS = {"TD1": (3, 30), "TD2": (2, 36), "TD3": (2, 44)}


def parse_date(value: Optional[str]) -> Optional[date]:
    """For COMPARISON only. The stored transcription is never replaced by this."""
    if not value:
        return None
    s = strip_accents(str(value)).strip().upper()
    for rx, order in DATE_RX:
        m = rx.match(s)
        if not m:
            continue
        g = dict(zip(order, m.groups()))
        try:
            if "y" in g:
                y = int(g["y"])
                y = y if y > 99 else (2000 + y if y < 30 else 1900 + y)
            else:
                yy = int(g["yy"])
                y = 2000 + yy if yy < 30 else 1900 + yy
            mth = MONTHS3.get(g.get("mon"), 0) or int(g.get("m", 0))
            return date(y, mth, int(g["d"]))
        except Exception:
            return None
    return None


def mrz_check_digit(s: str) -> Optional[str]:
    w, total = (7, 3, 1), 0
    for i, c in enumerate(s):
        if c == "<":      v = 0
        elif c.isdigit(): v = int(c)
        elif c.isalpha(): v = ord(c.upper()) - 55
        else:             return None
        total += v * w[i % 3]
    return str(total % 10)


def validate_mrz(value: Optional[str]) -> Dict[str, Any]:
    """Structure, charset, length and ICAO check digits. DIAGNOSTIC ONLY.

    A failing check digit proves a character was misread; it does not reveal which one or what it
    should be. Using it to choose a replacement would be hallucination wearing a checksum."""
    r: Dict[str, Any] = {"status": "absent", "format": None, "line_lengths": [], "checks": {},
                         "flags": [], "note": "diagnostic only; never modifies the OCR result"}
    if not value:
        return r
    lines = [l.strip().replace(" ", "") for l in str(value).splitlines() if l.strip()]
    r["line_lengths"] = [len(l) for l in lines]
    if any("?" in l for l in lines):
        r["flags"].append("contains_unreadable_characters")
    fmt = next((k for k, (n, ln) in MRZ_SPECS.items()
                if len(lines) == n and all(abs(len(l) - ln) <= 2 for l in lines)), None)
    r["format"] = fmt
    if fmt is None:
        r["status"] = "structure_invalid"
        r["flags"].append(f"unexpected_layout:{len(lines)}x{r['line_lengths']}")
        return r
    if any(re.search(r"[^A-Z0-9<?]", l) for l in lines):
        r["flags"].append("unexpected_characters")
    try:
        if fmt in ("TD2", "TD3"):
            l2 = lines[1]
            for name, (a, b, cd) in {"document_number": (0, 9, 9), "date_of_birth": (13, 19, 19),
                                     "expiry_date": (21, 27, 27)}.items():
                if len(l2) > cd:
                    r["checks"][name] = {"computed": mrz_check_digit(l2[a:b]), "printed": l2[cd]}
        else:
            r["checks"]["document_number"] = {"computed": mrz_check_digit(lines[0][5:14]),
                                              "printed": lines[0][14]}
            r["checks"]["date_of_birth"] = {"computed": mrz_check_digit(lines[1][0:6]),
                                            "printed": lines[1][6]}
            r["checks"]["expiry_date"] = {"computed": mrz_check_digit(lines[1][8:14]),
                                          "printed": lines[1][14]}
    except Exception as exc:
        r["flags"].append(f"checksum_error:{exc}")
    verdicts = [c["computed"] == c["printed"] for c in r["checks"].values()
                if c.get("computed") is not None]
    r["status"] = ("not_verifiable" if not verdicts else
                   "valid" if all(verdicts) else "checksum_mismatch")
    if r["status"] == "checksum_mismatch":
        r["flags"].append("checksum_mismatch:" + ",".join(
            k for k, c in r["checks"].items() if c.get("computed") != c.get("printed")))
    return r


def looks_like_mrz(value: Optional[str]) -> bool:
    """Structural confirmation that a CANDIDATE region really was an MRZ.

    This is the decisive discriminator against Arabic text, printed fields, photos and
    decoration: whatever the morphology scored, the transcription must look like MRZ lines."""
    if not value:
        return False
    lines = [l.strip().replace(" ", "") for l in str(value).splitlines() if l.strip()]
    if len(lines) < 2:
        return False
    ok = [bool(MRZ_LINE_RX.match(l)) for l in lines[:3]]
    fillers = sum(l.count("<") for l in lines)
    return sum(ok) >= 2 and fillers >= 3


def validate_field(name: str, value: Optional[str]) -> Dict[str, Any]:
    """Format checks. Flags only: no validator may change a character."""
    e: Dict[str, Any] = {"format_valid": None, "flags": []}
    if not value:
        return e
    if "?" in str(value):
        e["flags"].append("contains_unreadable_characters")
    if name in ("date_of_birth", "issue_date", "expiry_date"):
        d = parse_date(value)
        e["format_valid"] = d is not None
        if d is None:
            e["flags"].append("unrecognised_date_format")
        else:
            e["parsed_date"] = d.isoformat()
            if name == "date_of_birth":
                age = (date.today() - d).days / 365.25
                if not 0 <= age <= 120:
                    e["format_valid"] = False
                    e["flags"].append(f"implausible_age:{age:.0f}")
    elif name in ("document_number", "personal_number"):
        core = str(value).upper().replace(" ", "")
        e["format_valid"] = bool(re.fullmatch(r"[A-Z0-9<?\-/]{4,20}", core))
        risky = sorted(set(core) & set("O0I1L5S8B2ZG6"))
        if risky and re.search(r"\d", core) and re.search(r"[A-Z]", core):
            # FLAG ONLY. Replacing O with 0 here would be the pipeline guessing.
            e["flags"].append("possible_ocr_confusion:" + "".join(risky))
    elif name.endswith("_latin"):
        e["format_valid"] = bool(re.fullmatch(r"[A-Za-z '?\-\.]+", str(value)))
        if not e["format_valid"]:
            e["flags"].append("non_latin_characters_in_latin_field")
    elif name == "sex":
        e["format_valid"] = str(value).strip().upper() in {"M", "F", "X", "H", "MALE", "FEMALE",
                                                           "MASCULIN", "FEMININ"}
    return e


print("CELL 27 ready: map_lines_to_fields(), validate_mrz(), looks_like_mrz()")

In [ ]:
# =========================================================================
# CELL 28 — CONFIDENCE AND FIELD RECORDS  (visual evidence only)
# =========================================================================
# Confidence is COMPUTED from measurable evidence, never taken from the model's word alone, and
# never from database or cross-document agreement -- those are verification, not evidence.
CONFIDENCE_WEIGHTS = {
    "pass_agreement": 0.30,     # PASS A lines vs PASS B schema: two independent reads
    "visual_clarity": 0.25,     # measured page/region quality
    "character_legibility": 0.20,  # absence of '?' markers in the transcription
    "format_validity": 0.15,    # deterministic format check
    "region_resolution": 0.10,  # glyph height actually delivered to the model
}


def compute_confidence(value: Optional[str], model_band: str, agreement: Optional[float],
                       visual_clarity: float, format_valid: Optional[bool],
                       text_height: Optional[float], cfg: Config = CFG) -> Dict[str, Any]:
    """Returns {"confidence": float, "confidence_band": str, "components": {...}, "caps": [...]}"""
    if not value:
        return {"confidence": 0.0, "confidence_band": "unreadable", "components": {},
                "caps_applied": ["no_value"]}
    comps: Dict[str, Optional[float]] = {
        "pass_agreement": agreement,
        "visual_clarity": float(np.clip(visual_clarity, 0, 1)),
        "character_legibility": 0.0 if "?" in str(value) else 1.0,
        "format_validity": None if format_valid is None else (1.0 if format_valid else 0.0),
        "region_resolution": None if not text_height else
        float(np.clip(text_height / (1.5 * cfg.MIN_TEXT_HEIGHT), 0, 1)),
    }
    usable = {k: v for k, v in comps.items() if v is not None}
    total_w = sum(CONFIDENCE_WEIGHTS[k] for k in usable) or 1.0
    score = sum(CONFIDENCE_WEIGHTS[k] * v for k, v in usable.items()) / total_w

    caps: List[str] = []
    def cap(limit, why):
        nonlocal score
        if score > limit:
            score, _ = limit, caps.append(why)
    if "?" in str(value):
        cap(0.40, "unreadable_characters_present")
    if format_valid is False:
        cap(0.50, "format_invalid")
    if model_band == "low":
        cap(0.60, "model_reported_low")
    if model_band == "unreadable":
        cap(0.35, "model_reported_unreadable")
    band = ("high" if score >= 0.80 else "medium" if score >= 0.60
            else "low" if score >= 0.35 else "unreadable")
    return {"confidence": round(float(score), 3), "confidence_band": band,
            "components": comps, "caps_applied": caps}


def make_field(name: str, value: Optional[str], model_band: str, *, document: str, page: Optional[int],
               region: str, ocr_task: str, preprocessing_variant: Optional[str] = None,
               retry_number: int = 0, agreement: Optional[float] = None,
               visual_clarity: float = 0.5, text_height: Optional[float] = None,
               text_type: Optional[str] = None, raw_output: Optional[str] = None,
               cfg: Config = CFG) -> Dict[str, Any]:
    """One auditable field record: value + confidence + full provenance."""
    val = validate_field(name, value)
    conf = compute_confidence(value, model_band, agreement, visual_clarity,
                              val.get("format_valid"), text_height, cfg)
    return {"value": value,
            "confidence": conf["confidence"], "confidence_band": conf["confidence_band"],
            "confidence_components": conf["components"], "caps_applied": conf["caps_applied"],
            "model_reported_confidence": model_band,
            "source_document": document, "source_page": page, "source_region": region,
            "ocr_task": ocr_task, "preprocessing_variant": preprocessing_variant,
            "retry_number": retry_number, "text_type": text_type,
            "validation": val, "raw_model_output": (raw_output or "")[:800],
            "match_status": None}       # filled during verification; never alters "value"


def empty_field(name: str, document: str, reason: str = "not_extracted") -> Dict[str, Any]:
    return {"value": None, "confidence": 0.0, "confidence_band": "unreadable",
            "confidence_components": {}, "caps_applied": [reason],
            "model_reported_confidence": "unreadable", "source_document": document,
            "source_page": None, "source_region": None, "ocr_task": None,
            "preprocessing_variant": None, "retry_number": 0, "text_type": None,
            "validation": {"format_valid": None, "flags": []}, "raw_model_output": "",
            "match_status": None, "failure_reason": reason}


def normalize_for_comparison(value: Optional[str]) -> Optional[str]:
    """Comparison key ONLY. Never stored in place of a raw value."""
    if not value:
        return None
    s = re.sub(r"[^A-Z ]+", " ", strip_accents(str(value)).upper())
    return re.sub(r"\s+", " ", s).strip() or None


def agreement_score(a: Optional[str], b: Optional[str]) -> Optional[float]:
    """Agreement between two independent reads of the same pixels."""
    na, nb = normalize_for_comparison(a), normalize_for_comparison(b)
    if na is None or nb is None:
        return None
    if na == nb:
        return 1.0
    return round(difflib.SequenceMatcher(None, na, nb).ratio(), 3)


print("CELL 28 ready: compute_confidence(), make_field()")

In [ ]:
# =========================================================================
# CELL 29 — DOCUMENT CLASSIFICATION / TITLE VERIFICATION
# =========================================================================
# The model TRANSCRIBES the heading; Python decides MATCH / MISMATCH / UNCERTAIN.
# Asking a VLM "is this the signature card?" invites agreement -- a leading question gets a
# leading answer. Transcribe-then-compare keeps the verdict deterministic and auditable.
#
# Note: section 26 gives the expected title as "SPICIMEN DE SIGNATURE", which looks like a typo
# for "SPECIMEN". Both spellings are accepted and the title actually seen is recorded verbatim,
# so the discrepancy stays visible instead of being normalised away.
EXPECTED_TITLES: Dict[str, Dict[str, Any]] = {
    IDENTITY_DOC: {"any_of": ["PASSEPORT", "PASSPORT", "CARTE NATIONALE D IDENTITE",
                              "CARTE D IDENTITE", "TITRE DE SEJOUR", "CARTE DE SEJOUR",
                              "PERMIS DE CONDUIRE", "REPUBLIQUE", "IDENTITY CARD"],
                   "label": "identity document"},
    DOMICILE_DOC: {"any_of": ["FACTURE", "QUITTANCE", "ATTESTATION DE DOMICILE",
                              "JUSTIFICATIF DE DOMICILE", "ELECTRICITE", "RELEVE",
                              "CONTRAT DE BAIL", "AVIS D ECHEANCE", "INVOICE"],
                   "label": "proof of address"},
    CONVENTION_DOC: {"any_of": ["CONVENTION DE COMPTE", "CONVENTION COMPTE",
                                "OUVERTURE DE COMPTE", "CONDITIONS GENERALES"],
                     "label": "account convention"},
    # The expected title is SPECIMEN DE SIGNATURE. SPICIMEN is kept only as a tolerated OCR
    # variant when matching what was read -- it is no longer the expected value.
    SIGNATURE_DOC: {"any_of": ["SPECIMEN DE SIGNATURE", "SPECIMEN SIGNATURE",
                               "SPICIMEN DE SIGNATURE"],
                    "label": "SPECIMEN DE SIGNATURE"},
}
FATCA_COMPONENTS = {
    # The contiguous phrase matters: "FATCA" plus "IDENTIFICATION" as loose tokens also occurs in
    # component 2's title, which would make component 1 swallow component 2's page.
    # NO expected page count. The number of pages a component spans is DISCOVERED from the PDF
    # and reported; it is never assumed, and never used to accept or reject the component.
    "component_1": {"all_of": ["FATCA IDENTIFICATION FORMS"],
                    "label": "FATCA Identification forms(v1.0)"},
    # "<<US PERSON>>" is treated as a wildcard: the anchor phrases must both be present.
    "component_2": {"all_of": ["FORMULAIRE D IDENTIFICATION", "AU REGARD DE LA LOI FATCA"],
                    "any_of": ["SOUSCRIPTEUR", "ASSURE", "US PERSON"],
                    "label": "FORMULAIRE D'IDENTIFICATION <<US PERSON>> AU REGARD DE LA LOI "
                             "FATCA SOUSCRIPTEUR/ASSURE"},
}


def normalize_title(value: Optional[str]) -> str:
    if not value:
        return ""
    s = re.sub(r"[^A-Z0-9 ]+", " ", strip_accents(str(value)).upper())
    return re.sub(r"\s+", " ", s).strip()


def classify_title(observed: Optional[str], other: Optional[str], spec: Dict[str, Any],
                   cfg: Config = CFG) -> Dict[str, Any]:
    """Deterministic comparison. UNCERTAIN when nothing was readable -- an unreadable title is
    not a wrong document. MISMATCH only when a title WAS read and does not correspond."""
    hay = normalize_title(" | ".join([x for x in (observed, other) if x]))
    if not hay:
        return {"status": "UNCERTAIN", "observed_title": observed, "score": None,
                "reason": "title not readable"}
    missing = [p for p in spec.get("all_of", []) if normalize_title(p) not in hay]
    hits, scores = [], []
    for phrase in spec.get("any_of", []) or spec.get("all_of", []):
        n = normalize_title(phrase)
        if n and n in hay:
            hits.append(phrase)
            scores.append(1.0)
        else:
            best = max((difflib.SequenceMatcher(None, n, hay[i:i + len(n)]).ratio()
                        for i in range(0, max(1, len(hay) - len(n) + 1), 4)), default=0.0)
            scores.append(best)
    score = round(max(scores) if scores else 0.0, 3)
    if missing:
        return {"status": "MISMATCH", "observed_title": observed, "score": score,
                "threshold": cfg.TITLE_MATCH_THRESHOLD,
                "reason": "missing required phrase: " + "; ".join(missing)}
    if hits or score >= cfg.TITLE_MATCH_THRESHOLD:
        return {"status": "MATCH", "observed_title": observed, "score": score,
                "matched_phrase": hits[0] if hits else None,
                "threshold": cfg.TITLE_MATCH_THRESHOLD,
                "reason": f"phrase found: {hits[0]}" if hits else f"fuzzy score {score}"}
    return {"status": "MISMATCH", "observed_title": observed, "score": score,
            "threshold": cfg.TITLE_MATCH_THRESHOLD,
            "reason": f"no expected phrase (best {score} < {cfg.TITLE_MATCH_THRESHOLD})"}


print("CELL 29 ready: EXPECTED_TITLES, FATCA_COMPONENTS, classify_title()")

In [ ]:
# =========================================================================
# CELL — PROMPTS FOR ALGERIAN KYC DOCUMENTS
# =========================================================================
# Written for the documents that actually arrive: Algerian CNIBE / passeport / permis, French
# administrative wording with Arabic alongside, photocopied and stamped scans.
#
# Deliberate choices:
#  * Algerian names carry several components (BEN MOHAMED, ABD EL KADER, OULD KACI). The prompt
#    never says "the first word is the surname" -- it reads the labelled fields as printed.
#  * Latin fields must come from Latin characters PRINTED on the document. Algerian IDs show the
#    name in both scripts; transliterating the Arabic would fabricate a spelling.
#  * French administrative date forms (JJ/MM/AAAA, "Né(e) le") and place wording
#    (commune, daïra, wilaya) are named so the model knows what the labels mean.
#  * Nothing is corrected, completed or reordered.
ALGERIAN_CONTEXT = """These are Algerian KYC documents: French administrative wording, often with Arabic alongside, frequently photocopied, stamped, skewed or low-contrast.
Labels you may see: Nom / اللقب, Prénom(s) / الاسم, Né(e) le, Date de naissance, Lieu de naissance, Commune, Daïra, Wilaya, Nationalité (Algérienne), Sexe (M/F), N° du document, Délivré le, Expire le, Autorité de délivrance, N° d'identification nationale (NIN).
Algerian names often have several components (for example BEN MOHAMED, ABD EL KADER, OULD KACI). Transcribe every component exactly as printed, in the printed order. Do not split, merge or reorder them, and do not decide which component is the surname -- read the labelled field."""

OCR_SYSTEM = f"""You are a visual transcription engine for Algerian identity and banking documents. The image is the only source of truth.

{ALGERIAN_CONTEXT}

RULES
- Transcribe only what is visible. Never guess, infer, complete, correct or reorder.
- Latin fields: copy only Latin characters actually printed on this image. If the name appears only in Arabic, the Latin field is null -- never transliterate or translate Arabic.
- Keep dates exactly as printed (JJ/MM/AAAA or otherwise). Do not reformat.
- Mark a single unreadable character with ?. If a value cannot be read, use null.
- Never use another document, a database, or knowledge of Algerian names to fill a gap.
- JSON only. No explanation, no reasoning, no repetition."""


def identity_page_prompt(fields: Optional[Sequence[str]] = None, with_title: bool = True) -> str:
    """Flat output: ~8 output tokens per field instead of ~20 for a nested object.
    Token budget 224 -- measured against this schema, not chosen arbitrarily."""
    fields = list(fields or IDENTITY_KEYS)
    keys = ", ".join(f'"{f}"' for f in fields)
    title_line = ('\n"document_title": the printed heading (for example CARTE NATIONALE '
                  "D'IDENTITE, PASSEPORT, PERMIS DE CONDUIRE), exactly as written."
                  if with_title else "")
    pairs = ['"surname_latin": "BEN MOHAMED"', '"given_names_latin": "ABD EL KADER"',
             '"date_of_birth": "01/01/1980"', '"place_of_birth": "ORAN"',
             '"document_number": "12AB4?678"']
    if with_title:
        pairs.append('"document_title": "CARTE NATIONALE D IDENTITE"')
    example = "{" + ", ".join(pairs) + ', "unsure": ["document_number"]}'
    return f"""Read this Algerian identity document page. Report: {keys}.{title_line}

Return ONE flat JSON object: each key maps to the transcribed string, or null if it is not on this page. Add "unsure": [field names whose characters you cannot see clearly].

Shape (use the values from THIS image):
{example}

"place_of_birth" is the commune/wilaya as printed. "issuing_authority" is the daïra or wilaya office as printed. JSON only."""


MRZ_PROMPT = """This image is the machine readable zone (MRZ) of an Algerian identity document, cropped and enlarged. Read ONLY this region.

Transcribe the lines exactly: preserve every <, the character order and the line order. No spaces. Algerian passports use DZA as the issuing code; do not "fix" anything you see to match an expected pattern. Check digits must NOT be used to decide an unclear character. Mark an unreadable character with ?.

Return: {"mrz": "P<DZABEN<MOHAMED<<ABD<EL<KADER<<<<<<<<<<<<<\\n1234567<8DZA8001019M3001011<<<<<<<<<<<<<<04", "unsure": []}

with the lines of THIS image separated by \\n, or null if unreadable. JSON only."""

TITLE_PROMPT = """Transcribe the printed heading at the top of this Algerian document image.

Return: {"document_title": "SPECIMEN DE SIGNATURE"}

Copy it exactly as printed, including French accents and any unusual spelling. Do not translate or correct. Use null if nothing is readable. JSON only."""


def name_prompt(handwritten: bool = False, with_title: bool = True,
                anchor: Optional[str] = None) -> str:
    """Name extraction for the secondary documents. `anchor` names the printed label the value
    sits next to or under, found dynamically -- never a hardcoded coordinate."""
    hw = (" The name is HANDWRITTEN in Latin script: read the strokes letter by letter and mark "
          "any letter you cannot read with ?. Algerian handwritten names often have several "
          "components -- transcribe all of them in the written order.") if handwritten else ""
    where = f' The value is written next to or below the printed label "{anchor}".' \
        if anchor else ""
    pairs = ['"surname_latin": "BEN MOHAMED"', '"given_names_latin": "ABD EL KADER"',
             '"text_type": "handwritten"']
    if with_title:
        pairs.append('"document_title": "CONVENTION DE COMPTE"')
    example = "{" + ", ".join(pairs) + ', "unsure": []}'
    return f"""Read the client's name from this Algerian document image.{hw}{where}

Return ONE flat JSON object shaped like:
{example}

Use null for a value you cannot read. Only Latin characters actually written here: if the name is written only in Arabic, return null rather than transliterating. Do not correct spelling and do not use any other document. "text_type" is printed, handwritten or mixed. JSON only."""


PROMPT_TOKEN_BUDGETS = {
    "identity_page": ("MAX_NEW_TOKENS_IDENTITY", 224,
                      "12 flat fields ~8 tokens each + title + unsure list ~= 130, plus headroom"),
    "mrz":           ("MAX_NEW_TOKENS_MRZ", 160,
                      "2 MRZ lines of 44 chars ~= 90 tokens plus JSON scaffolding"),
    "name":          ("MAX_NEW_TOKENS_NAME", 128, "2 names + title + text_type"),
    "title":         ("MAX_NEW_TOKENS_TITLE", 96, "one heading string"),
    "identity_lines": ("MAX_NEW_TOKENS_LINES", 512,
                       "retry only: a full page of transcribed lines"),
}
print("CELL 30 ready: Algerian-context prompts")
for k, (name, val, why) in PROMPT_TOKEN_BUDGETS.items():
    print(f"  {k:<16} {name:<26} {val:>4}   {why}")

In [ ]:
# =========================================================================
# CELL 31 — OPTIMISED QWEN INFERENCE ENGINE  (batched, no per-token sync)
# =========================================================================
# Three changes against the previous implementation, all about latency, none about quality:
#
# 1. NO per-token GPU->CPU synchronisation. The previous LogitsProcessor called .tolist() at every
#    decoding step, stalling the pipeline once per token. Token log-probabilities are now OFF by
#    default and the code path is gone entirely: output_scores stays False, so no [batch, vocab]
#    tensor is retained per step and nothing is read back mid-generation.
# 2. O(n) stopping. _FastBraceStop keeps brace depth as state and decodes only the tokens added
#    since the previous check, instead of re-decoding the whole tail every 8 steps.
# 3. Micro-batching. Independent tasks are grouped by generation budget and executed together,
#    with automatic halving on OOM. Every result stays keyed to its own
#    (customer_id, document, page, task) -- no shared conversational context, no leakage.
@dataclass
class InferenceTask:
    key: str
    image: Image.Image
    prompt: str
    max_new_tokens: int
    task: str
    customer_id: str = ""
    document: str = ""
    page: Optional[int] = None
    retry_number: int = 0
    fields: Tuple[str, ...] = ()
    meta: Dict[str, Any] = field(default_factory=dict)


@dataclass
class GenResult:
    raw_model_output: str
    n_output_tokens: int
    n_input_tokens: int
    inference_time: float
    generation_time: float
    token_texts: List[str] = field(default_factory=list)
    token_logprobs: List[float] = field(default_factory=list)
    truncated: bool = False
    error: Optional[str] = None
    error_kind: Optional[str] = None
    degraded: Optional[str] = None
    batch_size: int = 1


def is_oom_error(exc: BaseException) -> bool:
    if torch is not None and isinstance(exc, getattr(torch.cuda, "OutOfMemoryError", ())):
        return True
    t = f"{type(exc).__name__}: {exc}".lower()
    return "out of memory" in t or "outofmemory" in t


class _FastBraceStop:
    """Stop each sequence when its JSON object closes. O(n) total, not O(n^2).

    State (brace depth, in-string, escape) is carried between checks, so only the tokens added
    since the previous check are decoded."""

    def __init__(self, tokenizer, prompt_len: int, batch_size: int, open_offset: int = 1,
                 every: int = 16):
        self.tok = tokenizer
        self.prompt_len = prompt_len
        self.every = every
        self.pos = [prompt_len] * batch_size
        self.depth = [open_offset] * batch_size
        self.in_str = [False] * batch_size
        self.esc = [False] * batch_size
        self.flags = [False] * batch_size

    def __call__(self, input_ids, scores, **kw):
        cur = input_ids.shape[1]
        n = input_ids.shape[0]
        if (cur - self.prompt_len) >= 8 and (cur - self.prompt_len) % self.every == 0:
            for i in range(n):
                if self.flags[i] or self.pos[i] >= cur:
                    continue
                chunk = self.tok.decode(input_ids[i, self.pos[i]:cur],
                                        skip_special_tokens=True)
                self.pos[i] = cur
                d, ins, esc = self.depth[i], self.in_str[i], self.esc[i]
                for c in chunk:
                    if ins:
                        if esc:        esc = False
                        elif c == "\\": esc = True
                        elif c == '"': ins = False
                        continue
                    if c == '"':   ins = True
                    elif c == "{": d += 1
                    elif c == "}":
                        d -= 1
                        if d <= 0:
                            self.flags[i] = True
                            break
                self.depth[i], self.in_str[i], self.esc[i] = d, ins, esc
        return torch.tensor(self.flags, dtype=torch.bool, device=input_ids.device)


class BatchedOCREngine:
    """Executes InferenceTasks. One model, one processor, micro-batched, deterministic."""

    def __init__(self, engine, cfg: Config = CFG):
        self.engine = engine
        self.cfg = cfg
        self.batch_size = max(1, cfg.INFERENCE_BATCH_SIZE)
        self.oom_events = 0

    # ------------------------------------------------------------------
    def _mock(self, tasks: Sequence[InferenceTask]) -> Dict[str, GenResult]:
        return {t.key: GenResult(raw_model_output="{}", n_output_tokens=0, n_input_tokens=0,
                                 inference_time=0.001, generation_time=0.001) for t in tasks}

    def _generate(self, tasks: Sequence[InferenceTask]) -> Dict[str, GenResult]:
        """One generate() over a micro-batch. Raises on OOM so the caller can split."""
        from transformers import StoppingCriteriaList
        eng, cfg = self.engine, self.cfg
        texts, images = [], []
        for t in tasks:
            messages = [{"role": "system", "content": [{"type": "text", "text": OCR_SYSTEM}]},
                        {"role": "user", "content": [{"type": "image"},
                                                     {"type": "text", "text": t.prompt}]}]
            try:
                s = eng.processor.apply_chat_template(messages, tokenize=False,
                                                      add_generation_prompt=True,
                                                      enable_thinking=not cfg.DISABLE_THINKING)
            except TypeError:
                s = eng.processor.apply_chat_template(messages, tokenize=False,
                                                      add_generation_prompt=True)
            texts.append(s + "{")
            images.append(t.image)

        t_all = time.perf_counter()
        with PROF.stage("processor"):
            inputs = eng.processor(text=texts, images=images, padding=True,
                                   return_tensors="pt").to(eng.model.device)
        prompt_len = int(inputs["input_ids"].shape[1])
        mnt = max(t.max_new_tokens for t in tasks)
        stopper = StoppingCriteriaList([_FastBraceStop(eng.tokenizer, prompt_len, len(tasks), 1,
                                                       cfg.BRACE_CHECK_EVERY)])
        # output_scores retains a [batch, vocab] tensor PER STEP and forces a per-token read to
        # use it. Both are removed: confidence is computed from evidence (agreement, clarity,
        # legibility, format, resolution), which needs no decoder internals.

        Profiler.sync(cfg)
        t_gen = time.perf_counter()
        with torch.inference_mode():          # no autograd graph, no grad buffers
            out = eng.model.generate(
                **inputs, max_new_tokens=mnt,
                do_sample=False, num_beams=1,          # TEMPERATURE = 0.0
                repetition_penalty=cfg.REPETITION_PENALTY,
                no_repeat_ngram_size=cfg.NO_REPEAT_NGRAM_SIZE,
                stopping_criteria=stopper,
                return_dict_in_generate=True, output_scores=False,
                pad_token_id=getattr(eng.tokenizer, "pad_token_id", None)
                             or getattr(eng.tokenizer, "eos_token_id", None))
        Profiler.sync(cfg)
        gen_s = time.perf_counter() - t_gen
        STAGE_TOTALS["generation"] += gen_s
        STAGE_CALLS["generation"] += 1

        seqs = out.sequences[:, prompt_len:].detach().to("cpu")   # ONE transfer for the batch
        results: Dict[str, GenResult] = {}
        pad = getattr(eng.tokenizer, "pad_token_id", None)
        for i, t in enumerate(tasks):
            ids = seqs[i]
            if pad is not None:
                keep = ids != pad
                ids = ids[keep] if keep.any() else ids
            n = int(ids.shape[0])
            text = "{" + eng.tokenizer.decode(ids, skip_special_tokens=True)
            results[t.key] = GenResult(
                raw_model_output=text, n_output_tokens=n, n_input_tokens=prompt_len,
                inference_time=round((time.perf_counter() - t_all) / len(tasks), 3),
                generation_time=round(gen_s / len(tasks), 3),
                truncated=bool(n >= t.max_new_tokens), batch_size=len(tasks))
        del out, inputs, seqs
        return results

    # ------------------------------------------------------------------
    def run(self, tasks: Sequence[InferenceTask]) -> Dict[str, GenResult]:
        """Execute tasks, grouped into micro-batches. Results are keyed by task.key."""
        if not tasks:
            return {}
        if CFG.DRY_RUN:
            raise RuntimeError("DRY_RUN is on: inference was requested but must not run. "
                               "Set CFG.DRY_RUN = False to execute.")
        if isinstance(self.engine, MockEngine):
            return self._mock(tasks)
        out: Dict[str, GenResult] = {}
        # Group by generation budget: a batch runs until its LONGEST member finishes, so mixing
        # a 96-token name task with a 640-token lines task would waste the short one's slot.
        buckets: Dict[int, List[InferenceTask]] = defaultdict(list)
        for t in tasks:
            buckets[t.max_new_tokens].append(t)
        for budget, group in buckets.items():
            i, bs = 0, self.batch_size
            while i < len(group):
                chunk = group[i:i + bs]
                try:
                    out.update(self._generate(chunk))
                    i += len(chunk)
                except Exception as exc:
                    if not is_oom_error(exc):
                        for t in chunk:                    # fail closed, keep the batch alive
                            out[t.key] = GenResult("", 0, 0, 0.0, 0.0,
                                                   error=f"{type(exc).__name__}: {exc}",
                                                   error_kind="inference")
                        i += len(chunk)
                        continue
                    release_cuda_cache()                   # only on OOM, never per page
                    self.oom_events += 1
                    if bs > 1:
                        bs = max(1, bs // 2)
                        self.batch_size = bs               # remember: do not re-crash every batch
                        log.warning("OOM: micro-batch reduced to %d", bs)
                        continue
                    t = chunk[0]
                    f = self.cfg.OOM_DOWNSCALE_FACTOR
                    if t.meta.get("oom_downscales", 0) < self.cfg.OOM_MAX_DOWNSCALES:
                        t.meta["oom_downscales"] = t.meta.get("oom_downscales", 0) + 1
                        t.image = t.image.resize((max(64, int(t.image.width * f)),
                                                  max(64, int(t.image.height * f))),
                                                 Image.LANCZOS)
                        log.warning("OOM on a single task; retrying at %d%%", int(f * 100))
                        continue
                    out[t.key] = GenResult("", 0, 0, 0.0, 0.0, error=str(exc), error_kind="oom")
                    i += 1
        for t in tasks:
            r = out.get(t.key)
            if r is None:
                continue
            QWEN_COUNTER["total"] += 1
            QWEN_COUNTER[t.customer_id] = QWEN_COUNTER.get(t.customer_id, 0) + 1
            record_perf(customer_id=t.customer_id, document=t.document, pdf=t.meta.get("pdf"),
                        page=t.page, task=t.task, roi_type=t.meta.get("roi", "full_page"),
                        retry_number=t.retry_number,
                        image_width=t.image.width, image_height=t.image.height,
                        visual_tokens=int(t.image.width * t.image.height / 784),
                        input_tokens=r.n_input_tokens, output_tokens=r.n_output_tokens,
                        max_new_tokens=t.max_new_tokens,
                        inference_time=r.inference_time, generation_time=r.generation_time,
                        tokens_per_s=round(r.n_output_tokens / max(1e-6, r.generation_time), 1),
                        batch_size=r.batch_size, truncated=r.truncated,
                        status="ERROR" if r.error_kind else "OK", error=r.error)
            # Heartbeat: one compact line per call, so the notebook is never silently "frozen".
            slow = " SLOW" if r.inference_time > CFG.QWEN_CALL_WARN_SECONDS else ""
            print(f"[QWEN {QWEN_COUNTER['total']}] cust={t.customer_id} "
                  f"doc={Path(t.document).stem[:18]} page={t.page} task={t.task} "
                  f"in={t.image.width}x{t.image.height} mnt={t.max_new_tokens} "
                  f"out={r.n_output_tokens}tok {r.inference_time:.1f}s{slow}", flush=True)
            if WATCHDOG["deadline"] and time.perf_counter() > WATCHDOG["deadline"]:
                raise TimeoutError(
                    f"BENCHMARK_MAX_SECONDS exceeded during customer={t.customer_id} "
                    f"document={t.document} page={t.page} task={t.task} "
                    f"qwen_call={QWEN_COUNTER['total']} max_new_tokens={t.max_new_tokens} "
                    f"image={t.image.width}x{t.image.height} "
                    f"elapsed={time.perf_counter() - WATCHDOG['start']:.0f}s")
        return out


QWEN_COUNTER: Dict[str, int] = defaultdict(int)
WATCHDOG: Dict[str, Optional[float]] = {"deadline": None, "start": None}


def arm_watchdog(seconds: Optional[float]) -> None:
    WATCHDOG["start"] = time.perf_counter()
    WATCHDOG["deadline"] = (WATCHDOG["start"] + seconds) if seconds else None


OCR_ENGINE = BatchedOCREngine(ENGINE, CFG)


def warm_up(cfg: Config = CFG) -> Dict[str, Any]:
    """One throwaway inference. Excluded from steady-state measurements."""
    if isinstance(ENGINE, MockEngine):
        return {"ran": False, "reason": "mock engine"}
    img = Image.new("RGB", (768, 512), "white")
    t = InferenceTask(key="warmup", image=img, prompt='Return exactly: {"ok": true}',
                      max_new_tokens=16, task="warmup")
    t0 = time.perf_counter()
    try:
        OCR_ENGINE.run([t])
        dt = time.perf_counter() - t0
        out = {"ran": True, "warmup_time": round(dt, 2)}
        try:
            toks = max(1, int(PERF_ROWS[-1]["output_tokens"])) if PERF_ROWS else 16
            MEASURED_DECODE_RATE[0] = round(toks / max(1e-6, dt), 1)
            out["decode_tokens_per_s"] = MEASURED_DECODE_RATE[0]
        except Exception:
            pass
    except Exception as exc:
        out = {"ran": False, "error": f"{type(exc).__name__}: {exc}"}
    PERF_ROWS.clear()                    # the warm-up must not pollute steady-state statistics
    STAGE_TOTALS.clear()
    STAGE_CALLS.clear()
    out["memory"] = gpu_snapshot("after_warmup")
    return out


WARMUP_STATS: Dict[str, Any] = {"ran": False}
print("CELL 31 ready: BatchedOCREngine, warm_up()")

In [ ]:
# =========================================================================
# CELL — ANCHOR-BASED FIELD LOCATION  (no hardcoded coordinates, no fixed pages)
# =========================================================================
# Handwritten fields are located from the printed LABEL next to them. When the PDF carries a text
# layer, pymupdf's search_for() returns the label's rectangle in PDF points, so the crop follows
# the label wherever the layout puts it -- different scan sizes, margins, orientations and form
# revisions all work. Scanned PDFs with no text layer fall back to tiles.
#
# Layouts supported (sections 25, 26):
#   FATCA layout A : value written AFTER "NOM"           -> band to the RIGHT of the label
#   FATCA layout B : values written BELOW "Nom"/"Prénom"  -> band BELOW each label
#   Signature      : "Nom et Prénom ou Raison Sociale"    -> band to the right, then below
#                    or the handwriting under "Numéro de compte"
FIELD_ANCHORS: Dict[str, List[Dict[str, Any]]] = {
    FATCA_DOC: [
        {"label": "Nom et Prénom", "direction": "right", "field": "combined"},
        {"label": "NOM", "direction": "right", "field": "surname_latin"},
        {"label": "Nom", "direction": "below", "field": "surname_latin"},
        {"label": "Prénom", "direction": "below", "field": "given_names_latin"},
        {"label": "Prenom", "direction": "below", "field": "given_names_latin"},
    ],
    SIGNATURE_DOC: [
        {"label": "Nom et Prénom ou Raison Sociale", "direction": "below", "field": "combined"},
        {"label": "Nom et Prenom ou Raison Sociale", "direction": "below", "field": "combined"},
        {"label": "Numéro de compte", "direction": "below", "field": "combined"},
        {"label": "Numero de compte", "direction": "below", "field": "combined"},
    ],
    DOMICILE_DOC: [{"label": "Nom", "direction": "right", "field": "combined"}],
    CONVENTION_DOC: [{"label": "Titulaire", "direction": "right", "field": "combined"},
                     {"label": "Nom", "direction": "right", "field": "combined"}],
}


def find_anchor_regions(cache: "PageCache", page: "CachedPage", document: str,
                        cfg: Config = CFG) -> List[Dict[str, Any]]:
    """Locate label rectangles via the text layer and derive the value band next to/below them."""
    out: List[Dict[str, Any]] = []
    if cache.doc is None or not page.has_text_layer:
        return out
    try:
        pg = cache.doc[page.page_number - 1]
    except Exception:
        return out
    pw, ph = page.pdf_rect[2] - page.pdf_rect[0], page.pdf_rect[3] - page.pdf_rect[1]
    for spec in FIELD_ANCHORS.get(document, []):
        try:
            rects = pg.search_for(spec["label"])
        except Exception:
            rects = []
        if not rects:
            continue
        r = rects[0]
        h = max(r.y1 - r.y0, ph * 0.012)
        if spec["direction"] == "right":
            band = (r.x1, r.y0 - 0.4 * h, min(page.pdf_rect[2], r.x1 + 0.75 * pw), r.y1 + 0.6 * h)
        else:                                            # below
            band = (max(page.pdf_rect[0], r.x0 - 0.02 * pw), r.y1,
                    min(page.pdf_rect[2], r.x0 + 0.6 * pw), r.y1 + 2.4 * h)
        out.append({"label": spec["label"], "direction": spec["direction"],
                    "field": spec["field"], "pdf_rect": tuple(band),
                    "label_rect": (r.x0, r.y0, r.x1, r.y1)})
        if len(out) >= 3:
            break
    return out


def anchor_crop(cache: "PageCache", page: "CachedPage", anchor: Dict[str, Any],
                cfg: Config = CFG) -> Optional[Dict[str, Any]]:
    """Render the anchored band straight from the PDF at handwriting resolution."""
    img = cache.clip(page.page_number, anchor["pdf_rect"], cfg.REGION_RENDER_DPI)
    if img is None or img.width < 80:
        return None
    if img.width < cfg.REGION_MAX_WIDTH:
        f = min(cfg.HANDWRITING_UPSCALE_FACTOR, cfg.REGION_MAX_WIDTH / max(1, img.width))
        if f > 1.02:
            img = img.resize((int(img.width * f), int(img.height * f)), Image.LANCZOS)
    processed, ops = preprocess_handwriting(img, "standard", cfg)
    return {"image": processed, "original": img, "label": anchor["label"],
            "direction": anchor["direction"], "field": anchor["field"],
            "source": "pdf_anchor_clip", "ops": ops,
            "dimensions": [processed.width, processed.height],
            "approx_visual_tokens": int(processed.width * processed.height / 784)}


def combine_name_parts(surname: Optional[str], given: Optional[str]) -> Optional[str]:
    """`combined_name_raw`: the parts joined for comparison, with the raw parts kept separate."""
    parts = [p for p in (surname, given) if p]
    return " ".join(parts) if parts else None


print("CELL 32 ready: find_anchor_regions(), anchor_crop() -- labels, not coordinates")

In [ ]:
# =========================================================================
# CELL 33 — OCR TASK PLANNER
# =========================================================================
# Nothing runs unless the plan asks for it (section 43). The planner is pure CPU: it decides
# which Qwen calls are actually required, using cheap signals first --
#   * the PDF text layer (free) for headings and document classification,
#   * CV MRZ candidate detection (cheap) on EVERY identity page,
#   * page quality (cheap) to skip blank pages entirely.
# Expensive multimodal inference is reserved for what those cannot answer.
@dataclass
class PagePlan:
    page_number: int
    blank: bool
    needs_identity_extract: bool = False
    needs_name_extract: bool = False
    needs_title_call: bool = False
    title_from_text_layer: Optional[str] = None
    mrz_candidates: List[Any] = field(default_factory=list)
    mrz_text_hint: Optional[str] = None
    reason: str = ""


def plan_document(cache: PageCache, document: str, cfg: Config = CFG) -> List[PagePlan]:
    """Decide the work for one document. No Qwen calls happen here."""
    plans: List[PagePlan] = []
    is_identity = document == IDENTITY_DOC
    title_seen = False
    for page in cache.pages:
        view, meta, quality = cache.view(page)
        if quality["is_blank"]:
            plans.append(PagePlan(page.page_number, True, reason="blank page: no GPU work"))
            continue
        plan = PagePlan(page.page_number, False)
        plan.title_from_text_layer = text_layer_title(page) if cfg.USE_PDF_TEXT_LAYER else None
        # A title call is only needed when the text layer cannot supply the heading, and only for
        # the first non-blank page of a document.
        plan.needs_title_call = (not title_seen) and plan.title_from_text_layer is None
        title_seen = True

        if is_identity:
            plan.needs_identity_extract = True
            with PROF.stage("mrz_detect"):
                plan.mrz_candidates = detect_mrz_candidates(view, page.page_number, cfg)
            plan.mrz_text_hint = text_layer_mrz_hint(page) if cfg.USE_PDF_TEXT_LAYER else None
            plan.reason = (f"identity page: extract fields; "
                           f"{len(plan.mrz_candidates)} MRZ candidate(s)")
        else:
            plan.needs_name_extract = True
            plan.reason = "secondary document: verify heading + read name"
        plans.append(plan)
    return plans


def plan_summary(plans: Sequence[PagePlan]) -> Dict[str, Any]:
    return {"pages": len(plans), "blank": sum(p.blank for p in plans),
            "identity_extracts": sum(p.needs_identity_extract for p in plans),
            "name_extracts": sum(p.needs_name_extract for p in plans),
            "title_calls": sum(p.needs_title_call for p in plans),
            "mrz_candidate_pages": [p.page_number for p in plans if p.mrz_candidates]}


print("CELL 33 ready: plan_document()")

In [ ]:
# =========================================================================
# CELL — EXECUTION PLAN AND DRY RUN
# =========================================================================
# The plan is computed from the ACTUAL opened PDFs (real page counts, real detected regions) and
# printed BEFORE any inference. This is the control that was missing: an architectural explosion
# is now visible in seconds instead of five hours.
@dataclass
class PlannedTask:
    customer_id: str
    document_type: str
    pdf: str
    page: Optional[int]
    task: str
    roi_type: str
    reason: str
    estimated_qwen_call: int
    estimated_max_new_tokens: int

    @property
    def document(self) -> str:
        return self.document_type


def plan_customer(cust: "CustomerDocs", cfg: Config = CFG
                  ) -> Tuple[List[PlannedTask], List[Dict[str, Any]]]:
    """Open each selected PDF once, discover its real page count, decide the minimal task set."""
    tasks: List[PlannedTask] = []
    doc_rows: List[Dict[str, Any]] = []
    for doc in DOCUMENTS_REQUIRED:
        path = cust.path(doc)
        if not path:
            doc_rows.append({"customer_id": cust.customer_id, "document": doc,
                             "page_count": 0, "planned_calls": 0, "note": "missing"})
            continue
        try:
            with PageCache(path, cfg) as cache:
                n_pages = len(cache.pages)          # discovered, never assumed
                before = len(tasks)
                if doc == IDENTITY_DOC:
                    mrz_pages, text_pages = [], []
                    for page in cache.pages:
                        view, meta, quality = cache.view(page)
                        if quality["is_blank"]:
                            continue
                        cands = detect_mrz_candidates(view, page.page_number, cfg)
                        page.mrz_candidates = cands
                        if cands:
                            mrz_pages.append(page.page_number)
                        # Every non-blank page gets a field call, because the executor does the
                        # same. The plan MUST match what will actually run: a plan that
                        # under-predicts is worse than no plan, since it hides the explosion it
                        # exists to reveal.
                        text_pages.append(page.page_number)
                    for pn in text_pages:
                        tasks.append(PlannedTask(cust.customer_id, doc, Path(path).name, pn,
                                                 "identity_page", "full_page",
                                                 "printed identity fields", 1,
                                                 cfg.MAX_NEW_TOKENS_IDENTITY))
                    if mrz_pages:
                        tasks.append(PlannedTask(cust.customer_id, doc, Path(path).name,
                                                 mrz_pages[0], "identity_mrz", "mrz_band",
                                                 f"MRZ candidate on page(s) {mrz_pages}", 1,
                                                 cfg.MAX_NEW_TOKENS_MRZ))
                elif doc == FATCA_DOC:
                    need_title = [p.page_number for p in cache.pages
                                  if not cache.view(p)[2]["is_blank"] and not p.has_text_layer]
                    for pn in need_title:
                        tasks.append(PlannedTask(cust.customer_id, doc, Path(path).name, pn,
                                                 "fatca_component_detection", "title_band",
                                                 "component detection (no text layer)", 1,
                                                 cfg.MAX_NEW_TOKENS_TITLE))
                    tasks.append(PlannedTask(cust.customer_id, doc, Path(path).name, None,
                                             "fatca_handwriting", "anchor_or_tile",
                                             "handwritten Nom/Prenom", 1,
                                             cfg.MAX_NEW_TOKENS_NAME))
                else:
                    tname = ("signature_verification" if doc == SIGNATURE_DOC else
                             "domicile_name" if doc == DOMICILE_DOC else "convention_name")
                    tasks.append(PlannedTask(cust.customer_id, doc, Path(path).name, 1, tname,
                                             "anchor_or_page",
                                             "document verification + client name", 1,
                                             cfg.MAX_NEW_TOKENS_NAME))
                planned = len(tasks) - before
                doc_rows.append({"customer_id": cust.customer_id, "document": doc,
                                 "page_count": n_pages, "planned_calls": planned,
                                 "selected_file": Path(path).name,
                                 "duplicates": ";".join(getattr(cust, "duplicates", {})
                                                        .get(doc, [])),
                                 "note": ""})
                if planned > cfg.MAX_QWEN_CALLS_PER_PDF:
                    log.warning("TASK EXPLOSION: %s / %s plans %d calls (limit %d). "
                                "Processing of this document will be skipped.",
                                cust.customer_id, doc, planned, cfg.MAX_QWEN_CALLS_PER_PDF)
                    doc_rows[-1]["note"] = "exceeds MAX_QWEN_CALLS_PER_PDF -> skipped"
                    tasks = tasks[:before]
        except Exception as exc:
            doc_rows.append({"customer_id": cust.customer_id, "document": doc, "page_count": -1,
                             "planned_calls": 0, "note": f"open failed: {exc}"})
    # retries are bounded and only fire on failure; expose the ceiling, not a guess
    return tasks, doc_rows


def build_execution_plan(targets: List["CustomerDocs"], cfg: Config = CFG
                         ) -> Tuple[pd.DataFrame, pd.DataFrame]:
    all_tasks, all_docs = [], []
    for cust in targets:
        t, d = plan_customer(cust, cfg)
        all_tasks.extend(t)
        all_docs.extend(d)
    tdf = pd.DataFrame([asdict(t) for t in all_tasks])
    ddf = pd.DataFrame(all_docs)
    if len(tdf):
        tdf.to_csv(DIRS["reports"] / "execution_plan.csv", index=False, encoding="utf-8-sig")
    return tdf, ddf


def print_execution_plan(tdf: pd.DataFrame, ddf: pd.DataFrame, cfg: Config = CFG) -> None:
    print("=" * 78)
    print("EXECUTION PLAN (no inference has run)")
    if ddf.empty:
        print("  nothing to do")
        return
    for cid, grp in ddf.groupby("customer_id"):
        print(f"\nCustomer: {cid}")
        for _, r in grp.iterrows():
            pages = "missing" if r["page_count"] == 0 else f"{int(r['page_count'])} pages"
            dup = f" (+{len(str(r.get('duplicates') or '').split(';')) - 1} duplicate)" \
                if r.get("duplicates") else ""
            note = f"  [{r['note']}]" if r.get("note") else ""
            print(f"  {r['document']:<28} {pages:<10} planned calls: "
                  f"{int(r['planned_calls'])}{dup}{note}")
    print("-" * 78)
    n_calls = len(tdf)
    n_cust = ddf["customer_id"].nunique()
    n_pages = int(ddf["page_count"].clip(lower=0).sum())
    max_retry = cfg.MAX_TARGETED_RETRIES
    print(f"customers {n_cust} | PDFs {int((ddf['page_count'] > 0).sum())} | pages {n_pages}")
    print(f"planned Qwen calls: {n_calls}  (worst case with retries: "
          f"{n_calls + max_retry * max(1, n_calls // 4)})")
    print(f"calls per customer: {n_calls / max(1, n_cust):.1f} | "
          f"per page: {n_calls / max(1, n_pages):.2f}")
    if len(tdf):
        print("\nby task:")
        print(tdf.groupby("task").agg(
            calls=("task", "size"),
            max_new_tokens=("estimated_max_new_tokens", "max")).to_string())
        rate = MEASURED_DECODE_RATE[0]
        if rate:
            est = tdf["estimated_max_new_tokens"].sum() / rate
            print(f"\nestimated inference time at the measured {rate:.1f} tok/s: "
                  f"{est/60:.1f} min upper bound (calls stop early when the JSON closes)")
        else:
            print("\nrun warm_up() to measure the decode rate and get a time estimate")
    print("=" * 78)
    if n_calls > cfg.MAX_QWEN_CALLS_PER_PDF * max(1, int((ddf['page_count'] > 0).sum())):
        log.warning("planned call count is above the configured ceiling -- review before running")


MEASURED_DECODE_RATE = [0.0]      # tokens/s, filled by warm_up()
print("CELL 34 ready: build_execution_plan(), print_execution_plan()  [DRY_RUN preview]")

In [ ]:
# =========================================================================
# CELL — CALL-BUDGET VALIDATION  (runs BEFORE any inference)
# =========================================================================
# The previous notebook could only reveal an inference explosion by running for hours. These
# ceilings are checked against the finite plan and raise immediately.
class TaskExplosion(RuntimeError):
    pass


def validate_call_budget(plan_tasks: pd.DataFrame, plan_docs: pd.DataFrame,
                         cfg: Config = CFG) -> None:
    if plan_tasks is None or plan_tasks.empty:
        print("budget check: no planned calls")
        return
    problems: List[str] = []
    total = len(plan_tasks)
    if total > cfg.MAX_TOTAL_QWEN_CALLS:
        problems.append(f"TOTAL {total} > MAX_TOTAL_QWEN_CALLS {cfg.MAX_TOTAL_QWEN_CALLS}")

    per_cust = plan_tasks.groupby("customer_id").size()
    for cid, n in per_cust.items():
        if n > cfg.MAX_QWEN_CALLS_PER_CUSTOMER:
            problems.append(f"customer {cid}: {n} calls > "
                            f"MAX_QWEN_CALLS_PER_CUSTOMER {cfg.MAX_QWEN_CALLS_PER_CUSTOMER}")

    per_pdf = plan_tasks.groupby(["customer_id", "document_type"]).size()
    for (cid, doc), n in per_pdf.items():
        if n > cfg.MAX_QWEN_CALLS_PER_PDF:
            problems.append(f"{cid}/{doc}: {n} calls > "
                            f"MAX_QWEN_CALLS_PER_PDF {cfg.MAX_QWEN_CALLS_PER_PDF}")

    paged = plan_tasks[plan_tasks["page"].notna()]
    if len(paged):
        per_page = paged.groupby(["customer_id", "document_type", "page"]).size()
        for (cid, doc, pg), n in per_page.items():
            if n > cfg.MAX_QWEN_CALLS_PER_PAGE:
                problems.append(f"{cid}/{doc} page {int(pg)}: {n} calls > "
                                f"MAX_QWEN_CALLS_PER_PAGE {cfg.MAX_QWEN_CALLS_PER_PAGE}")

    print(f"budget check: {total} planned calls | "
          f"max/customer {int(per_cust.max())} | max/pdf {int(per_pdf.max())}")
    if problems:
        for p in problems[:10]:
            print("  !!", p)
        raise TaskExplosion(
            f"{len(problems)} budget violation(s). Nothing was sent to the GPU. "
            "Review dry_run_plan.csv, or raise the ceiling deliberately in CELL 2.")


print("CELL 35 ready: validate_call_budget()")

In [ ]:
# =========================================================================
# CELL 36 — IDENTITY EXTRACTION  (one batched pass + targeted retries)
# =========================================================================
def _fields_from_parsed(parsed: Dict[str, Any], names: Sequence[str], *, document: str,
                        page: Optional[int], region: str, task: str, variant: Optional[str],
                        retry: int, quality: Dict[str, Any], gen: GenResult,
                        agreements: Optional[Dict[str, float]] = None,
                        text_height: Optional[float] = None,
                        cfg: Config = CFG) -> Dict[str, Any]:
    agreements = agreements or {}
    return {f: make_field(f, parsed.get(f, {}).get("value"),
                          parsed.get(f, {}).get("confidence", "unreadable"),
                          document=document, page=page, region=region, ocr_task=task,
                          preprocessing_variant=variant, retry_number=retry,
                          agreement=agreements.get(f),
                          visual_clarity=quality.get("visual_clarity", 0.5),
                          text_height=text_height if text_height is not None
                          else quality.get("text_height_px_in_view"),
                          text_type=parsed.get(f, {}).get("text_type"),
                          raw_output=gen.raw_model_output, cfg=cfg)
            for f in names}


def extract_identity(cache: PageCache, plans: List[PagePlan], customer_id: str,
                     ocr: BatchedOCREngine, errors: List[Dict[str, Any]],
                     cfg: Config = CFG) -> Dict[str, Any]:
    """Pass 1: one combined call per non-blank page, all pages batched together.
    Pass 2: targeted retries only for pages whose mandatory fields are still missing.
    MRZ: cheap CV already ran in the planner; Qwen runs only on the winning candidate."""
    by_page = {p.page_number: p for p in plans}
    pages = {p.page_number: p for p in cache.pages}

    # ---------------- PASS 1: one task per page, executed as one micro-batched group ----------
    tasks: List[InferenceTask] = []
    for plan in plans:
        if plan.blank or not plan.needs_identity_extract:
            continue
        page = pages[plan.page_number]
        view, meta, quality = cache.view(page)
        tasks.append(InferenceTask(
            key=f"id_p{plan.page_number}", image=view,
            prompt=identity_page_prompt(with_title=plan.title_from_text_layer is None),
            max_new_tokens=cfg.MAX_NEW_TOKENS_IDENTITY, task="identity_page",
            customer_id=customer_id, document=IDENTITY_DOC, page=plan.page_number,
            fields=tuple(IDENTITY_KEYS)))
    results = ocr.run(tasks)

    per_page_fields: Dict[int, Dict[str, Any]] = {}
    page_records: List[Dict[str, Any]] = []
    observed_titles: List[str] = []
    for plan in plans:
        page = pages[plan.page_number]
        view, meta, quality = cache.view(page)
        rec = {"customer_id": customer_id, "document": IDENTITY_DOC,
               "page_number": plan.page_number, "width": page.width, "height": page.height,
               "render_dpi": page.render_dpi, "render_time": page.render_time,
               "has_text_layer": page.has_text_layer, "plan": plan.reason,
               "n_model_calls": 0, "inference_time": 0.0, "status": "OK",
               "page_quality": quality, "preprocessing": meta,
               "mrz_candidate_count": len(plan.mrz_candidates),
               "mrz_candidates": [asdict(c) for c in plan.mrz_candidates]}
        if plan.blank:
            rec["status"] = "SKIPPED_BLANK"
            page_records.append(rec)
            continue
        gen = results.get(f"id_p{plan.page_number}")
        if gen is None:
            page_records.append(rec)
            continue
        rec["n_model_calls"] += 1
        rec["inference_time"] += gen.inference_time
        with PROF.stage("parse"):
            parsed, ok, warn = parse_json_response(gen.raw_model_output,
                                                   IDENTITY_KEYS + ["document_title"])
        if gen.error_kind:
            rec["status"] = "SYSTEM_ERROR"
            errors.append({"customer_id": customer_id, "document": IDENTITY_DOC,
                           "page": plan.page_number, "stage": "identity_pass1",
                           "error_type": gen.error_kind, "error_message": gen.error,
                           "traceback": "", "timestamp": utcnow()})
        title = plan.title_from_text_layer or (parsed.get("document_title") or {}).get("value")
        if title:
            observed_titles.append(title)
        rec["document_title"] = title
        rec["title_source"] = "pdf_text_layer" if plan.title_from_text_layer else "qwen"
        per_page_fields[plan.page_number] = _fields_from_parsed(
            parsed, IDENTITY_KEYS, document=IDENTITY_DOC, page=plan.page_number,
            region="full_page", task="identity_page",
            variant=";".join(meta.get("applied", [])), retry=0, quality=quality, gen=gen,
            cfg=cfg)
        rec["fields_found"] = sum(1 for f in per_page_fields[plan.page_number].values()
                                  if f["value"])
        rec["parse_ok"] = ok
        rec["raw_model_output"] = gen.raw_model_output[:1500]
        page_records.append(rec)

    # ---------------- PASS 2: targeted retries, DOCUMENT-level, batched ---------------------
    # The retry decision is made after looking at ALL pages together, not page by page. A
    # passport's page 2 legitimately carries only the MRZ, so per-page retries would burn a call
    # re-reading a page that was never going to hold identity fields. A field is only retried
    # when it is missing from the whole document.
    doc_missing = [f for f in MANDATORY_FIELDS
                   if all((fl.get(f) or {}).get("value") is None
                          for fl in per_page_fields.values())]
    retry_tasks: List[InferenceTask] = []
    retry_plan: Dict[str, Dict[str, Any]] = {}
    if doc_missing:
        # Rank pages by how much they already yielded, then by clarity: the page that produced
        # something is the one most likely to hold the rest.
        ranked = sorted(
            per_page_fields.keys(),
            key=lambda pn: (-sum(1 for v in per_page_fields[pn].values() if v["value"]),
                            -(cache.view(pages[pn])[2].get("visual_clarity") or 0)))
        for pn in ranked[:max(1, cfg.MAX_TARGETED_RETRIES)]:
            view, meta, quality = cache.view(pages[pn])
            if quality.get("visual_clarity", 0) < 0.10:
                continue                              # hopeless page: do not spend a call on it
            # A different TASK, not the same request again: transcribe lines verbatim.
            retry_tasks.append(InferenceTask(
                key=f"id_lines_p{pn}", image=view, prompt=LINES_PROMPT,
                max_new_tokens=cfg.MAX_NEW_TOKENS_LINES, task="identity_lines",
                customer_id=customer_id, document=IDENTITY_DOC, page=pn, retry_number=1))
            retry_plan[f"id_lines_p{pn}"] = {"page": pn, "missing": doc_missing, "kind": "lines"}
    if retry_tasks:
        for key, gen in ocr.run(retry_tasks).items():
            info = retry_plan[key]
            pn, missing = info["page"], info["missing"]
            page = pages[pn]
            view, meta, quality = cache.view(page)
            with PROF.stage("parse"):
                lines, _, ok = parse_lines_response(gen.raw_model_output)
            mapped = map_lines_to_fields(lines) if ok else {}
            for f in missing:
                v = (mapped.get(f) or {}).get("value")
                if v is not None:
                    per_page_fields[pn][f] = make_field(
                        f, v, "medium", document=IDENTITY_DOC, page=pn,
                        region="full_page_lines", ocr_task="identity_lines",
                        preprocessing_variant=";".join(meta.get("applied", [])), retry_number=1,
                        visual_clarity=quality.get("visual_clarity", 0.5),
                        text_height=quality.get("text_height_px_in_view"),
                        raw_output=gen.raw_model_output, cfg=cfg)
            for r in page_records:
                if r.get("page_number") == pn:
                    r["n_model_calls"] += 1
                    r["inference_time"] += gen.inference_time
                    r["retries"] = r.get("retries", 0) + 1
                    r["lines_transcribed"] = len(lines)

    # ---------------- merge across pages ----------------
    final, conflicts = {}, []
    for f in IDENTITY_KEYS:
        cands = [(pn, fl[f]) for pn, fl in per_page_fields.items()
                 if fl.get(f, {}).get("value") is not None]
        if not cands:
            final[f] = empty_field(f, IDENTITY_DOC, "not_visible_on_any_page")
            continue
        cands.sort(key=lambda t: -t[1]["confidence"])
        best = cands[0][1]
        rivals = [c for c in cands[1:]
                  if normalize_for_comparison(c[1]["value"]) !=
                     normalize_for_comparison(best["value"])]
        if [c for c in rivals if c[1]["confidence"] >= best["confidence"] - 0.05]:
            conflicts.append({"field": f, "candidates": [{"page": pn, "value": n["value"],
                                                          "confidence": n["confidence"]}
                                                         for pn, n in cands]})
            final[f] = empty_field(f, IDENTITY_DOC, "conflicting_pages")
        else:
            final[f] = best

    mrz = extract_mrz(cache, plans, customer_id, ocr, errors, cfg)
    final["mrz"] = mrz["field"]
    title_verdict = classify_title(observed_titles[0] if observed_titles else None,
                                   " | ".join(observed_titles[1:]) if len(observed_titles) > 1
                                   else None, EXPECTED_TITLES[IDENTITY_DOC], cfg)
    return {"document": IDENTITY_DOC, "source_pdf": str(cache.pdf_path),
            "page_count": len(cache.pages),
            "expected_document": EXPECTED_TITLES[IDENTITY_DOC]["label"],
            "detected_document": title_verdict.get("observed_title"),
            "document_verification_status": title_verdict["status"],
            "title_check": title_verdict, "fields": final, "conflicts": conflicts,
            "mrz": mrz["report"], "pages": page_records,
            "n_model_calls": sum(r.get("n_model_calls", 0) for r in page_records)
                             + mrz["report"].get("n_model_calls", 0),
            "plan": plan_summary(plans)}


print("CELL 36 ready: extract_identity()")

In [ ]:
# =========================================================================
# CELL 37 — MRZ OCR  (cheap CV first; Qwen only on a plausible candidate)
# =========================================================================
def extract_mrz(cache: PageCache, plans: List[PagePlan], customer_id: str,
                ocr: BatchedOCREngine, errors: List[Dict[str, Any]],
                cfg: Config = CFG) -> Dict[str, Any]:
    """Every identity page was inspected by the CV detector in the planner -- that coverage is
    unchanged and costs no GPU time. Qwen is invoked ONLY for the winning candidate, and only a
    second time if the first read was not structurally an MRZ."""
    all_cands = [c for p in plans for c in p.mrz_candidates]
    report: Dict[str, Any] = {
        "mrz_detected": False, "mrz_candidate_page": None, "mrz_region": None,
        "mrz_detection_method": None, "mrz_detection_score": None, "mrz_source": None,
        "mrz_crop_width": None, "mrz_crop_height": None, "mrz_upscaled_char_height": None,
        "mrz_inference_time": 0.0, "mrz_validation_status": None, "mrz_raw_output": None,
        "attempts": [], "n_model_calls": 0,
        "pages_inspected": [p.page_number for p in plans if not p.blank],
        "candidates_examined": len(all_cands),
        "candidate_pages": sorted({c.page_number for c in all_cands}),
        "text_layer_hint_pages": [p.page_number for p in plans if p.mrz_text_hint]}

    if not all_cands:
        # No Qwen call at all: there is no plausible MRZ region on any page.
        report["failure_reason"] = "no MRZ-like region on any page (CV detector)"
        return {"report": report, "field": empty_field("mrz", IDENTITY_DOC, "no_mrz_region")}

    best = localize_mrz(all_cands)
    page = next(p for p in cache.pages if p.page_number == best.page_number)
    view, meta, quality = cache.view(page)
    report.update({"mrz_detected": True, "mrz_candidate_page": best.page_number,
                   "mrz_region": list(best.bbox), "mrz_detection_method": best.method,
                   "mrz_detection_score": best.score,
                   "mrz_candidate_metrics": {"n_lines": best.n_lines,
                                             "width_frac": best.width_frac, "fill": best.fill,
                                             "height_uniformity": best.height_uniformity,
                                             "pitch_regularity": best.pitch_regularity,
                                             "saturation": best.saturation,
                                             "rel_y": best.rel_y}})

    variants = ["standard", "aggressive", "binarised"][:1 + cfg.MAX_TARGETED_RETRIES]
    value, band, raw_out, used = None, "unreadable", "", None
    for attempt, variant in enumerate(variants, start=1):
        with PROF.stage("mrz_preprocess"):
            crop = crop_mrz_cached(cache, page, best, variant, cfg)
        if attempt == 1 and should_save_debug(customer_id, IDENTITY_DOC, cfg):
            save_debug(customer_id, IDENTITY_DOC, best.page_number, "mrz", "mrz_candidate",
                       crop["original"], cfg)
        gen = ocr.run([InferenceTask(key=f"mrz_a{attempt}", image=crop["image"],
                                     prompt=MRZ_PROMPT,
                                     max_new_tokens=cfg.MAX_NEW_TOKENS_MRZ, task="mrz",
                                     customer_id=customer_id, document=IDENTITY_DOC,
                                     page=best.page_number, retry_number=attempt - 1)]
                      )[f"mrz_a{attempt}"]
        report["n_model_calls"] += 1
        report["mrz_inference_time"] = round(report["mrz_inference_time"] + gen.inference_time, 3)
        with PROF.stage("parse"):
            parsed, ok, _ = parse_json_response(gen.raw_model_output, ["mrz"])
        cand_value = parsed["mrz"]["value"]
        with PROF.stage("validate"):
            structural = looks_like_mrz(cand_value)
            validation = validate_mrz(cand_value)
        report["attempts"].append({
            "attempt": attempt, "variant": variant,
            "reason": "first pass" if attempt == 1 else
                      f"previous attempt produced {'no value' if value is None else 'non-MRZ text'};"
                      f" switching preprocessing to '{variant}'",
            "source": crop["source"], "ops": crop["ops"],
            "crop": [crop["crop_width"], crop["crop_height"]],
            "char_height_px": crop["estimated_char_height_px"],
            "sufficient_resolution": crop["sufficient_resolution"],
            "visual_tokens": crop["approx_visual_tokens"],
            "inference_time": gen.inference_time, "structurally_mrz": structural,
            "validation_status": validation["status"]})
        report.update({"mrz_crop_width": crop["crop_width"],
                       "mrz_crop_height": crop["crop_height"],
                       "mrz_upscaled_char_height": crop["estimated_char_height_px"],
                       "mrz_source": crop["source"],
                       "mrz_raw_output": gen.raw_model_output[:1500],
                       "mrz_validation_status": validation["status"],
                       "mrz_validation": validation})
        if gen.error_kind:
            errors.append({"customer_id": customer_id, "document": IDENTITY_DOC,
                           "page": best.page_number, "stage": f"mrz_attempt_{attempt}",
                           "error_type": gen.error_kind, "error_message": gen.error,
                           "traceback": "", "timestamp": utcnow()})
            break
        if cand_value is not None and structural:
            value, band, raw_out, used = cand_value, parsed["mrz"]["confidence"], \
                gen.raw_model_output, crop
            break            # EARLY EXIT: a good MRZ read ends the ladder immediately
        if cand_value is not None:
            report["rejected_non_mrz_transcription"] = cand_value[:200]
        # A failing CHECKSUM alone never triggers another attempt: the transcription stands as
        # read and is flagged. Retrying until a checksum passes would be guessing.

    if value is None:
        if should_save_debug(customer_id, IDENTITY_DOC, cfg, failure=True):
            save_debug(customer_id, IDENTITY_DOC, best.page_number, "mrz", "mrz_failed",
                       crop["image"], cfg)
        report["failure_reason"] = (f"MRZ region located on page {best.page_number} but not "
                                    f"readable after {len(report['attempts'])} variants")
        fld = empty_field("mrz", IDENTITY_DOC, "mrz_unreadable")
        fld.update({"source_page": best.page_number, "source_region": list(best.bbox),
                    "ocr_task": "mrz"})
        return {"report": report, "field": fld}

    fld = make_field("mrz", value, band, document=IDENTITY_DOC, page=best.page_number,
                     region=f"mrz:{best.bbox}", ocr_task="mrz",
                     preprocessing_variant=used["variant"],
                     retry_number=len(report["attempts"]) - 1,
                     visual_clarity=quality["visual_clarity"],
                     text_height=used["estimated_char_height_px"], raw_output=raw_out, cfg=cfg)
    fld["mrz_validation"] = report["mrz_validation"]     # diagnostic only; value untouched
    return {"report": report, "field": fld}


def crop_mrz_cached(cache: PageCache, page: CachedPage, candidate, variant: str,
                    cfg: Config = CFG) -> Dict[str, Any]:
    """MRZ crop through the cached PDF handle, with the preprocessing variant applied once."""
    key = (page.page_number, tuple(candidate.bbox), variant)
    region = cache.region(page, candidate.bbox, cfg.REGION_RENDER_DPI, cfg.MRZ_MAX_WIDTH,
                          pad=0.02)
    original = region["image"]
    n_lines = max(2, candidate.n_lines or 2)
    est_char = original.height / n_lines * 0.55
    img = original
    if est_char < cfg.MRZ_MIN_CHAR_HEIGHT:            # resolution target, not a page-wide resize
        f = min(cfg.MRZ_UPSCALE_FACTOR, cfg.MRZ_MIN_CHAR_HEIGHT / max(1e-6, est_char))
        f = min(f, cfg.MRZ_MAX_WIDTH / max(1, img.width))
        if f > 1.02:
            img = img.resize((int(img.width * f), int(img.height * f)), Image.LANCZOS)
            est_char *= f
    processed, ops = preprocess_mrz(img, variant, cfg)
    return {"original": original, "image": processed, "source": region["source"],
            "bbox": region["bbox"], "variant": variant, "ops": ops,
            "crop_width": processed.width, "crop_height": processed.height,
            "estimated_char_height_px": round(est_char, 1),
            "sufficient_resolution": bool(est_char >= cfg.MRZ_MIN_CHAR_HEIGHT),
            "approx_visual_tokens": int(processed.width * processed.height / 784)}


print("CELL 37 ready: extract_mrz() -- Qwen only when a candidate exists")

In [ ]:
# =========================================================================
# CELL 38 — SECONDARY DOCUMENTS  (domicile, convention, FATCA, signature)
# =========================================================================
def should_save_debug(customer_id: str, document: str, cfg: Config = CFG,
                      failure: bool = False) -> bool:
    """Selective debugging. Writing thousands of PNGs is real I/O time, so production defaults
    to off and only sampled customers or genuine failures produce files."""
    if failure and cfg.SAVE_DEBUG_ON_FAILURE:
        return True
    if not cfg.SAVE_DEBUG_IMAGES:
        return False
    if cfg.DEBUG_SAMPLE_CUSTOMERS and customer_id not in cfg.DEBUG_SAMPLE_CUSTOMERS:
        return False
    if cfg.DEBUG_SAMPLE_DOCUMENTS and document not in cfg.DEBUG_SAMPLE_DOCUMENTS:
        return False
    return True


def extract_secondary(cache: PageCache, plans: List[PagePlan], customer_id: str, document: str,
                      expected: Dict[str, Any], handwritten: bool, ocr: BatchedOCREngine,
                      errors: List[Dict[str, Any]], cfg: Config = CFG) -> Dict[str, Any]:
    """One combined call per page (heading + name), all pages batched, early exit once both
    names are read. The heading comes free from the PDF text layer when there is one."""
    pages = {p.page_number: p for p in cache.pages}
    tasks, order = [], []
    for plan in plans:
        if plan.blank:
            continue
        page = pages[plan.page_number]
        view, meta, quality = cache.view(page)
        tasks.append(InferenceTask(
            key=f"{document}_p{plan.page_number}", image=view,
            prompt=name_prompt(handwritten, with_title=plan.title_from_text_layer is None),
            max_new_tokens=cfg.MAX_NEW_TOKENS_NAME, task="name",
            customer_id=customer_id, document=document, page=plan.page_number))
        order.append(plan)
        if cfg.SECONDARY_FIRST_PAGE_ONLY:
            break             # the client name is on the first page of these forms in practice
    results = ocr.run(tasks)

    names = {f: empty_field(f, document, "not_extracted") for f in NAME_FIELDS}
    page_records, observed_titles = [], []
    for plan in plans:
        page = pages[plan.page_number]
        rec = {"customer_id": customer_id, "document": document,
               "page_number": plan.page_number, "width": page.width, "height": page.height,
               "render_dpi": page.render_dpi, "render_time": page.render_time,
               "has_text_layer": page.has_text_layer, "plan": plan.reason,
               "n_model_calls": 0, "inference_time": 0.0,
               "status": "SKIPPED_BLANK" if plan.blank else "OK"}
        gen = results.get(f"{document}_p{plan.page_number}")
        if plan.blank or gen is None:
            page_records.append(rec)
            continue
        view, meta, quality = cache.view(page)
        rec.update({"page_quality": quality, "preprocessing": meta,
                    "n_model_calls": 1, "inference_time": gen.inference_time})
        with PROF.stage("parse"):
            parsed, ok, _ = parse_json_response(gen.raw_model_output,
                                                NAME_FIELDS + ["document_title"])
        title = plan.title_from_text_layer or (parsed.get("document_title") or {}).get("value")
        if title:
            observed_titles.append(title)
        rec["document_title"] = title
        rec["title_source"] = "pdf_text_layer" if plan.title_from_text_layer else "qwen"
        verdict = classify_title(title, None, expected, cfg)
        # A document that is demonstrably the WRONG document is not a source of identity data.
        if verdict["status"] != "MISMATCH":
            got = _fields_from_parsed(parsed, NAME_FIELDS, document=document,
                                      page=plan.page_number, region="full_page",
                                      task="handwriting" if handwritten else "name",
                                      variant=";".join(meta.get("applied", [])), retry=0,
                                      quality=quality, gen=gen, cfg=cfg)
            for f in NAME_FIELDS:
                if names[f]["value"] is None and got[f]["value"] is not None:
                    names[f] = got[f]
        page_records.append(rec)
        if all(n["value"] is not None for n in names.values()):
            break                                  # EARLY EXIT: nothing left to read

    # ---- targeted retry: high-resolution crop, ONLY for names still missing ----
    missing = [f for f in NAME_FIELDS if names[f]["value"] is None]
    if missing and cfg.MAX_TARGETED_RETRIES > 0 and order:
        plan = order[0]
        page = pages[plan.page_number]
        view, meta, quality = cache.view(page)
        verdict = classify_title(observed_titles[0] if observed_titles else None, None,
                                 expected, cfg)
        if verdict["status"] != "MISMATCH":
            box = page_tiles(view, cfg.PAGE_TILES)[0]
            with PROF.stage("roi_prepare"):
                region = cache.region(page, box, cfg.REGION_RENDER_DPI, cfg.REGION_MAX_WIDTH,
                                      pad=0.01)
                img, ops = preprocess_handwriting(region["image"], "standard", cfg)
            gen = ocr.run([InferenceTask(
                key=f"{document}_hw", image=img, prompt=name_prompt(handwritten, False),
                max_new_tokens=cfg.MAX_NEW_TOKENS_NAME,
                task="handwriting" if handwritten else "name_retry",
                customer_id=customer_id, document=document, page=plan.page_number,
                retry_number=1)])[f"{document}_hw"]
            with PROF.stage("parse"):
                p2, ok2, _ = parse_json_response(gen.raw_model_output, NAME_FIELDS)
            got = _fields_from_parsed(p2, NAME_FIELDS, document=document,
                                      page=plan.page_number,
                                      region=f"tile:{region['bbox']}",
                                      task="handwriting" if handwritten else "name_retry",
                                      variant="+".join(ops), retry=1, quality=quality, gen=gen,
                                      text_height=(quality.get("text_height_px_in_view") or 0) * 1.8,
                                      cfg=cfg)
            for f in missing:
                if got[f]["value"] is not None:
                    names[f] = got[f]
            for r in page_records:
                if r.get("page_number") == plan.page_number:
                    r["n_model_calls"] += 1
                    r["inference_time"] += gen.inference_time
            if should_save_debug(customer_id, document, cfg,
                                 failure=any(names[f]["value"] is None for f in missing)):
                task = ("fatca_handwriting_crop" if document == FATCA_DOC else
                        "signature_handwriting_crop" if document == SIGNATURE_DOC else
                        "name_crop")
                save_debug(customer_id, document, plan.page_number, task, "retry_1", img, cfg)

    verdict = classify_title(observed_titles[0] if observed_titles else None,
                             " | ".join(observed_titles[1:]) if len(observed_titles) > 1 else None,
                             expected, cfg)
    return {"document": document, "source_pdf": str(cache.pdf_path),
            "page_count": len(cache.pages), "expected_document": expected.get("label"),
            "detected_document": verdict.get("observed_title"),
            "document_verification_status": verdict["status"], "title_check": verdict,
            "document_title_verified": verdict["status"] == "MATCH",
            "fields": names, "pages": page_records,
            "n_model_calls": sum(r.get("n_model_calls", 0) for r in page_records),
            "plan": plan_summary(plans)}


def extract_fatca_doc(cache: PageCache, plans: List[PagePlan], customer_id: str,
                      ocr: BatchedOCREngine, errors: List[Dict[str, Any]],
                      cfg: Config = CFG) -> Dict[str, Any]:
    """Component detection from headings (text layer when available, otherwise ONE batched
    title call per page), then handwriting OCR only on component 2's page."""
    pages = {p.page_number: p for p in cache.pages}
    need_title = [p for p in plans if not p.blank and p.title_from_text_layer is None]
    title_results = {}
    if need_title:
        title_results = ocr.run([InferenceTask(
            key=f"fatca_title_p{p.page_number}", image=cache.view(pages[p.page_number])[0],
            prompt=TITLE_PROMPT, max_new_tokens=cfg.MAX_NEW_TOKENS_TITLE, task="title",
            customer_id=customer_id, document=FATCA_DOC, page=p.page_number)
            for p in need_title])

    page_titles, page_records, calls = [], [], len(need_title)
    for plan in plans:
        page = pages[plan.page_number]
        rec = {"customer_id": customer_id, "document": FATCA_DOC,
               "page_number": plan.page_number, "width": page.width, "height": page.height,
               "render_dpi": page.render_dpi, "render_time": page.render_time,
               "has_text_layer": page.has_text_layer, "plan": plan.reason,
               "n_model_calls": 0, "inference_time": 0.0,
               "status": "SKIPPED_BLANK" if plan.blank else "OK"}
        title = plan.title_from_text_layer
        if title is None and not plan.blank:
            gen = title_results.get(f"fatca_title_p{plan.page_number}")
            if gen is not None:
                rec["n_model_calls"] = 1
                rec["inference_time"] = gen.inference_time
                with PROF.stage("parse"):
                    parsed, _, _ = parse_json_response(gen.raw_model_output, ["document_title"])
                title = (parsed.get("document_title") or {}).get("value")
        rec["document_title"] = title
        rec["title_source"] = "pdf_text_layer" if plan.title_from_text_layer else "qwen"
        page_titles.append({"page_number": plan.page_number, "title_text": title})
        page_records.append(rec)

    components, claimed = {}, set()
    for key in ("component_2", "component_1"):     # the specific title claims its pages first
        spec = FATCA_COMPONENTS[key]
        hits = []
        for pt in page_titles:
            if pt["page_number"] in claimed:
                continue
            if classify_title(pt["title_text"], None, spec, cfg)["status"] == "MATCH":
                hits.append(pt["page_number"])
                claimed.add(pt["page_number"])
        detected = bool(hits)
        components[key] = {
            "expected_title": spec["label"], "detected": detected, "title_verified": detected,
            "status": "MATCH" if detected else (
                "UNCERTAIN" if all(not pt["title_text"] for pt in page_titles) else "MISSING"),
            "pages_found": hits, "page_count": len(hits),   # DISCOVERED, never assumed
            "page_count_verified": None,   # no fixed-page rule; the count is reported as found
            "observed_titles": [pt["title_text"] for pt in page_titles]}

    names = {f: empty_field(f, FATCA_DOC, "not_extracted") for f in NAME_FIELDS}
    targets = components["component_2"]["pages_found"] or (
        [p["page_number"] for p in page_titles] if components["component_2"]["status"] ==
        "UNCERTAIN" else [])
    for pn in targets[:1]:                 # handwriting OCR only where the field actually is
        page = pages[pn]
        view, meta, quality = cache.view(page)
        with PROF.stage("roi_prepare"):
            region = cache.region(page, page_tiles(view, cfg.PAGE_TILES)[0],
                                  cfg.REGION_RENDER_DPI, cfg.REGION_MAX_WIDTH, pad=0.01)
            img, ops = preprocess_handwriting(region["image"], "standard", cfg)
        gen = ocr.run([InferenceTask(key="fatca_hw", image=img,
                                     prompt=name_prompt(True, False),
                                     max_new_tokens=cfg.MAX_NEW_TOKENS_NAME, task="handwriting",
                                     customer_id=customer_id, document=FATCA_DOC, page=pn)]
                      )["fatca_hw"]
        calls += 1
        with PROF.stage("parse"):
            parsed, _, _ = parse_json_response(gen.raw_model_output, NAME_FIELDS)
        names = _fields_from_parsed(parsed, NAME_FIELDS, document=FATCA_DOC, page=pn,
                                    region=f"tile:{region['bbox']}", task="handwriting",
                                    variant="+".join(ops), retry=0, quality=quality, gen=gen,
                                    cfg=cfg)
        for r in page_records:
            if r.get("page_number") == pn:
                r["n_model_calls"] += 1
                r["inference_time"] += gen.inference_time
        if should_save_debug(customer_id, FATCA_DOC, cfg,
                             failure=any(n["value"] is None for n in names.values())):
            save_debug(customer_id, FATCA_DOC, pn, "fatca_handwriting_crop", "hw", img, cfg)

    overall = ("MATCH" if all(c["status"] == "MATCH" for c in components.values()) else
               "UNCERTAIN" if any(c["status"] == "UNCERTAIN" for c in components.values())
               else "MISMATCH")
    return {"document": FATCA_DOC, "source_pdf": str(cache.pdf_path),
            "page_count": len(cache.pages),
            "expected_document": "FATCA identification forms (two components)",
            "detected_document": "; ".join(t for t in [pt["title_text"] for pt in page_titles]
                                           if t) or None,
            "document_verification_status": overall,
            "component_1_detected": components["component_1"]["detected"],
            "component_1_header_verified": components["component_1"]["title_verified"],
            "component_1_page_count": components["component_1"]["page_count"],
            "component_2_detected": components["component_2"]["detected"],
            "component_2_title_verified": components["component_2"]["title_verified"],
            "components": components, "fields": names, "pages": page_records,
            "n_model_calls": calls, "plan": plan_summary(plans)}


print("CELL 38 ready: extract_secondary(), extract_fatca_doc()")

In [ ]:
# =========================================================================
# CELL 39 — CROSS-DOCUMENT COMPARISON  (after independent extraction, never before)
# =========================================================================
# Comparison NEVER modifies OCR. It only produces a verdict.
#   MATCH          both sides read reliably and identical after comparison-only normalisation
#   MISMATCH       both sides read reliably and different
#   UNCERTAIN      at least one side is low confidence, or contains ? characters
#   NOT_AVAILABLE  at least one side is null
# Unreadable is NEVER a mismatch.
MATCH, MISMATCH, UNCERTAIN, NOT_AVAILABLE = "MATCH", "MISMATCH", "UNCERTAIN", "NOT_AVAILABLE"
RELIABLE_BANDS = ("high", "medium")


def compare_values(doc_value: Optional[str], ref_value: Optional[str],
                   doc_band: str = "unreadable", method: str = "normalized_exact",
                   cfg: Config = CFG) -> Dict[str, Any]:
    a, b = normalize_for_comparison(doc_value), normalize_for_comparison(ref_value)
    out = {"document_value": doc_value, "reference_value": ref_value,
           "normalized_document_value": a, "normalized_reference_value": b,
           "comparison_method": method, "similarity_score": None,
           "threshold": cfg.NAME_FUZZY_THRESHOLD, "document_confidence_band": doc_band}
    if a is None or b is None:
        return {**out, "status": NOT_AVAILABLE,
                "reason": "value not available on one side -- unreadable is not a mismatch"}
    if doc_value and "?" in str(doc_value):
        return {**out, "status": UNCERTAIN, "reason": "document value has unreadable characters"}
    if a == b:
        return {**out, "status": MATCH if doc_band in RELIABLE_BANDS else UNCERTAIN,
                "similarity_score": 1.0,
                "reason": "identical after normalisation" if doc_band in RELIABLE_BANDS
                else "values agree but the document read is not reliable"}
    score = round(difflib.SequenceMatcher(None, a, b).ratio(), 3)
    out["similarity_score"] = score
    if doc_band not in RELIABLE_BANDS:
        return {**out, "status": UNCERTAIN, "comparison_method": "difflib_ratio",
                "reason": f"values differ (score {score}) and the document read is not reliable"}
    if score >= cfg.NAME_FUZZY_THRESHOLD:
        # Deliberately NOT reported as MATCH: a near-identical string is a reason for a human to
        # look, never a confirmation. The score and threshold are always exposed.
        return {**out, "status": UNCERTAIN, "comparison_method": "difflib_ratio",
                "reason": f"near match {score} >= {cfg.NAME_FUZZY_THRESHOLD} but not identical"}
    return {**out, "status": MISMATCH, "comparison_method": "difflib_ratio",
            "reason": f"values differ, score {score} < {cfg.NAME_FUZZY_THRESHOLD}"}


def compare_documents(documents: Dict[str, Optional[Dict[str, Any]]],
                      cfg: Config = CFG) -> Dict[str, Any]:
    """Identity is the reference, but agreement with it never overrides uncertainty elsewhere."""
    identity = documents.get(IDENTITY_DOC) or {}
    ref = identity.get("fields", {})
    rows, details = [], []
    for doc in DOCUMENTS_REQUIRED:
        res = documents.get(doc)
        if doc == IDENTITY_DOC:
            rows.append({"document": doc, "surname_latin": "reference",
                         "given_names_latin": "reference", "overall": "reference",
                         "document_verification_status":
                             (res or {}).get("document_verification_status", "NOT_AVAILABLE")})
            continue
        if res is None:
            rows.append({"document": doc, "surname_latin": NOT_AVAILABLE,
                         "given_names_latin": NOT_AVAILABLE, "overall": NOT_AVAILABLE,
                         "document_verification_status": "MISSING"})
            continue
        statuses = {}
        for f in NAME_FIELDS:
            node = res.get("fields", {}).get(f, {})
            refnode = ref.get(f, {})
            cmp_ = compare_values(node.get("value"), refnode.get("value"),
                                  node.get("confidence_band", "unreadable"), cfg=cfg)
            cmp_.update({"source_document": doc, "reference_source": IDENTITY_DOC, "field": f})
            statuses[f] = cmp_["status"]
            details.append(cmp_)
        overall = (MISMATCH if MISMATCH in statuses.values() else
                   UNCERTAIN if UNCERTAIN in statuses.values() else
                   NOT_AVAILABLE if all(s == NOT_AVAILABLE for s in statuses.values()) else MATCH)
        rows.append({"document": doc, **statuses, "overall": overall,
                     "document_verification_status": res.get("document_verification_status")})
    return {"matrix": rows, "comparisons": details}


print("CELL 39 ready: compare_documents()")

In [ ]:
# =========================================================================
# CELL 40 — DATABASE VERIFICATION  (four columns, comparison only)
# =========================================================================
def _name_tokens(value: Optional[str]) -> Optional[List[str]]:
    n = normalize_for_comparison(value)
    return sorted(n.split()) if n else None


def compare_name_any_order(doc_surname: Optional[str], doc_given: Optional[str],
                           db_name: Optional[str], doc_band: str,
                           cfg: Config = CFG) -> Dict[str, Any]:
    """`Nom abrégé tiers` may be LAST FIRST or FIRST LAST. Compare as token multisets so both
    orders match, and report which order was observed. Normalisation is for comparison only."""
    parts = [p for p in (doc_surname, doc_given) if p]
    doc_full = " ".join(parts) if parts else None
    out = {"document_value": doc_full, "reference_value": db_name,
           "normalized_document_value": normalize_for_comparison(doc_full),
           "normalized_reference_value": normalize_for_comparison(db_name),
           "comparison_method": "token_multiset_any_order",
           "similarity_score": None, "threshold": cfg.NAME_FUZZY_THRESHOLD,
           "document_confidence_band": doc_band, "observed_order": None}
    dt, rt = _name_tokens(doc_full), _name_tokens(db_name)
    if dt is None or rt is None:
        return {**out, "status": NOT_AVAILABLE,
                "reason": "name not available on one side -- unreadable is not a mismatch"}
    if doc_full and "?" in doc_full:
        return {**out, "status": UNCERTAIN, "reason": "document name has unreadable characters"}
    if dt == rt:
        order = None
        if doc_surname and db_name:
            nb = normalize_for_comparison(db_name) or ""
            ns = normalize_for_comparison(doc_surname) or ""
            order = "LAST_FIRST" if nb.startswith(ns) else "FIRST_LAST"
        return {**out, "status": MATCH if doc_band in RELIABLE_BANDS else UNCERTAIN,
                "similarity_score": 1.0, "observed_order": order,
                "reason": "same name tokens in either order" if doc_band in RELIABLE_BANDS
                else "tokens agree but the document read is not reliable"}
    score = round(difflib.SequenceMatcher(None, " ".join(dt), " ".join(rt)).ratio(), 3)
    out["similarity_score"] = score
    if doc_band not in RELIABLE_BANDS:
        return {**out, "status": UNCERTAIN, "reason": "document read is not reliable"}
    if score >= cfg.NAME_FUZZY_THRESHOLD:
        return {**out, "status": UNCERTAIN,
                "reason": f"near match {score} >= {cfg.NAME_FUZZY_THRESHOLD} but tokens differ"}
    return {**out, "status": MISMATCH, "reason": f"name tokens differ, score {score}"}


def compare_dates(doc_value: Optional[str], db_value: Optional[str], doc_band: str,
                  cfg: Config = CFG) -> Dict[str, Any]:
    """Format differences are tolerated by parsing BOTH sides. The extracted representation is
    preserved exactly; only the comparison uses the parsed form."""
    d1, d2 = parse_date(doc_value), parse_date(db_value)
    out = {"document_value": doc_value, "reference_value": db_value,
           "normalized_document_value": d1.isoformat() if d1 else None,
           "normalized_reference_value": d2.isoformat() if d2 else None,
           "comparison_method": "parsed_date_equality", "similarity_score": None,
           "document_confidence_band": doc_band}
    if not doc_value or not db_value:
        return {**out, "status": NOT_AVAILABLE, "reason": "date not available on one side"}
    if "?" in str(doc_value):
        return {**out, "status": UNCERTAIN, "reason": "date has unreadable characters"}
    if d1 is None or d2 is None:
        return {**out, "status": UNCERTAIN,
                "reason": "a date could not be parsed for comparison; raw values preserved"}
    if d1 == d2:
        return {**out, "status": MATCH if doc_band in RELIABLE_BANDS else UNCERTAIN,
                "similarity_score": 1.0, "reason": "same calendar date"}
    return {**out, "status": MISMATCH if doc_band in RELIABLE_BANDS else UNCERTAIN,
            "reason": "different calendar dates"}


def compare_database(customer_id: str, documents: Dict[str, Optional[Dict[str, Any]]],
                     db_row: Optional[Dict[str, str]], cfg: Config = CFG) -> Dict[str, Any]:
    """ONLY Id tiers, Nom abrégé tiers, Date de naissance, Date d'expiration du Document."""
    identity = (documents.get(IDENTITY_DOC) or {}).get("fields", {})

    def node(f):
        return identity.get(f, {}) or {}

    if db_row is None:
        na = {"document_value": None, "reference_value": None, "status": NOT_AVAILABLE,
              "comparison_method": "none", "reason": "no database record for this customer"}
        return {"customer_id": customer_id, "db_id": None,
                "database_comparison": {"customer_id": na, "name": na,
                                        "date_of_birth": na, "expiry_date": na},
                "overall": NOT_AVAILABLE}

    db_id = db_row.get("customer_id")
    id_cmp = {"document_value": customer_id, "reference_value": db_id,
              "comparison_method": "exact_string", "similarity_score": None,
              "status": MATCH if str(customer_id).strip() == str(db_id).strip() else MISMATCH,
              "reason": "folder name compared with Id tiers"}

    name_cmp = compare_name_any_order(node("surname_latin").get("value"),
                                      node("given_names_latin").get("value"),
                                      db_row.get("name"),
                                      node("surname_latin").get("confidence_band", "unreadable"),
                                      cfg)
    dob_cmp = compare_dates(node("date_of_birth").get("value"), db_row.get("date_of_birth"),
                            node("date_of_birth").get("confidence_band", "unreadable"), cfg)
    exp_cmp = compare_dates(node("expiry_date").get("value"), db_row.get("expiry_date"),
                            node("expiry_date").get("confidence_band", "unreadable"), cfg)

    statuses = [id_cmp["status"], name_cmp["status"], dob_cmp["status"], exp_cmp["status"]]
    overall = (MISMATCH if MISMATCH in statuses else
               UNCERTAIN if UNCERTAIN in statuses else
               MATCH if MATCH in statuses else NOT_AVAILABLE)
    return {"customer_id": customer_id, "db_id": db_id,
            "database_comparison": {"customer_id": id_cmp, "name": name_cmp,
                                    "date_of_birth": dob_cmp, "expiry_date": exp_cmp},
            "overall": overall}


print("CELL 40 ready: compare_database() [four authorised columns only]")

In [ ]:
# =========================================================================
# CELL 41 — CUSTOMER AGGREGATION  (one PageCache per document, opened once)
# =========================================================================
SECONDARY_SPECS = {
    DOMICILE_DOC: {"expected": EXPECTED_TITLES[DOMICILE_DOC], "handwritten": True},
    CONVENTION_DOC: {"expected": EXPECTED_TITLES[CONVENTION_DOC], "handwritten": True},
    SIGNATURE_DOC: {"expected": EXPECTED_TITLES[SIGNATURE_DOC], "handwritten": True},
}


def overall_kyc_status(documents, matrix, db_check, cust, cfg: Config = CFG) -> Dict[str, Any]:
    """PASS / MISMATCH / UNCERTAIN / INCOMPLETE. Unreadable is never a mismatch."""
    reasons: List[str] = []
    missing_docs = [d for d in DOCUMENTS_REQUIRED if not cust.documents.get(d)]
    identity = documents.get(IDENTITY_DOC)
    if identity is None:
        return {"overall_kyc_status": "INCOMPLETE", "processing_status": "no_identity_document",
                "reasons": ["identity document missing or unprocessable"],
                "needs_manual_review": True, "overall_quality": None}

    mismatch_docs = [r["document"] for r in matrix["matrix"] if r.get("overall") == MISMATCH]
    wrong_type = [r["document"] for r in matrix["matrix"]
                  if r.get("document_verification_status") == "MISMATCH"]
    db_status = db_check.get("overall")
    unreadable_mandatory = [f for f in MANDATORY_FIELDS
                            if (identity["fields"].get(f) or {}).get("value") is None]
    mrz = identity.get("mrz", {})

    if mismatch_docs:
        reasons.append("name_mismatch_vs_identity:" + ",".join(mismatch_docs))
    if wrong_type:
        reasons.append("wrong_document_type:" + ",".join(wrong_type))
    if db_status == MISMATCH:
        reasons.append("database_mismatch:" + ",".join(
            k for k, v in db_check["database_comparison"].items() if v["status"] == MISMATCH))
    if unreadable_mandatory:
        reasons.append("unreadable_mandatory_fields:" + ",".join(unreadable_mandatory))
    if mrz.get("mrz_detected") and not (identity["fields"].get("mrz") or {}).get("value"):
        reasons.append("mrz_region_found_but_unreadable")
    if (identity["fields"].get("mrz") or {}).get("mrz_validation", {}).get("status") == \
            "checksum_mismatch":
        reasons.append("mrz_checksum_mismatch")
    if missing_docs:
        reasons.append("missing_documents:" + ",".join(missing_docs))

    if mismatch_docs or wrong_type or db_status == MISMATCH:
        status = "MISMATCH"
    elif missing_docs:
        status = "INCOMPLETE"
    elif unreadable_mandatory or db_status in (UNCERTAIN, NOT_AVAILABLE) or \
            any(r.get("overall") == UNCERTAIN for r in matrix["matrix"]):
        status = "UNCERTAIN"
    elif reasons:
        # Anything still flagged -- a failed MRZ check digit, an MRZ region located but not read
        # -- means something is unresolved. PASS must mean nothing is outstanding.
        status = "UNCERTAIN"
    else:
        status = "PASS"

    clarity = [p.get("page_quality", {}).get("visual_clarity")
               for d in documents.values() if d for p in d.get("pages", [])
               if p.get("page_quality")]
    return {"overall_kyc_status": status, "processing_status": "completed", "reasons": reasons,
            "needs_manual_review": status != "PASS",
            "overall_quality": round(float(np.mean(clarity)), 3) if clarity else None}


def process_customer(cust: "CustomerDocs", ocr: BatchedOCREngine, db: Optional[pd.DataFrame],
                     cfg: Config = CFG) -> Dict[str, Any]:
    """STAGE A: independent extraction, document by document, each PDF opened exactly once.
    STAGE B: CPU-side comparison. Never the reverse."""
    t0 = time.perf_counter()
    errors: List[Dict[str, Any]] = []
    documents: Dict[str, Optional[Dict[str, Any]]] = {d: None for d in DOCUMENTS_REQUIRED}
    plans_by_doc: Dict[str, Any] = {}

    def _run(doc: str, fn):
        path = cust.path(doc)
        if not path:
            return
        try:
            with PageCache(path, cfg) as cache:          # opened ONCE, closed on exit
                with PROF.stage("plan"):
                    plans = plan_document(cache, doc, cfg)
                plans_by_doc[doc] = plan_summary(plans)
                documents[doc] = fn(cache, plans)
        except Exception as exc:
            errors.append({"customer_id": cust.customer_id, "document": doc, "page": None,
                           "stage": "document", "error_type": type(exc).__name__,
                           "error_message": str(exc), "traceback": traceback.format_exc(),
                           "timestamp": utcnow()})
            log.error("%s failed for %s: %s", doc, cust.customer_id, exc)

    _run(IDENTITY_DOC, lambda c, p: extract_identity(c, p, cust.customer_id, ocr, errors, cfg))
    if cfg.PROCESS_SECONDARY_DOCUMENTS:
        for doc, spec in SECONDARY_SPECS.items():
            _run(doc, lambda c, p, d=doc, s=spec: extract_secondary(
                c, p, cust.customer_id, d, s["expected"], s["handwritten"], ocr, errors, cfg))
        _run(FATCA_DOC, lambda c, p: extract_fatca_doc(c, p, cust.customer_id, ocr, errors, cfg))

    with PROF.stage("cross_document"):
        matrix = compare_documents(documents, cfg)
    with PROF.stage("database_match"):
        db_check = compare_database(cust.customer_id, documents,
                                    database_record(db, cust.customer_id), cfg)
    verdict = overall_kyc_status(documents, matrix, db_check, cust, cfg)

    by_doc = defaultdict(dict)
    for c in matrix["comparisons"]:
        by_doc[c["source_document"]][c["field"]] = c["status"]
    for doc, res in documents.items():
        if res:
            for f, node in res.get("fields", {}).items():
                if f in NAME_FIELDS:
                    node["match_status"] = by_doc.get(doc, {}).get(f)

    identity = documents.get(IDENTITY_DOC) or {}
    return {"customer_id": cust.customer_id, "run_id": RUN_ID, "extracted_at_utc": utcnow(),
            "engine": {"model": getattr(ENGINE, "name", "?"),
                       "attention": getattr(ENGINE, "attn", "?"),
                       "quantization": getattr(ENGINE, "quantization", {}),
                       "batch_size": ocr.batch_size,
                       "max_image_dimension": cfg.MAX_IMAGE_DIMENSION,
                       "pdf_render_dpi": cfg.PDF_RENDER_DPI,
                       "region_render_dpi": cfg.REGION_RENDER_DPI},
            "document_presence": {PRESENCE_COLUMNS[d]: bool(cust.documents.get(d))
                                  for d in DOCUMENTS_REQUIRED},
            "documents": documents, "plans": plans_by_doc,
            "identity_fields": identity.get("fields", {}),
            "mrz": identity.get("mrz", {"mrz_detected": False}),
            "cross_document_verification": matrix["matrix"],
            "cross_document_comparisons": matrix["comparisons"],
            "database_verification": db_check, **verdict,
            "page_count_total": sum((d or {}).get("page_count", 0) for d in documents.values()),
            "n_model_calls": sum((d or {}).get("n_model_calls", 0) for d in documents.values()),
            "total_customer_time": round(time.perf_counter() - t0, 2),
            "errors": errors}


print("CELL 41 ready: process_customer()")

In [ ]:
# =========================================================================
# CELL 42 — OPTIMISED DRIVER
# =========================================================================
# Redesigned, not wrapped. Initialisation happens once, outside every loop:
#   model + processor  -> CELL 9  (process singleton)
#   database           -> loaded once into an indexed frame
#   discovery          -> once for the whole batch
# Per customer: each PDF is opened once, pages rendered once, analysed once, planned once, and
# only the planned inference tasks run -- micro-batched.
def run_pipeline(targets: Optional[List["CustomerDocs"]] = None, ocr: BatchedOCREngine = None,
                 db: Optional[pd.DataFrame] = None, cfg: Config = CFG,
                 label: str = "optimized") -> pd.DataFrame:
    ocr = ocr or OCR_ENGINE
    db = db if db is not None else DB
    targets = targets if targets is not None else TARGETS
    rows, consecutive_errors, skipped = [], 0, 0
    gpu_snapshot(f"{label}_start")
    t_start = time.perf_counter()

    for i, cust in enumerate(targets, 1):
        dest = DIRS["results"] / f"{cust.customer_id}.json"
        # Benchmarks NEVER skip: a benchmark that silently reuses old results measures nothing.
        if cfg.RESUME and dest.exists() and label == "optimized" and not WATCHDOG["deadline"]:
            skipped += 1
            log.info("[%d/%d] %s skipped (RESUME: result already exists)",
                     i, len(targets), cust.customer_id)
            continue
        try:
            rec = process_customer(cust, ocr, db, cfg)
        except Exception as exc:
            log.error("[%d/%d] %s FAILED: %s", i, len(targets), cust.customer_id, exc)
            rec = {"customer_id": cust.customer_id, "overall_kyc_status": "INCOMPLETE",
                   "processing_status": "error", "needs_manual_review": True,
                   "reasons": [str(exc)], "documents": {}, "identity_fields": {}, "mrz": {},
                   "cross_document_verification": [], "database_verification": {},
                   "n_model_calls": 0, "page_count_total": 0, "total_customer_time": 0.0,
                   "errors": [{"customer_id": cust.customer_id, "document": None, "page": None,
                               "stage": "process_customer", "error_type": type(exc).__name__,
                               "error_message": str(exc), "traceback": traceback.format_exc(),
                               "timestamp": utcnow()}]}
        if label == "optimized":
            dest.write_text(json.dumps(rec, ensure_ascii=False, indent=2, default=str),
                            encoding="utf-8")
        mrz = rec.get("mrz") or {}
        # Concise progress only: no model outputs, no per-page prints.
        log.info("[%d/%d] %s %s | %dp %.1fs calls=%d | MRZ p%s read=%s | db=%s",
                 i, len(targets), cust.customer_id, rec.get("overall_kyc_status"),
                 rec.get("page_count_total", 0), rec.get("total_customer_time", 0),
                 rec.get("n_model_calls", 0), mrz.get("mrz_candidate_page"),
                 bool((rec.get("identity_fields", {}).get("mrz") or {}).get("value")),
                 (rec.get("database_verification") or {}).get("overall"))
        rows.append({"customer_id": cust.customer_id, "mode": label,
                     "overall_kyc_status": rec.get("overall_kyc_status"),
                     "pages": rec.get("page_count_total", 0),
                     "time_s": rec.get("total_customer_time"),
                     "model_calls": rec.get("n_model_calls"),
                     "mrz_page": mrz.get("mrz_candidate_page"),
                     "mrz_read": bool((rec.get("identity_fields", {}).get("mrz") or {}).get("value")),
                     "database": (rec.get("database_verification") or {}).get("overall"),
                     "needs_manual_review": rec.get("needs_manual_review")})
        if rec.get("processing_status") == "error":
            consecutive_errors += 1
            if consecutive_errors >= cfg.MAX_CONSECUTIVE_ERRORS:
                log.error("ABORTING after %d consecutive failures. gpu_report(); free_model() "
                          "or restart; raise RESERVE_VRAM_GIB or lower MAX_VISUAL_TOKENS.",
                          consecutive_errors)
                break
        else:
            consecutive_errors = 0

    gpu_snapshot(f"{label}_end")
    elapsed = time.perf_counter() - t_start
    if skipped:
        log.warning("%d of %d customers were SKIPPED by CFG.RESUME (existing result files). "
                    "Qwen metrics below cover only the %d that actually ran. Set "
                    "CFG.RESUME=False or clear %s to reprocess.",
                    skipped, len(targets), len(rows), DIRS["results"])
    log.info("%s: %d customers in %.1f min (%.1f s/customer)",
             label, len(rows), elapsed / 60, elapsed / max(1, len(rows)))
    df = pd.DataFrame(rows)
    df.attrs["elapsed_s"] = elapsed
    return df


print("CELL 42 ready: run_pipeline()")

In [ ]:
# =========================================================================
# CELL 43 — PERFORMANCE TELEMETRY
# =========================================================================
def performance_frames() -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """(per-call metrics, per-task aggregate, per-stage aggregate)."""
    inf = pd.DataFrame(PERF_ROWS)
    if inf.empty:
        return inf, pd.DataFrame(), stage_report()
    total = float(inf["inference_time"].sum()) or 1.0
    by_task = inf.groupby("task").agg(
        number_of_calls=("inference_time", "size"),
        total_time=("inference_time", "sum"),
        average_time=("inference_time", "mean"),
        median_time=("inference_time", "median"),
        p95_time=("inference_time", lambda s: s.quantile(0.95)),
        avg_output_tokens=("output_tokens", "mean"),
        avg_input_tokens=("input_tokens", "mean"),
        avg_visual_tokens=("visual_tokens", "mean"),
        avg_batch_size=("batch_size", "mean"),
        retries=("retry_number", lambda s: int((s > 0).sum())),
    ).round(3)
    by_task["percentage_of_total"] = (100 * by_task["total_time"] / total).round(1)
    by_task = by_task.sort_values("total_time", ascending=False).reset_index()
    return inf, by_task, stage_report()


def print_performance() -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    inf, by_task, by_stage = performance_frames()
    print("=" * 78)
    print("PER-STAGE (all measured work, GPU and CPU)")
    print(by_stage.to_string(index=False) if len(by_stage) else "  no data")
    print("-" * 78)
    print("PER-TASK (Qwen calls only)")
    print(by_task.to_string(index=False) if len(by_task) else "  no data")
    if len(inf):
        print("-" * 78)
        print(f"Qwen calls: {len(inf)} | total inference {inf['inference_time'].sum():.1f}s | "
              f"avg {inf['inference_time'].mean():.2f}s | "
              f"output tokens avg {inf['output_tokens'].mean():.0f}")
        gpu_t = float(inf["inference_time"].sum())
        cpu_t = sum(v for k, v in STAGE_TOTALS.items()
                    if k in ("pdf_render", "preprocess", "page_analysis", "mrz_detect",
                             "roi_render", "roi_prepare", "mrz_preprocess", "parse",
                             "validate", "plan"))
        verdict = ("GPU-bound" if gpu_t > 2 * cpu_t else
                   "CPU-bound" if cpu_t > 2 * gpu_t else "mixed CPU/GPU")
        print(f"GPU (inference) {gpu_t:.1f}s vs CPU (render/preprocess/parse) {cpu_t:.1f}s "
              f"-> {verdict}")
        if cpu_t > gpu_t:
            print("  The H100 is waiting on CPU work: raise CPU_WORKERS or lower PDF_RENDER_DPI"
                  " before touching the model.")
    print("=" * 78)
    return inf, by_task, by_stage


print("CELL 43 ready: print_performance()")

In [ ]:
# =========================================================================
# CELL 44 — BENCHMARK: legacy architecture vs optimised, measured on THIS machine
# =========================================================================
# LEGACY_MODE reproduces the previous notebook's execution characteristics on the same documents:
#   * batch size 1
#   * token log-probs captured per step (the per-token device sync)
#   * brace check every 8 tokens (the O(n^2) stop)
#   * no PDF text layer (a Qwen call for every heading)
#   * separate line-transcription call on every page, not only as a retry
# Nothing about image resolution or OCR coverage differs, so the comparison isolates EXECUTION.
def _apply_mode(cfg: Config, mode: str) -> Config:
    """`legacy` deliberately reproduces the SLOW architecture for comparison. It is opt-in
    (CFG.BENCHMARK_LEGACY) because running it is exactly the multi-hour behaviour we removed."""
    c = Config(**{**asdict(cfg)})
    if mode == "legacy":
        c.INFERENCE_BATCH_SIZE = 1
        c.BRACE_CHECK_EVERY = 8              # the O(n^2) stop interval
        c.USE_PDF_TEXT_LAYER = False         # a Qwen call for every heading
        c.SECONDARY_FIRST_PAGE_ONLY = False  # a name call for every page of every document
    return c


def benchmark_pipeline(sample: Optional[List["CustomerDocs"]] = None,
                       modes: Optional[Sequence[str]] = None,
                       cfg: Config = CFG) -> pd.DataFrame:
    """Measured benchmark with a hard time limit.

    Defaults to ONE customer, OPTIMIZED ONLY. The legacy pipeline runs only when
    CFG.BENCHMARK_LEGACY is True, and the watchdog stops any run that exceeds
    CFG.BENCHMARK_MAX_SECONDS, reporting exactly where it was."""
    if cfg.DRY_RUN:
        print("CFG.DRY_RUN is True -- benchmark refuses to run inference. Set DRY_RUN=False.")
        return pd.DataFrame()
    sample = sample or TARGETS[:cfg.BENCHMARK_CUSTOMERS]
    if modes is None:
        modes = ("legacy", "optimized") if cfg.BENCHMARK_LEGACY else ("optimized",)
    if not sample:
        print("no customers to benchmark")
        return pd.DataFrame()
    print(f"BENCHMARK: {len(sample)} customer(s), modes={list(modes)}, "
          f"time limit {cfg.BENCHMARK_MAX_SECONDS:.0f}s per mode")
    rows = []
    for mode in modes:
        PERF_ROWS.clear(); STAGE_TOTALS.clear(); STAGE_CALLS.clear(); QWEN_COUNTER.clear()
        if torch is not None and torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        mcfg = _apply_mode(cfg, mode)
        engine = BatchedOCREngine(ENGINE, mcfg)
        arm_watchdog(cfg.BENCHMARK_MAX_SECONDS)
        t0 = time.perf_counter()
        timed_out = None
        try:
            df = run_pipeline(sample, engine, DB, mcfg, label=mode)
        except TimeoutError as exc:
            timed_out = str(exc)
            log.error("BENCHMARK STOPPED: %s", exc)
            df = pd.DataFrame()
        finally:
            arm_watchdog(None)
        elapsed = time.perf_counter() - t0
        inf = pd.DataFrame(PERF_ROWS)
        pages = int(df["pages"].sum()) if len(df) else 0
        rows.append({
            "mode": mode, "customers": len(df), "pages": pages,
            "total_time_s": round(elapsed, 1),
            "avg_pdf_time_s": round(elapsed / max(1, len(df)), 1),
            "qwen_calls": len(inf),
            "calls_per_customer": round(len(inf) / max(1, len(df)), 1),
            "avg_call_s": round(float(inf["inference_time"].mean()), 2) if len(inf) else None,
            "p95_call_s": round(float(inf["inference_time"].quantile(0.95)), 2) if len(inf) else None,
            "avg_output_tokens": round(float(inf["output_tokens"].mean()), 1) if len(inf) else None,
            "peak_gpu_gib": round(torch.cuda.max_memory_allocated() / 2**30, 2)
            if (torch is not None and torch.cuda.is_available()) else None,
            "oom_events": engine.oom_events, "timed_out": timed_out})
    out = pd.DataFrame(rows)
    if {"legacy", "optimized"} <= set(out["mode"]):
        lg = out[out["mode"] == "legacy"].iloc[0]
        op = out[out["mode"] == "optimized"].iloc[0]
        if lg["total_time_s"] and op["total_time_s"]:
            print(f"\nMEASURED: {lg['total_time_s'] / max(1e-9, op['total_time_s']):.2f}x faster, "
                  f"{int(lg['qwen_calls'])} -> {int(op['qwen_calls'])} Qwen calls")
    print(out.to_string(index=False))
    out.to_csv(DIRS["reports"] / "benchmark_comparison.csv", index=False, encoding="utf-8-sig")
    return out


def tune_batch_size(candidates: Sequence[int] = (1, 2, 4), cfg: Config = CFG) -> int:
    """Measure, do not guess. Times an identical set of synthetic tasks at each batch size."""
    if isinstance(ENGINE, MockEngine):
        print("mock engine: skipping")
        return 1
    img = Image.new("RGB", (cfg.MAX_IMAGE_DIMENSION, int(cfg.MAX_IMAGE_DIMENSION * 0.7)), "white")
    best, best_rate = 1, 0.0
    for n in candidates:
        eng = BatchedOCREngine(ENGINE, Config(**{**asdict(cfg), "INFERENCE_BATCH_SIZE": n}))
        tasks = [InferenceTask(key=f"t{i}", image=img, prompt=identity_page_prompt(),
                               max_new_tokens=128, task="bench") for i in range(n * 2)]
        if torch is not None and torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        t0 = time.perf_counter()
        try:
            eng.run(tasks)
            dt = time.perf_counter() - t0
            rate = len(tasks) / dt
            peak = (torch.cuda.max_memory_allocated() / 2**30
                    if (torch is not None and torch.cuda.is_available()) else 0)
            print(f"  batch {n}: {dt:5.1f}s for {len(tasks)} tasks = {rate:.2f} tasks/s, "
                  f"peak {peak:.1f} GiB, oom_events={eng.oom_events}")
            if rate > best_rate and eng.oom_events == 0:
                best, best_rate = n, rate
        except Exception as exc:
            print(f"  batch {n}: failed ({type(exc).__name__})")
            break
    print(f"\nrecommended: CFG.INFERENCE_BATCH_SIZE = {best}")
    return best


print("CELL 44 ready: benchmark_pipeline(), tune_batch_size()")

In [ ]:
# =========================================================================
# CELL 45 — OUTPUT GENERATION  (buffered; files opened once)
# =========================================================================
def load_results(results_dir: Path = None) -> List[Dict[str, Any]]:
    out = []
    for p in sorted(Path(results_dir or DIRS["results"]).glob("*.json")):
        try:
            out.append(json.loads(p.read_text(encoding="utf-8")))
        except Exception as exc:
            log.warning("unreadable result %s: %s", p.name, exc)
    return out


def build_processing_log(records: List[Dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for r in records:
        for doc, res in (r.get("documents") or {}).items():
            if not res:
                continue
            for p in res.get("pages", []):
                rows.append({"customer_id": r["customer_id"], "document": doc,
                             "page_number": p.get("page_number"), "width": p.get("width"),
                             "height": p.get("height"), "render_dpi": p.get("render_dpi"),
                             "render_time": p.get("render_time"),
                             "has_text_layer": p.get("has_text_layer"),
                             "title_source": p.get("title_source"),
                             "inference_time": p.get("inference_time"),
                             "n_model_calls": p.get("n_model_calls"),
                             "mrz_candidates": p.get("mrz_candidate_count"),
                             "quality": (p.get("page_quality") or {}).get("quality"),
                             "text_height_in_view":
                                 (p.get("page_quality") or {}).get("text_height_px_in_view"),
                             "retries": p.get("retries", 0), "plan": p.get("plan"),
                             "status": p.get("status")})
    return pd.DataFrame(rows)


def flatten_customer(r: Dict[str, Any]) -> Dict[str, Any]:
    row: Dict[str, Any] = OrderedDict(customer_id=r["customer_id"],
                                      overall_kyc_status=r.get("overall_kyc_status"),
                                      processing_status=r.get("processing_status"),
                                      needs_manual_review=r.get("needs_manual_review"),
                                      overall_quality=r.get("overall_quality"),
                                      reasons=";".join(r.get("reasons") or []),
                                      page_count_total=r.get("page_count_total"),
                                      total_customer_time=r.get("total_customer_time"),
                                      n_model_calls=r.get("n_model_calls"))
    row.update(r.get("document_presence") or {})
    for f in IDENTITY_FIELDS:
        n = (r.get("identity_fields") or {}).get(f) or {}
        row[f] = n.get("value")
        row[f + "_confidence"] = n.get("confidence")
        row[f + "_band"] = n.get("confidence_band")
        row[f + "_page"] = n.get("source_page")
        row[f + "_task"] = n.get("ocr_task")
    mrz = r.get("mrz") or {}
    row.update({"mrz_detected": mrz.get("mrz_detected"),
                "mrz_candidate_page": mrz.get("mrz_candidate_page"),
                "mrz_detection_score": mrz.get("mrz_detection_score"),
                "mrz_crop_width": mrz.get("mrz_crop_width"),
                "mrz_char_height_px": mrz.get("mrz_upscaled_char_height"),
                "mrz_inference_time": mrz.get("mrz_inference_time"),
                "mrz_validation_status": mrz.get("mrz_validation_status"),
                "mrz_source": mrz.get("mrz_source")})
    for doc in DOCUMENTS_REQUIRED[1:]:
        res = (r.get("documents") or {}).get(doc) or {}
        key = re.sub(r"[^a-z]+", "_", doc.lower()).strip("_")
        row[f"{key}_verification"] = res.get("document_verification_status", "MISSING")
        for f in NAME_FIELDS:
            row[f"{key}_{f}"] = ((res.get("fields") or {}).get(f) or {}).get("value")
    if (r.get("documents") or {}).get(FATCA_DOC):
        fa = r["documents"][FATCA_DOC]
        row.update({"component_1_detected": fa.get("component_1_detected"),
                    "component_1_header_verified": fa.get("component_1_header_verified"),
                    "component_1_page_count": fa.get("component_1_page_count"),
                    "component_2_detected": fa.get("component_2_detected"),
                    "component_2_title_verified": fa.get("component_2_title_verified")})
    for x in r.get("cross_document_verification") or []:
        key = re.sub(r"[^a-z]+", "_", x["document"].lower()).strip("_")
        row[f"identity_vs_{key}_name"] = x.get("overall")
    dbc = (r.get("database_verification") or {}).get("database_comparison") or {}
    for k in ("customer_id", "name", "date_of_birth", "expiry_date"):
        row[f"db_{k}_match"] = (dbc.get(k) or {}).get("status")
    row["errors"] = len(r.get("errors") or [])
    return row


def write_outputs(records: List[Dict[str, Any]] = None, cfg: Config = CFG) -> Dict[str, Path]:
    """All files written once, from accumulated records -- no per-field open/close."""
    records = records if records is not None else load_results()
    rep, paths = DIRS["reports"], {}

    with (rep / "identity_extraction_results.jsonl").open("w", encoding="utf-8") as fh:
        fh.write("\n".join(json.dumps(r, ensure_ascii=False, default=str) for r in records))
    paths["jsonl"] = rep / "identity_extraction_results.jsonl"

    flat = pd.DataFrame([flatten_customer(r) for r in records]) if records else pd.DataFrame()
    flat.to_csv(rep / "identity_extraction_results.csv", index=False, encoding="utf-8-sig")
    paths["identity_csv"] = rep / "identity_extraction_results.csv"

    pd.DataFrame([{"customer_id": r["customer_id"], **m} for r in records
                  for m in (r.get("cross_document_verification") or [])]).to_csv(
        rep / "cross_document_verification.csv", index=False, encoding="utf-8-sig")
    paths["cross_document"] = rep / "cross_document_verification.csv"

    dbrows = []
    for r in records:
        d = r.get("database_verification") or {}
        c = d.get("database_comparison") or {}
        dbrows.append({"customer_id": r["customer_id"], "db_id": d.get("db_id"),
                       "id_match_status": (c.get("customer_id") or {}).get("status"),
                       "ocr_full_name": (c.get("name") or {}).get("document_value"),
                       "db_full_name": (c.get("name") or {}).get("reference_value"),
                       "name_match_status": (c.get("name") or {}).get("status"),
                       "name_similarity": (c.get("name") or {}).get("similarity_score"),
                       "name_observed_order": (c.get("name") or {}).get("observed_order"),
                       "ocr_date_of_birth": (c.get("date_of_birth") or {}).get("document_value"),
                       "db_date_of_birth": (c.get("date_of_birth") or {}).get("reference_value"),
                       "dob_match_status": (c.get("date_of_birth") or {}).get("status"),
                       "ocr_expiry_date": (c.get("expiry_date") or {}).get("document_value"),
                       "db_expiry_date": (c.get("expiry_date") or {}).get("reference_value"),
                       "expiry_match_status": (c.get("expiry_date") or {}).get("status"),
                       "overall": d.get("overall")})
    pd.DataFrame(dbrows).to_csv(rep / "database_verification.csv", index=False,
                                encoding="utf-8-sig")
    paths["database"] = rep / "database_verification.csv"

    final_cols = ["customer_id", "overall_kyc_status", "needs_manual_review", "overall_quality",
                  "reasons", "surname_latin", "given_names_latin", "date_of_birth",
                  "expiry_date", "document_number", "mrz", "mrz_candidate_page",
                  "db_name_match", "db_date_of_birth_match", "db_expiry_date_match",
                  "db_customer_id_match", "errors"]
    if len(flat):
        flat[[c for c in final_cols if c in flat.columns]].to_csv(
            rep / "final_kyc_results.csv", index=False, encoding="utf-8-sig")
    paths["final"] = rep / "final_kyc_results.csv"

    build_processing_log(records).to_csv(rep / "processing_log.csv", index=False,
                                         encoding="utf-8-sig")
    paths["processing_log"] = rep / "processing_log.csv"

    inf, by_task, by_stage = performance_frames()
    inf.to_csv(rep / "inference_metrics.csv", index=False, encoding="utf-8-sig")
    by_task.to_csv(rep / "performance_by_task.csv", index=False, encoding="utf-8-sig")
    by_stage.to_csv(rep / "performance_summary.csv", index=False, encoding="utf-8-sig")
    paths.update({"inference_metrics": rep / "inference_metrics.csv",
                  "performance_by_task": rep / "performance_by_task.csv",
                  "performance_summary": rep / "performance_summary.csv"})

    pd.DataFrame([e for r in records for e in (r.get("errors") or [])],
                 columns=["customer_id", "document", "page", "stage", "error_type",
                          "error_message", "traceback", "timestamp"]).to_csv(
        rep / "errors.csv", index=False, encoding="utf-8-sig")
    paths["errors"] = rep / "errors.csv"

    for k, v in paths.items():
        print(f"  {k:<22}: {v}")
    if len(flat):
        print("\nKYC status:", dict(flat["overall_kyc_status"].value_counts()))
        print("MRZ read  :", int(flat["mrz"].notna().sum()), "/", len(flat))
    return paths


print("CELL 45 ready: write_outputs()")

## Driver

In [ ]:
# =========================================================================
# CELL — DRIVER
# =========================================================================
with PROF.stage("discovery"):
    catalog_df = extract_zip(CFG.ZIP_PATH, CFG)
    customers, doc_inventory = select_documents(catalog_df)
    presence_df = build_presence_report(customers, catalog_df, CFG)
    TARGETS = select_targets(customers, CFG)
doc_inventory.to_csv(DIRS["reports"] / "document_inventory.csv", index=False,
                     encoding="utf-8-sig")

MODE = "DRY_RUN" if CFG.DRY_RUN else ("BENCHMARK" if CFG.BENCHMARK_MODE else "FULL")
if CFG.BENCHMARK_MODE:
    TARGETS = TARGETS[:CFG.BENCHMARK_CUSTOMERS]

with PROF.stage("planning"):
    plan_tasks_df, plan_docs_df = build_execution_plan(TARGETS, CFG)
plan_tasks_df.to_csv(DIRS["reports"] / "dry_run_plan.csv", index=False, encoding="utf-8-sig")

n_pdfs = int((plan_docs_df["page_count"] > 0).sum()) if len(plan_docs_df) else 0
n_pages = int(plan_docs_df["page_count"].clip(lower=0).sum()) if len(plan_docs_df) else 0
print("=" * 58)
print("KYC PIPELINE START")
print("=" * 58)
print(f"Mode:                      {MODE}")
print(f"Customers:                 {len(TARGETS)}")
print(f"PDFs:                      {n_pdfs}")
print(f"Pages:                     {n_pages}")
print(f"Planned tasks:             {len(plan_tasks_df)}")
print(f"Expected Qwen calls:       {len(plan_tasks_df)}")
print(f"Max allowed (total):       {CFG.MAX_TOTAL_QWEN_CALLS}")
print(f"Max allowed per customer:  {CFG.MAX_QWEN_CALLS_PER_CUSTOMER}")
print(f"Max allowed per PDF:       {CFG.MAX_QWEN_CALLS_PER_PDF}")
print(f"Attention backend:         {ATTENTION_IMPL}")
print(f"Model:                     {getattr(ENGINE, 'name', '?')}")
print("=" * 58)
validate_call_budget(plan_tasks_df, plan_docs_df, CFG)      # raises BEFORE any inference
print_execution_plan(plan_tasks_df, plan_docs_df, CFG)

if CFG.DRY_RUN:
    print("\nDRY_RUN is ON: no Qwen call was made. dry_run_plan.csv written.")
    print("Review the plan, then set CFG.DRY_RUN = False and re-run this cell.")
else:
    WARMUP_STATS = warm_up(CFG)
    print("warm-up:", WARMUP_STATS)
    arm_watchdog(CFG.BENCHMARK_MAX_SECONDS if CFG.BENCHMARK_MODE else None)
    try:
        batch_df = run_pipeline(TARGETS, OCR_ENGINE, DB, CFG)
    except TimeoutError as exc:
        log.error("STOPPED BY WATCHDOG: %s", exc)
        batch_df = pd.DataFrame()
    finally:
        arm_watchdog(None)
    records = load_results()
    paths = write_outputs(records, CFG)
    inf_df, task_df, stage_df = print_performance()
    print(f"\nTOTAL QWEN CALLS: {QWEN_COUNTER['total']}")
    if len(batch_df):
        display(batch_df)

In [ ]:
# =========================================================================
# CELL 47 — BENCHMARK RUN  (optional: measures legacy vs optimised on your machine)
# =========================================================================
# Optimized only, one customer, hard time limit. Legacy comparison requires
# CFG.BENCHMARK_LEGACY = True and will deliberately take much longer.
# bench_df = benchmark_pipeline()
print("run: bench_df = benchmark_pipeline()")
print(f"     modes={('legacy','optimized') if CFG.BENCHMARK_LEGACY else ('optimized',)} | "
      f"customers={CFG.BENCHMARK_CUSTOMERS} | limit={CFG.BENCHMARK_MAX_SECONDS:.0f}s")

## Performance analysis and self-audit

### What was causing > 1 hour per PDF

Counting calls first: the previous notebook issued **15–30 Qwen calls per customer** (lines +
fields per page, a title call per page, up to 2 tile retries per page, up to 3 MRZ attempts).
That alone cannot produce an hour — it would require **120–240 s per call**, which is absurd for
a 27B FP8 model on an H100. So the dominant cost was inside each call, and two of the three
causes were in the previous notebook's own generation plumbing:

1. **A device synchronisation on every generated token.** `_LogprobRecorder` ran as a
   `LogitsProcessor` and called `.tolist()` at each step, forcing `cudaDeviceSynchronize` —
   768 stalls per `lines` call, each preceded by an fp32 `log_softmax` over a ~152k vocabulary.
   The H100 spent most of its time idle waiting for Python.
2. **An O(n²) stopping criterion.** `_BraceStop` re-decoded the entire generated tail every 8
   tokens and walked it character by character: ~36,500 re-decoded tokens for one `lines` call,
   on the CPU, between GPU steps.
3. **Redundant work around the calls:** the PDF was re-opened for every ROI clip, the heading was
   a Qwen call even when the PDF had a text layer, `lines` and `fields` both ran on every page,
   and every call ran at batch size 1.

### What changed

| Change | Mechanism |
|---|---|
| Token log-prob capture removed entirely; `output_scores=False` always | removes 128–768 device syncs per call and the per-step vocab tensor |
| `_FastBraceStop` keeps brace depth as state, decodes only new tokens, checks every 16 | O(n) instead of O(n²) |
| `PageCache` opens the PDF once, renders once, and clips ROIs through the same handle | no repeated `pymupdf.open`, no repeated render |
| Page view (quality + preprocessing) computed once and cached on the page object | no duplicate CLAHE/deskew |
| PDF **text layer** supplies headings and MRZ page hints when present | removes ~1 call per document |
| One combined call per page: document title **and** all identity fields | 2 calls/page → 1 |
| `lines` demoted from a per-page pass to a retry | ~1 call/page saved |
| Task planner: blank pages, absent fields and already-satisfied documents generate no tasks | fewer calls, same coverage |
| Micro-batching by generation budget, with OOM halving | real H100 utilisation |
| Debug images off by default, with `DEBUG_SAMPLE_CUSTOMERS` | no thousands of PNG writes |

**Expected: 15–30 calls → roughly 6–10 per customer.** The notebook does not assert a speedup:
`benchmark_pipeline()` runs the same customers in `legacy` and `optimized` mode and prints the
measured ratio, call counts and peak GPU memory from your machine.

### What was NOT traded away

Resolution policy, the MRZ pipeline, handwriting crops, every extracted field, the
anti-hallucination rules and the verification isolation are unchanged. MRZ candidate detection
still runs on **every** identity page — it is pure CV and costs no GPU time; what changed is that
Qwen is now invoked only for the winning candidate instead of unconditionally.

### Self-audit

| | |
|---|---|
| Driver redesigned, not wrapped | CELL 33 `run_pipeline` + CELL 30 `process_customer` |
| Bottlenecks measured | CELL 12 profiler, CELL 34 per-stage / per-task tables |
| Qwen calls counted | `inference_metrics.csv`, `calls_per_customer` in the benchmark |
| Model + processor loaded once | CELL 9 singleton; `free_model()` to release |
| Pages rendered once and cached | `PageCache` |
| Duplicate preprocessing eliminated | `PageCache.view()` memoised per page |
| `torch.inference_mode()` | `BatchedOCREngine._generate` |
| SDPA, no FlashAttention | CELL 6; `flash_attn` never imported or installed |
| No unnecessary CUDA sync | `Profiler.sync` only when `PROFILE_CUDA_SYNC` is on |
| `empty_cache()` only on OOM | `BatchedOCREngine.run` |
| Task-specific `max_new_tokens` | 384 identity / 160 MRZ / 128 name / 96 title / 512 lines |
| Deterministic generation | `do_sample=False`, `TEMPERATURE=0.0` |
| Batching evaluated and tunable | `INFERENCE_BATCH_SIZE`, `tune_batch_size()` |
| Retries bounded and different each time | `MAX_TARGETED_RETRIES`, variant ladder |
| MRZ: every page inspected, page 1 or 2, cheap detector, Qwen only if justified | CELLS 21, 23 |
| Database: four columns, CPU-side, indexed once | CELL 8 + `compare_database` |
| Unreadable ≠ mismatch; database never fills OCR | `compare_values`, `compare_database` |

### Two things to check on your hardware first

**Run `gpu_report()` before anything else.** If the previous run was also affected by CPU
offload — `device_map="auto"` with a `cpu` budget will silently place decoder layers in host
memory, and every forward then streams weights over PCIe — no amount of batching will help. The
FP8 weights are ~25 GiB, so on an 80 GiB H100 with `RESERVE_VRAM_GIB=8` they fit comfortably;
but on a shared or 40 GiB card they may not. The loader logs the device map summary and free VRAM
after loading.

**Then run `print_performance()` on a handful of customers.** It prints whether the run is
GPU-bound, CPU-bound or mixed. If it reports CPU-bound, the fix is `PDF_RENDER_DPI` and
`CPU_WORKERS`, not the model — and changing generation settings would be wasted effort.